UPscaledEV Project Main Code
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER

Readme:
This code aims at formulating a complex optimization framework with DERs of building, EV, PV and BESS, to solve a value-stacking problem with wholesale market and demand response market participation. It's based on Yi-An Chen's sclaedev codd and more consice, robust and well-constructed.
Labels:
NOTE: means there's a note and please read it
FIXME: means it's a bug or a problem that needs solving
TODO: means it's a to-do task, but not critical to the code running
VERSION: means there're more than one version for the diversity purpose, possibly shows in objective function choices or results analyses
Acknowledgement:
I gratefully acknowledge the support from my PI Jan Kleissl and TotalEnergies team, and the contributions from Yi-An Chen, whose prior work laid the foundation for this project. Special thanks to the UCSD Grid Lab team members.

Set working path and import packages

In [1]:
import os
os.chdir('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024') 
print("Path is:", os.getcwd(), "\n")
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from datetime import timedelta
from datetime import datetime
import time
from time import process_time
import calendar
import holidays
from pathlib import Path
import cvxpy as cp
import sys
from forecast_ED_PD import KnownUser, UnKnownUser
import glob
import warnings
from tqdm import tqdm
import gurobipy as gp
import seaborn as sns
import plotly.express as px

Path is: /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024 



Parameters

In [2]:
# MPC parameters
import warnings
warnings.filterwarnings(
    "ignore",
    message="Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.",
    category=UserWarning,
)
dt_m_EV = 15
dt_h = dt_m_EV/60
interval = timedelta(minutes=dt_m_EV)
c_BESS_penalty = 1e-3  # penalty coefficient for BESS ramping
M = 1e5  # large number for big-M method
# BESS parameters
SOC_BESS_min = 0.05
SOC_BESS_max = 0.95
P_BESS_max = 250                     # kW
C_BESS = 332                         # kWh
gamma = 0.9                          # efficiency
# Plot styles
palette = plt.get_cmap('tab10').colors
label_fontsize = 12
title_fontsize = 14
tick_fontsize = 10
legend_fontsize = 8
linewidth = 1.5
# https://sdcommunitypower.org/wp-content/uploads/2025/02/Schedule-for-WebsiteRes_2021V.pdf
# TODO: holiday on-off peaks not considered here, winter offpeak hour different
season = 'Summer' # 'Summer'/'Winter' (6/1-10/31; 11/1-5/31 2025-2026)
# Demand Charge and TOU
c_NCD = 15.38
if season == 'Summer':
    c_PD = 3.05
    c_delivery_sofnof = 0.03921
    c_delivery_on = 0.335
    c_energy_on = c_delivery_on + 0.490
    c_energy_off = c_delivery_sofnof + 0.175
    c_energy_superoff = c_delivery_sofnof + 0.0815
    onpeak = list(range(16*4, 21*4))          # 4pm–9pm
    offpeak = list(range(6*4, 16*4)) + list(range(21*4, 24*4))         # 6am–4pm, 9pm–12am
    superoffpeak = list(range(0, 6*4))        # 12am–6am
else:
    c_PD = 0.63
    c_delivery_sofnof = 0.03978
    c_delivery_on = 0.384
    c_energy_on = c_delivery_on + 0.169
    c_energy_off = c_delivery_sofnof + 0.095
    c_energy_superoff = c_delivery_sofnof + 0.073
    onpeak = list(range(16*4, 21*4))          # 4pm–9pm
    offpeak = list(range(6*4, 16*4)) + list(range(21*4, 24*4))         # 6am–4pm, 9pm–12am
    superoffpeak = list(range(0, 6*4))        # 12am–6am
c_e_TOU_AL = np.ones(96)
c_e_TOU_AL[onpeak] = c_energy_on
c_e_TOU_AL[offpeak] = c_energy_off
c_e_TOU_AL[superoffpeak] = c_energy_superoff
# plot c_e_TOU_AL
fig, ax = plt.subplots(figsize=(10, 5))
time_index = pd.date_range(start='2025-06-01', periods=96, freq='15min')
ax.plot(time_index, c_e_TOU_AL, marker='')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('Time-of-Use Electricity Rates for San Diego Community Power (Summer)', fontsize=14)
ax.set_xlabel('Time of Day', fontsize=12)
ax.set_ylabel('Electricity Rate ($/kWh)', fontsize=12)
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plot_dir = "Results/Plots"
# plt.show()
plt.savefig(os.path.join(plot_dir, 'c_e_TOU_AL_Summer.png'))
plt.close(fig)
# Tax
c_tax_DWR = 0.00580                               # x Total kWh
c_tax_oEESbo_Franchise = 0.0688 * c_tax_DWR       # x Total kWh
c_tax_CA_Surcharge = 0.00030                      # x Total kWh
c_tax_CA_Regulatory = 0.00058                     # x Total kWh
# x Total Bill (UDC+Commodity)
c_tax_SD_Franchise = 0.0578
c_tax_all = c_tax_DWR + c_tax_oEESbo_Franchise + c_tax_CA_Surcharge + c_tax_CA_Regulatory
# Forecast methods
Numb_EVs = 0
Numb_AbsDiffEVs = 0
Fc_SessionkWh = 'PersistenceSessionkWh'  # 'PerfectSessionkWh' #'PersistenceSessionkWh'
Fc_NumbEV = 'PersistenceNumbEV'          # 'PerfectNumbEV'     #'PersistenceNumbEV'
Fc_AtArrival = 'PerfectatArrival'    # 'PerfectatArrival'  #'MLatArrival' from Avik
Fc_building = 'Perfect'              # 'Perfectat'  #'Persistence'
Fc_PV = 'Perfect'                    # 'Perfectat'  #'Persistence'
# NOTE: to get the Dispatch file before baseline calculation, run the baseline.py file before running this file
DAM = 1 # 1/0: w/o day-ahead market participation (Demand Response)
# Case configuration. Set RUN_CASE1=True to also run the reduced-service Case1.
# Default False forces 100% energy service for all EV drivers and speeds up tests.
RUN_CASE1 = False
Cases = ['Base', 'Case1'] if RUN_CASE1 else ['Base']
DA_CASE_IDX = len(Cases)  # index for the DA/V0G reference case in len(Cases)+1 arrays
RT_DA_COMMITMENT_MODE = 'settlement'  # CAISO two-settlement: fixed DA parameter plus RT deviation settlement
MPC_VERSION_LABEL = 'Shrinking MPC'
SHOW_6PANEL_SUMMARY_BOX = False  # The old RT summary box used local solver-output values, not daily totals.
PLOT_DAILY_6PANEL_DATES = 'all'  # []: skip daily 6-panel; None/'all': all days; or use [1, 2] / ['20250701']
ANALYSIS_DAYS_CONFIG = None  # None uses RUN_DAYS_CONFIG for final financial plots
SAVE_FINAL_FINANCIAL_FIGURES = False  # main notebooks save CSVs; paper figures are made in paper_financial_comparison_plots.ipynb
Direction_Aligned_AddOn = True # True/False: whether to add constraints when direction aligned in WM
#sym:WM_Mode switch for market participation
WM_Mode = 'full'  # 'full' or 'wm_only' or 'retail_only'
Enable_WM = (WM_Mode in ['full'])
year = 2025
RUN_DAYS_CONFIG = list(range(1, 31))
GUROBI_MIPGAP = 1e-2
SOLVER_THREADS = 6
RT_SOLVER_TIME_LIMIT = None  # no normal RT cutoff: every accepted step must reach the target MIPGap
RT_SOLVER_EMERGENCY_TIME_LIMIT = 300.0  # watchdog only; hitting it aborts the scenario instead of accepting an incumbent
REQUIRE_RT_OPTIMAL = True
RT_SOLVER_STATUS_COUNTS = {}
RT_SOLVER_LIMIT_EVENTS = []
def _record_rt_solver_status(problem, date_label, interval_idx, case_label):
    """Record whether Gurobi actually reached its RT TimeLimit."""
    status = str(problem.status)
    RT_SOLVER_STATUS_COUNTS[status] = RT_SOLVER_STATUS_COUNTS.get(status, 0) + 1
    extra = getattr(getattr(problem, 'solver_stats', None), 'extra_stats', None)
    gurobi_status = getattr(extra, 'Status', None)
    hit_limit = (status == 'user_limit') or (gurobi_status == 9)
    if hit_limit:
        def _safe_attr(name):
            try:
                return float(getattr(extra, name))
            except Exception:
                return float('nan')
        RT_SOLVER_LIMIT_EVENTS.append({
            'date': pd.Timestamp(date_label).strftime('%Y-%m-%d'),
            'interval': int(interval_idx),
            'case': str(case_label),
            'forecast': str(Fc_SessionkWh),
            'mode': str(WM_Mode),
            'cvxpy_status': status,
            'gurobi_status': gurobi_status,
            'runtime_s': _safe_attr('Runtime'),
            'mip_gap': _safe_attr('MIPGap'),
        })
# 自定义你要保存和分析的目标，可选 'DA' 或 'RT'
TARGET_SAVE = 'RT'
def _date_key_allowed(date_like, selected_dates):
    """Return whether a daily expensive plot should be saved for date_like."""
    if selected_dates is None or selected_dates == 'all':
        return True
    if selected_dates == [] or selected_dates == () or selected_dates == set():
        return False
    target = pd.Timestamp(date_like).strftime('%Y%m%d')
    allowed = set()
    for item in selected_dates:
        s = str(item)
        if s.isdigit() and len(s) <= 2:
            allowed.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(s):02d}")
        else:
            allowed.add(pd.Timestamp(item).strftime('%Y%m%d'))
    return target in allowed
RUN_MONTHS_CONFIG = [6, 7]


Data processing

In [3]:
if Fc_AtArrival == 'MLatArrival':
    User_Data = pd.read_csv("Driver_Table.csv")
    User_known = User_Data['driver_id'][User_Data['TotSession']>10].unique()
    UserNoBess = []
# create weekdays (excluding holidays) and weekends (including holidays) lists
Holidays = holidays.US(years=year)
Holidays_dates = list(Holidays.keys())
Holidays_dates = [date for date in Holidays_dates if date.year == year]
start_ind_Y = datetime(year,1,1)
end_ind_Y = datetime(year+1,1,1)
DateSeries_ThisY = []
while start_ind_Y < end_ind_Y:
    DateSeries_ThisY.append(start_ind_Y)
    start_ind_Y += timedelta(hours=24)
DateSeries_ThisY = pd.Series(DateSeries_ThisY)
Y_Weekends = DateSeries_ThisY[DateSeries_ThisY.dt.dayofweek>=5]
Y_Holidays = DateSeries_ThisY[DateSeries_ThisY.dt.date.isin(Holidays_dates)]
Y_WeekendsWH = pd.concat([Y_Weekends,Y_Holidays],axis=0).drop_duplicates(keep='first', inplace=False).sort_values(axis=0, ascending=True).reset_index(drop=True)
Y_WeekdaysWOH = DateSeries_ThisY[~(DateSeries_ThisY.isin(Y_WeekendsWH))].sort_values(axis=0, ascending=True).reset_index(drop=True)
if len(Y_WeekdaysWOH) + len(Y_WeekendsWH) != len(DateSeries_ThisY):
    print("Error in holiday identification!\n")
    sys.exit()
# VERSION: 2025 EV data
Data = pd.read_csv('2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv', low_memory=False)
Data['Interval start'] = pd.to_datetime(Data['Interval start'])
Data['Interval end'] = pd.to_datetime(Data['Interval end'])
Data['Session start'] = pd.to_datetime(Data['Session start'])
Data['Session end'] = pd.to_datetime(Data['Session end'])
Data['Interval max demand kW'] = pd.to_numeric(Data['Interval max demand kW'], errors='coerce')
Data['Interval average demand kW'] = pd.to_numeric(Data['Interval average demand kW'], errors='coerce')
print("Number of Intervals:", len(Data))
print("Test year:", year)
print("Date range:", Data['Interval start'].min(), "to", Data['Interval start'].max())
print("Unique sites:", len(Data['Site'].unique()))
print("Unique chargers:", Data['Station Name'].nunique(dropna=True))
Data['Car_'] = Data['10-digit UID'].astype(str)
# Read baseline data
dir_Input = os.path.join('Results/Dispatch')
filename_Input = dir_Input + '/2025_'+Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival +'_baseline.csv'
Dispatch_2025_baseline = pd.read_csv(filename_Input, low_memory=False, header = 0)           
Dispatch_2025_baseline['Interval start'] = pd.to_datetime(Dispatch_2025_baseline['Interval start'])


Number of Intervals: 374890
Test year: 2025
Date range: 2025-01-01 03:15:00 to 2025-07-31 21:00:00
Unique sites: 23
Unique chargers: 440


Define function for plotting and testing

In [4]:
# Helper: generate daily fig01 inside the day loop
def plot_daily_solver_choice_figures(
    TheDate_Day0,
    dt_h,
    Solver_Outputs,
    c_e_TOU_AL,
    Bid_Pr_DA,
    Bid_Pr_RT,
    AS_Pr_RU_DA,
    AS_Pr_RU_RT,
    AS_Pr_RD_DA,
    AS_Pr_RD_RT,
    AS_Pr_SP_DA,
    AS_Pr_SP_RT,
    AS_Pr_NSP_DA,
    AS_Pr_NSP_RT,
    alpha_RU,
    alpha_RD,
    alpha_SP,
    alpha_NSP,
    run_tag,
    M_Th_NCD,
    M_Th_PD,
    P_BESS_max,
    P_EV_max,
    mpc_version_label=None,
    forecast_label=None,
    threshold_case_idx=0,
):
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.ticker import MaxNLocator
    res = Solver_Outputs
    def _fit(vec, n):
        arr = np.asarray(vec, dtype=float).flatten()
        if arr.size == n:
            return arr
        if arr.size == 1:
            return np.full(n, float(arr[0]))
        if arr.size < n:
            return np.pad(arr, (0, n - arr.size), mode="edge")
        return arr[:n]
    n = len(np.asarray(res["p_GI"], dtype=float).flatten())
    x = np.arange(n)
    time_index = pd.date_range(start=TheDate_Day0, periods=n, freq=f"{int(round(dt_h * 60))}min")
    hour_step = max(1, int(round(2.0 / dt_h)))
    ticks = np.arange(0, n, hour_step)
    tick_labels = [time_index[i].strftime("%H:%M") for i in ticks]
    p_da = _fit(res["p_DA"], n)
    p_rt = _fit(res["p_RT"], n)
    c_ru_da_raw = np.abs(_fit(res["c_RU_DA"], n))
    c_ru_rt_raw = _fit(res["c_RU_RT"], n)
    c_rd_da_raw = np.abs(_fit(res["c_RD_DA"], n))
    c_rd_rt_raw = _fit(res["c_RD_RT"], n)
    c_sp_da_raw = np.abs(_fit(res["c_SP_DA"], n))
    c_sp_rt_raw = _fit(res["c_SP_RT"], n)
    c_nsp_da_raw = np.abs(_fit(res["c_NSP_DA"], n))
    c_nsp_rt_raw = _fit(res["c_NSP_RT"], n)
    p_ch = np.abs(_fit(res["p_ch_BESS"], n))
    p_dch = np.abs(_fit(res["p_dch_BESS"], n))
    p_ch_wm = np.abs(_fit(res["p_ch_BESS_WM"], n))
    p_dch_wm = np.abs(_fit(res["p_dch_BESS_WM"], n))
    p_ch_nwm = np.abs(_fit(res["p_ch_BESS_NWM"], n))
    p_dch_nwm = np.abs(_fit(res["p_dch_BESS_NWM"], n))
    p_bess = _fit(res["p_BESS"], n)
    p_ev = _fit(res["p_EV"], n)
    p_gi = _fit(res["p_GI"], n)
    soc = _fit(res["soc_BESS"], n)
    p_dch_wm_neg = -p_dch_wm
    p_dch_nwm_neg = -p_dch_nwm
    soc_pct = soc * 100.0
    bid_pr_da = _fit(Bid_Pr_DA, n)
    bid_pr_rt = _fit(Bid_Pr_RT, n)
    as_ru_da = _fit(AS_Pr_RU_DA, n)
    as_ru_rt = _fit(AS_Pr_RU_RT, n)
    as_rd_da = _fit(AS_Pr_RD_DA, n)
    as_rd_rt = _fit(AS_Pr_RD_RT, n)
    as_sp_da = _fit(AS_Pr_SP_DA, n)
    as_sp_rt = _fit(AS_Pr_SP_RT, n)
    as_nsp_da = _fit(AS_Pr_NSP_DA, n)
    as_nsp_rt = _fit(AS_Pr_NSP_RT, n)
    tou_price = _fit(c_e_TOU_AL, n)
    tp_lmp_da = bid_pr_da
    tp_lmp_rt = bid_pr_rt
    tp_ru_da = bid_pr_rt * alpha_RU + as_ru_da
    tp_ru_rt = bid_pr_rt * alpha_RU + as_ru_rt
    tp_rd_da = bid_pr_rt * alpha_RD + as_rd_da
    tp_rd_rt = bid_pr_rt * alpha_RD + as_rd_rt
    tp_sp_da = bid_pr_rt * alpha_SP + as_sp_da
    tp_sp_rt = bid_pr_rt * alpha_SP + as_sp_rt
    tp_nsp_da = bid_pr_rt * alpha_NSP + as_nsp_da
    tp_nsp_rt = bid_pr_rt * alpha_NSP + as_nsp_rt
    # --- Save daily price table (once per day — same prices for all 3 WM_Mode cases) ---
    price_table_dir = Path("Results/Plots/Daily_Price_Tables") / time_index[0].strftime("%Y%m")
    price_table_dir.mkdir(parents=True, exist_ok=True)
    price_table_path = price_table_dir / f"daily_prices_{time_index[0].strftime('%Y%m%d')}.csv"
    df_prices = pd.DataFrame({
        "time": time_index,
        "TOU_$/kWh": np.round(tou_price, 4),
        "LMP_DA_$/kWh": np.round(tp_lmp_da, 4),
        "LMP_RT_$/kWh": np.round(tp_lmp_rt, 4),
        "TP_RU_DA_$/kWh": np.round(tp_ru_da, 4),
        "TP_RU_RT_$/kWh": np.round(tp_ru_rt, 4),
        "TP_RD_DA_$/kWh": np.round(tp_rd_da, 4),
        "TP_RD_RT_$/kWh": np.round(tp_rd_rt, 4),
        "TP_SP_DA_$/kWh": np.round(tp_sp_da, 4),
        "TP_SP_RT_$/kWh": np.round(tp_sp_rt, 4),
        "TP_NSP_DA_$/kWh": np.round(tp_nsp_da, 4),
        "TP_NSP_RT_$/kWh": np.round(tp_nsp_rt, 4),
    })
    df_prices.to_csv(price_table_path, index=False, float_format="%.4f")
    vol = np.abs(np.diff(p_bess, prepend=p_bess[0]))
    turn_candidates = [
        i
        for i in range(1, n - 1)
        if (p_bess[i] - p_bess[i - 1]) * (p_bess[i + 1] - p_bess[i]) <= 0
    ]
    ranked = sorted(set(turn_candidates), key=lambda i: -float(vol[i]))
    if not ranked:
        ranked = [int(i) for i in np.argsort(-vol)]
    key_idx = []
    min_gap = max(2, int(round(0.75 / dt_h)))
    for idx in ranked:
        idx = int(idx)
        if all(abs(idx - j) >= min_gap for j in key_idx):
            key_idx.append(idx)
        if len(key_idx) >= 6:
            break
    key_idx = sorted(key_idx)
    c_e_tou = _fit(c_e_TOU_AL, n)
    wm_signal = tp_lmp_da + tp_lmp_rt + tp_ru_da + tp_ru_rt + tp_sp_da + tp_sp_rt + tp_nsp_da + tp_nsp_rt - tp_rd_da - tp_rd_rt
    reason_lines = []
    for num, idx in enumerate(key_idx, 1):
        mode = "dch" if p_bess[idx] < 0 else "ch"
        total_flow = p_dch[idx] if mode == "dch" else p_ch[idx]
        wm_comp = p_dch_wm[idx] if mode == "dch" else p_ch_wm[idx]
        nwm_comp = p_dch_nwm[idx] if mode == "dch" else p_ch_nwm[idx]
        if total_flow <= 1e-9:
            wm_nwm = "NWM"
        elif wm_comp <= 1e-6 and nwm_comp > 1e-6:
            wm_nwm = "NWM"
        elif nwm_comp <= 1e-6 and wm_comp > 1e-6:
            wm_nwm = "WM"
        else:
            wm_nwm = "WM" if wm_comp >= nwm_comp else "NWM"
        tou_pct = 100 * np.mean(c_e_tou <= c_e_tou[idx])
        wm_pct = 100 * np.mean(wm_signal <= wm_signal[idx])
        if wm_nwm == "WM":
            if mode == "dch":
                reason = f"WM signal high ({wm_pct:.0f}pctl) -> WM discharge"
            else:
                reason = f"WM signal low ({wm_pct:.0f}pctl) -> WM charge"
        else:
            if mode == "dch":
                reason = f"TOU high ({tou_pct:.0f}pctl) -> NWM discharge"
            else:
                reason = f"TOU low ({tou_pct:.0f}pctl) -> NWM charge"
        reason_lines.append(f"{num}) {time_index[idx].strftime('%H:%M')} {wm_nwm}-{mode}: {reason}")
    note_text = "Reasons for battery actions:\n" + "\n".join(reason_lines) if reason_lines else "No actions for this day."
    # BESS cycle count under throughput cap: dt_h * sum(ch + dch) <= 2*C*(SOCmax-SOCmin)
    throughput_kwh = dt_h * float(np.sum(p_ch + p_dch))
    cycle_den = np.nan
    try:
        cycle_den = 2.0 * float(C_BESS) * float(SOC_BESS_max - SOC_BESS_min)
    except Exception:
        cycle_den = np.nan
    cycle_count = throughput_kwh / cycle_den if np.isfinite(cycle_den) and cycle_den > 0 else np.nan
    def _panel_tag(ax, txt):
        ax.text(-0.05, 1.05, txt, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom")
    def _smart_legend(ax, handles=None, labels=None, max_cols=4):
        if handles is None or labels is None:
            handles, labels = ax.get_legend_handles_labels()
        n_items = len(labels)
        if n_items == 0:
            return None
        longest = max(len(str(label)) for label in labels)
        width_cap = 3 if longest > 18 else max_cols
        ncol = max(1, min(width_cap, (n_items + 1) // 2))
        fontsize = 9.5 if n_items > 8 else 10.5
        return ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.32), ncol=ncol, fontsize=fontsize)
    plot_dir = Path(f"Results/Plots/Solver_{TARGET_SAVE}_Choices") / run_tag / time_index[0].strftime("%Y%m%d")
    plot_dir.mkdir(parents=True, exist_ok=True)
    for old in plot_dir.glob("fig*.png"):
        old.unlink(missing_ok=True)
    # Build copy-friendly table for every timestep with BESS activity
    activity_mask = (p_ch + p_dch) > 1e-6
    wm_pct_all = np.array([100 * np.mean(wm_signal <= v) for v in wm_signal], dtype=float)
    tou_pct_all = np.array([100 * np.mean(c_e_tou <= v) for v in c_e_tou], dtype=float)
    def _classify_channel(k):
        if p_bess[k] >= 1e-6:
            if p_ch_wm[k] > 1e-6 and p_ch_nwm[k] > 1e-6:
                return "CH_MIX"
            if p_ch_wm[k] > 1e-6:
                return "CH_WM"
            if p_ch_nwm[k] > 1e-6:
                return "CH_NWM"
            return "CH"
        if p_bess[k] <= -1e-6:
            if p_dch_wm[k] > 1e-6 and p_dch_nwm[k] > 1e-6:
                return "DCH_MIX"
            if p_dch_wm[k] > 1e-6:
                return "DCH_WM"
            if p_dch_nwm[k] > 1e-6:
                return "DCH_NWM"
            return "DCH"
        return "IDLE"
    def _driver(channel, k):
        if channel == "DCH_WM":
            return "WM high"
        if channel == "CH_WM":
            return "WM low"
        if channel == "DCH_NWM":
            return "TOU high"
        if channel == "CH_NWM":
            return "TOU low"
        if channel.endswith("MIX"):
            return "Mixed"
        return "-"
    activity_rows = []
    for k in np.where(activity_mask)[0]:
        ch = _classify_channel(int(k))
        activity_rows.append({
            "time": time_index[k].strftime("%H:%M"),
            "channel_action": ch,
            "p_BESS_net(kW)": float(p_bess[k]),
            "p_ch_WM(kW)": float(p_ch_wm[k]),
            "p_dch_WM(kW)": float(p_dch_wm[k]),
            "p_ch_NWM(kW)": float(p_ch_nwm[k]),
            "p_dch_NWM(kW)": float(p_dch_nwm[k]),
            "SOC(%)": float(soc_pct[k]),
            "TOU($/kWh)": float(c_e_tou[k]),
            "LMP_DA($/kWh)": float(tp_lmp_da[k]),
            "LMP_RT($/kWh)": float(tp_lmp_rt[k]),
            "WM_signal($/kWh)": float(wm_signal[k]),
            "driver": _driver(ch, int(k)),
        })
    df_activity = pd.DataFrame(activity_rows)
    if not df_activity.empty:
        num_cols = [c for c in df_activity.columns if c not in ["time", "channel_action", "driver"]]
        df_activity[num_cols] = df_activity[num_cols].round(2)
    activity_csv = plot_dir / f"Bess_activity_table_{run_tag}.csv"
    if df_activity.empty:
        activity_txt_text = "No BESS activity timesteps for this day."
    else:
        activity_txt_text = df_activity.to_string(index=False)
    df_activity.to_csv(activity_csv, index=False, float_format='%.2f')
    fig = plt.figure(figsize=(28, 15), constrained_layout=False)
    date_str = time_index[0].strftime("%Y-%m-%d")
    mpc_txt = mpc_version_label or globals().get("MPC_VERSION_LABEL", "MPC")
    fc_txt = forecast_label or ("Perfect forecast" if globals().get("Fc_SessionkWh", "").startswith("Perfect") else "Persistence forecast")
    fig.suptitle(f"{mpc_txt} {TARGET_SAVE} 6-Panel | Date: {date_str} | Forecast: {fc_txt} | Mode: {WM_Mode}", fontsize=22, fontweight="bold", y=0.98)
    gs = fig.add_gridspec(3, 2, hspace=0.60, wspace=0.18, top=0.90, bottom=0.08)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharex=ax1)
    ax3 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax4 = fig.add_subplot(gs[1, 1], sharex=ax1)
    ax5 = fig.add_subplot(gs[2, 0], sharex=ax1)
    ax6 = fig.add_subplot(gs[2, 1], sharex=ax1)
    for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
        ax.set_xticks(ticks)
    ax1.plot(x, tp_ru_da, color="#2ca02c", linewidth=1.4, linestyle="-", label="TP RU DA")
    ax1.plot(x, tp_ru_rt, color="#2ca02c", linewidth=1.0, linestyle="--", alpha=0.85, label="TP RU RT")
    ax1.plot(x, tp_rd_da, color="#d62728", linewidth=1.4, linestyle="-", label="TP RD DA")
    ax1.plot(x, tp_rd_rt, color="#d62728", linewidth=1.0, linestyle="--", alpha=0.85, label="TP RD RT")
    ax1.plot(x, tp_sp_da, color="#9467bd", linewidth=1.4, linestyle="-", label="TP SP DA")
    ax1.plot(x, tp_sp_rt, color="#9467bd", linewidth=1.0, linestyle="--", alpha=0.85, label="TP SP RT")
    ax1.plot(x, tp_nsp_da, color="#8c564b", linewidth=1.4, linestyle="-", label="TP NSP DA")
    ax1.plot(x, tp_nsp_rt, color="#8c564b", linewidth=1.0, linestyle="--", alpha=0.85, label="TP NSP RT")
    ax1.plot(x, tp_lmp_da, color="#1f77b4", linewidth=1.9, linestyle="-", alpha=0.95, label="LMP DA")
    ax1.plot(x, tp_lmp_rt, color="#1f77b4", linewidth=1.2, linestyle="--", alpha=0.95, label="LMP RT")
    ax1.axhline(0, color="black", linewidth=0.9)
    ax1.set_ylabel("Total Price Contribution")
    ax1_tou = ax1.twinx()
    ax1_tou.plot(x, tou_price, color="#00a5a5", linewidth=2.2, linestyle="-.", label="TOU")
    ax1_tou.set_ylabel("TOU ($/kWh)", color="#008b8b")
    ax1_tou.tick_params(axis="y", labelcolor="#008b8b")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines_tou, labels_tou = ax1_tou.get_legend_handles_labels()
    _smart_legend(ax1, lines1 + lines_tou, labels1 + labels_tou)
    ax1.tick_params(labelbottom=False)
    _panel_tag(ax1, "(a) Total Expected Price")
    ax2.plot(x, as_ru_da, color="#2ca02c", linewidth=1.4, linestyle="-", label="Raw RU DA")
    ax2.plot(x, as_ru_rt, color="#2ca02c", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw RU RT")
    ax2.plot(x, as_rd_da, color="#d62728", linewidth=1.4, linestyle="-", label="Raw RD DA")
    ax2.plot(x, as_rd_rt, color="#d62728", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw RD RT")
    ax2.plot(x, as_sp_da, color="#9467bd", linewidth=1.4, linestyle="-", label="Raw SP DA")
    ax2.plot(x, as_sp_rt, color="#9467bd", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw SP RT")
    ax2.plot(x, as_nsp_da, color="#8c564b", linewidth=1.4, linestyle="-", label="Raw NSP DA")
    ax2.plot(x, as_nsp_rt, color="#8c564b", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw NSP RT")
    ax2.plot(x, tp_lmp_da, color="#1f77b4", linewidth=1.9, linestyle="-", alpha=0.95, label="LMP DA")
    ax2.plot(x, tp_lmp_rt, color="#1f77b4", linewidth=1.2, linestyle="--", alpha=0.95, label="LMP RT")
    ax2.axhline(0, color="black", linewidth=0.9)
    ax2.set_ylabel("Raw Price ($/kW)")
    ax2_tou = ax2.twinx()
    ax2_tou.plot(x, tou_price, color="#00a5a5", linewidth=2.2, linestyle="-.", label="TOU")
    ax2_tou.set_ylabel("TOU ($/kWh)", color="#008b8b")
    ax2_tou.tick_params(axis="y", labelcolor="#008b8b")
    lines2, labels2 = ax2.get_legend_handles_labels()
    lines_tou2, labels_tou2 = ax2_tou.get_legend_handles_labels()
    _smart_legend(ax2, lines2 + lines_tou2, labels2 + labels_tou2)
    ax2.tick_params(labelbottom=False)
    _panel_tag(ax2, "(b) Raw Prices")
    def _draw_stacked_bars(ax, components, ylabel):
        bar_w = 0.88
        pos_base = np.zeros(n)
        neg_base = np.zeros(n)
        for label, vec, color in components:
            vec = np.asarray(vec, dtype=float)
            pos = np.where(vec > 0, vec, 0.0)
            neg = np.where(vec < 0, vec, 0.0)
            added_label = False
            if np.any(pos):
                ax.bar(x, pos, width=bar_w, bottom=pos_base, color=color, alpha=0.82, label=label)
                pos_base += pos
                added_label = True
            if np.any(neg):
                this_label = "_nolegend_" if added_label else label
                ax.bar(x, neg, width=bar_w, bottom=neg_base, color=color, alpha=0.82, label=this_label)
                neg_base += neg
                added_label = True
            if not added_label:
                ax.bar(x, np.zeros(n), width=bar_w, color=color, alpha=0.82, label=label)
        ax.axhline(0, color="black", linewidth=0.9)
        ax.set_ylabel(ylabel)
        # 在调用 _draw_stacked_bars 之后添加：
        all_pos_sums = pos_base  # 这是函数内部累加后的最高点
        all_neg_sums = neg_base  # 这是函数内部累加后的最低点
        y_max = np.max(all_pos_sums) if np.any(all_pos_sums > 0) else 0
        y_min = np.min(all_neg_sums) if np.any(all_neg_sums < 0) else 0
        # 设置 5% 的缓冲，防止贴边，但又不至于留白太多
        if y_max == 0 and y_min == 0:
            ax.set_ylim(-1, 1) # 防错
        else:
            # 向上取一点点空间，向下取一点点空间
            ax.set_ylim(y_min * (1.05 if y_min < 0 else 0.95), y_max * 1.05)
    # Show the fixed DA commitment and the RT increment/decrement separately.
    # Their physical sum is enforced in the model, but is intentionally not plotted here.
    _baseline_kw = _fit(res["Baseline(kW)"], n)
    _P_BESS_max_local = float(P_BESS_max)
    _P_EV_max_local = P_EV_max
    _p_up_line   = _P_BESS_max_local + _baseline_kw
    _p_down_line = _P_BESS_max_local - _baseline_kw + _P_EV_max_local
    def _visible_components(components, tol=1e-5):
        return [(lbl, np.asarray(data, dtype=float), color) for lbl, data, color in components if np.nanmax(np.abs(np.asarray(data, dtype=float))) > tol]
    def _draw_signed_stacked_bars(ax, components, ylabel):
        # Match the original penalty-version panels: every named bid remains its own bar.
        # Positive and negative contributions are stacked independently from zero.
        positive_bottom = np.zeros(n)
        negative_bottom = np.zeros(n)
        for label, values, color in _visible_components(components):
            values = np.asarray(values, dtype=float)
            positive = np.where(values > 0.0, values, 0.0)
            negative = np.where(values < 0.0, values, 0.0)
            ax.bar(x, positive, width=0.88, bottom=positive_bottom, color=color, alpha=0.82, label=label)
            ax.bar(x, negative, width=0.88, bottom=negative_bottom, color=color, alpha=0.82)
            positive_bottom += positive
            negative_bottom += negative
        ax.axhline(0, color="black", linewidth=0.9)
        ax.set_ylabel(ylabel)
        return positive_bottom, negative_bottom
    up_components_c = [
        ("c_RU_DA", c_ru_da_raw, "#2ca02c"),
        ("c_RU_RT", c_ru_rt_raw, "#98df8a"),
        ("c_SP_DA", c_sp_da_raw, "#9467bd"),
        ("c_SP_RT", c_sp_rt_raw, "#c5b0d5"),
        ("c_NSP_DA", c_nsp_da_raw, "#8c564b"),
        ("c_NSP_RT", c_nsp_rt_raw, "#c49c94"),
    ]
    down_components_c = [
        ("c_RD_DA", c_rd_da_raw, "#d62728"),
        ("c_RD_RT", c_rd_rt_raw, "#ff9896"),
    ]
    components_c = [
        ("p_DA", p_da, "#1f77b4"),
        ("p_RT", p_rt, "#17becf"),
        *up_components_c,
        *((label, -vec, color) for label, vec, color in down_components_c),
    ]
    upper_bid_stack, lower_bid_stack = _draw_signed_stacked_bars(ax3, components_c, "Capacity / Bid (kW)")
    # The displayed lines are a settlement-aware visual envelope: they include the
    # physical resource bounds and the separately shown DA/RT financial decomposition.
    upper_plot_bound = np.maximum(_p_up_line, upper_bid_stack)
    lower_plot_bound = np.minimum(-_p_down_line, lower_bid_stack)
    ax3.plot(x, upper_plot_bound, color="#e6550d", linewidth=2.0, linestyle="-", label="upper bound", zorder=6)
    ax3.plot(x, lower_plot_bound, color="#3182bd", linewidth=2.0, linestyle="-", label="lower bound", zorder=6)
    _ymin, _ymax = ax3.get_ylim()
    _ymin = min(_ymin, float(np.nanmin(lower_plot_bound)))
    _ymax = max(_ymax, float(np.nanmax(upper_plot_bound)))
    _ypad = 0.05 * max(_ymax - _ymin, 1.0)
    ax3.set_ylim(_ymin - _ypad, _ymax + _ypad)
    # Validate the actual settlement position against the real model bounds separately.
    p_actual = p_da + p_rt
    physical_upper = np.maximum(p_actual, 0.0) + sum((vec for _, vec, _ in _visible_components(up_components_c)), np.zeros(n))
    physical_lower = np.minimum(p_actual, 0.0) - sum((vec for _, vec, _ in _visible_components(down_components_c)), np.zeros(n))
    _upper_violation = float(np.nanmax(physical_upper - _p_up_line))
    _lower_violation = float(np.nanmax((-_p_down_line) - physical_lower))
    if max(_upper_violation, _lower_violation) > 1e-3:
        raise ValueError(f"Settlement position exceeds physical bounds: upper={_upper_violation:.6g} kW, lower={_lower_violation:.6g} kW")
    _smart_legend(ax3)
    ax3.yaxis.set_major_locator(MaxNLocator(nbins=8))
    ax3.tick_params(labelbottom=False)
    _panel_tag(ax3, "(c) Bids")
    up_components_d = [
        ("Obl RU_DA", c_ru_da_raw * alpha_RU, "#2ca02c"),
        ("Obl RU_RT", c_ru_rt_raw * alpha_RU, "#98df8a"),
        ("Obl SP_DA", c_sp_da_raw * alpha_SP, "#9467bd"),
        ("Obl SP_RT", c_sp_rt_raw * alpha_SP, "#c5b0d5"),
        ("Obl NSP_DA", c_nsp_da_raw * alpha_NSP, "#8c564b"),
        ("Obl NSP_RT", c_nsp_rt_raw * alpha_NSP, "#c49c94"),
    ]
    down_components_d = [
        ("Obl RD_DA", c_rd_da_raw * alpha_RD, "#d62728"),
        ("Obl RD_RT", c_rd_rt_raw * alpha_RD, "#ff9896"),
    ]
    components_d = [
        ("Obl p_DA", p_da, "#1f77b4"),
        ("Obl p_RT", p_rt, "#17becf"),
        *up_components_d,
        *((label, -vec, color) for label, vec, color in down_components_d),
    ]
    _draw_signed_stacked_bars(ax4, components_d, "Energy Obligation")
    net_obligation = sum((np.asarray(vec, dtype=float) for _, vec, _ in _visible_components(components_d)), np.zeros(n))
    ax4.plot(x, net_obligation, color="black", linewidth=1.5, label="Net Obligation")
    _smart_legend(ax4)
    ax4.yaxis.set_major_locator(MaxNLocator(nbins=8))
    ax4.tick_params(labelbottom=False)
    _panel_tag(ax4, "(d) Energy Obligation (α * Capacity)")
    wm_flow = float(np.max(np.abs(p_ch_wm)) + np.max(np.abs(p_dch_wm)))
    if wm_flow > 1e-6:
        ax5.bar(x, p_ch_wm, width=0.92, color="#1f77b4", alpha=0.72, label="BESS ch WM")
        ax5.bar(x, p_ch_nwm, width=0.92, bottom=p_ch_wm, color="#17becf", alpha=0.72, label="BESS ch NWM")
        ax5.bar(x, p_dch_wm_neg, width=0.92, color="#d62728", alpha=0.72, label="BESS dch WM")
        ax5.bar(x, p_dch_nwm_neg, width=0.92, bottom=p_dch_wm_neg, color="#ff9896", alpha=0.72, label="BESS dch NWM")
    else:
        ax5.bar(x, p_ch_nwm, width=0.92, color="#17becf", alpha=0.72, label="BESS ch NWM")
        ax5.bar(x, p_dch_nwm_neg, width=0.92, color="#ff9896", alpha=0.72, label="BESS dch NWM")
    ax5.axhline(0, color="black", linewidth=0.9)
    ax5.set_ylabel("BESS Power (kW)")
    ax5.text(
        0.02,
        0.03,
        f"Cycle count: {cycle_count:.3f} / 1.000",
        transform=ax5.transAxes,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="lightgrey", boxstyle="round,pad=0.25"),
    )
    ax5_r = ax5.twinx()
    ax5_r.plot(x, soc_pct, color="#2ca02c", linewidth=1.8, alpha=0.95, label="SOC")
    ax5_r.set_ylabel("SOC (%)", color="#2ca02c")
    ax5_r.tick_params(axis="y", labelcolor="#2ca02c")
    for num, idx in enumerate(key_idx, 1):
        yv = soc_pct[idx]
        ax5_r.scatter(idx, yv, color="black", s=25, zorder=6)
        ax5_r.text(idx, yv + 1.8, str(num), fontsize=12, fontweight="bold", color="black", ha="center", va="bottom")
    h5, l5 = ax5.get_legend_handles_labels()
    h5_r, l5_r = ax5_r.get_legend_handles_labels()
    _smart_legend(ax5, h5 + h5_r, l5 + l5_r)
    ax5.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax5.set_xlabel("Time of day")
    _panel_tag(ax5, "(e) BESS Action & SOC")
    b_ev = _fit(res["Baseline(kW)"], n)
    ax6.step(x, p_ev, where="post", color="#1f77b4", linewidth=2.2, alpha=0.9, label="p_EV")
    ax6.step(x, b_ev, where="post", color="#6c757d", linewidth=1.8, linestyle="--", alpha=0.95, label="Baseline")
    ax6.fill_between(x, p_ev, b_ev, where=(p_ev >= b_ev), step="post", color="#f4a261", alpha=0.20, label="pEV > Baseline")
    ax6.fill_between(x, p_ev, b_ev, where=(p_ev < b_ev), step="post", color="#90caf9", alpha=0.20, label="pEV < Baseline")
    ax6.step(x, p_bess, where="post", color="#d62728", linewidth=2.0, alpha=0.9, label="p_BESS_net")
    ax6.step(x, p_gi, where="post", color="#2ca02c", linewidth=2.0, alpha=0.9, label="p_GI")
    ax6.axhline(0, color="black", linewidth=0.9)
    ncd_val = M_Th_NCD[threshold_case_idx]
    pd_val = M_Th_PD[threshold_case_idx]
    ax6.axhline(ncd_val, color="#ff9f1c", linestyle="--", alpha=0.8, linewidth=1.5, label=f"NCD Threshold")
    ax6.axhline(pd_val, color="#e76f51", linestyle="--", alpha=0.8, linewidth=1.5, label=f"PD Threshold")
    ax6.set_ylabel("Net Power (kW)")
    ax6.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax6.set_xlabel("Time of day")
    _smart_legend(ax6)
    _panel_tag(ax6, "(f) Overall System Power")
    def _to_scalar(v):
        if np.isscalar(v):
            return float(v)
        arr = np.asarray(v, dtype=float).flatten()
        return float(arr[0]) if arr.size > 0 else np.nan
    if globals().get("SHOW_6PANEL_SUMMARY_BOX", False):
        da_summary = (
            f"{TARGET_SAVE} Summary:\n"
            f"CostOpt=${_to_scalar(res['Cost_Opt']):.2f}, CostPD=${_to_scalar(res['Cost_PD']):.2f}, "
            f"CostNCD=${_to_scalar(res['Cost_NCD']):.2f}, CostTOU=${_to_scalar(res['Cost_TOU']):.2f}\n"
            f"RevenueEV=${_to_scalar(res['Revenue_EV']):.2f}, RevenueWM=${_to_scalar(res['Revenue_WM']):.2f}"
        )
        ax6.text(
            0.5,
            -0.45,
            da_summary,
            transform=ax6.transAxes,
            ha="center",
            va="top",
            fontsize=12,
            weight="normal",
            linespacing=1.5,
            bbox=dict(facecolor="#f8f9fa", alpha=0.9, edgecolor="lightgrey", boxstyle="round,pad=0.8"),
        )
    def align_yaxis(ax_left, ax_right, margin_factor=1.45):
        y1_min, y1_max = ax_left.get_ylim()
        y2_min, y2_max = ax_right.get_ylim()
        new_min = min(y1_min, y2_min)
        new_max = max(y1_max, y2_max)
        ax_left.set_ylim(new_min * margin_factor if new_min < 0 else new_min / margin_factor, new_max * margin_factor)
        ax_right.set_ylim(new_min * margin_factor if new_min < 0 else new_min / margin_factor, new_max * margin_factor)
    align_yaxis(ax1, ax2, margin_factor=1.45)
    align_yaxis(ax3, ax4, margin_factor=1.45)
    fig01 = plot_dir / f"{TARGET_SAVE}_6panel_{run_tag}.png"
    fig.savefig(fig01, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)
    return [str(fig01), str(activity_csv)]
def save_old_stairplot_beautified(
    H_Start_RT,
    TimeSeries_real,
    dt_m_EV,
    dt_h,
    EnergyDemand_Step_V0G_TheDates,
    EnergyDemand_Step_V1G_TheDates,
    p_GI_DA,
    dispatch_t0,
    Dispatch,
    p_BESS_value_RT,
    Baseline_96,
    EventHour_96,
    D_Th_NCD_96,
    D_Th_PD_96,
    Fc_SessionkWh,
    Fc_NumbEV,
    Fc_AtArrival,
    WM_Mode,
    cycle_label=None,
):
    from pathlib import Path
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    t_core = pd.to_datetime(TimeSeries_real[0])
    t_edges = TimeSeries_real[0].copy()
    t_edges.append(TimeSeries_real[0][-1] + timedelta(minutes=dt_m_EV))
    pw_v0g_max = np.asarray(EnergyDemand_Step_V0G_TheDates[0], dtype=float) / dt_h
    pw_v1g = np.asarray(EnergyDemand_Step_V1G_TheDates[0], dtype=float) / dt_h
    pw_opt_da = np.asarray(p_GI_DA, dtype=float)
    pw_opt_t0 = np.asarray(dispatch_t0[0], dtype=float)
    pw_opt_imp_base = np.asarray((Dispatch[0][0].sum(axis=1)) / dt_h + p_BESS_value_RT[0, :], dtype=float)
    pw_opt_imp_case1 = np.asarray((Dispatch[0][1].sum(axis=1)) / dt_h + p_BESS_value_RT[1, :], dtype=float)
    pw_baseline_base = np.asarray(Baseline_96[0].reshape(96) / dt_h, dtype=float)
    pw_baseline_case1 = np.asarray(Baseline_96[1].reshape(96) / dt_h, dtype=float)
    def _stairs(y):
        y = np.asarray(y, dtype=float)
        return np.concatenate(([y[0]], y))
    event_base = np.asarray(EventHour_96[0]).reshape(-1)
    event_case1 = np.asarray(EventHour_96[1]).reshape(-1)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.step(t_edges, _stairs(pw_opt_da), where="post", color="#0b5ed7", linewidth=2.8, label="V1G opt DA (100%)")
    ax.step(t_edges, _stairs(pw_opt_t0), where="post", color="#5bc0eb", linewidth=2.4, label="V1G opt RT t0 (100%)")
    ax.step(t_edges, _stairs(pw_opt_imp_base), where="post", color="#c1121f", linestyle="--", linewidth=2.2, label="V1G opt imp (100%)")
    ax.step(t_edges, _stairs(pw_opt_imp_case1), where="post", color="#f28482", linestyle="--", linewidth=2.2, label="V1G opt imp (eta%)")
    ax.step(t_edges, _stairs(pw_baseline_base), where="post", color="#2a9d8f", linestyle=":", linewidth=2.0, label="Baseline (100%)")
    ax.step(t_edges, _stairs(pw_baseline_case1), where="post", color="#8ecae6", linestyle=":", linewidth=2.0, label="Baseline (eta%)")
    ax.step(t_edges, _stairs(pw_v0g_max), where="post", color="#588157", linewidth=1.8, alpha=0.9, label="V0G")
    ax.step(t_edges, _stairs(pw_v1g), where="post", color="#a7c957", linewidth=1.8, alpha=0.9, label="V1G real")
    ymax = max(
        np.max(pw_opt_da),
        np.max(pw_opt_t0),
        np.max(pw_opt_imp_base),
        np.max(pw_opt_imp_case1),
        np.max(pw_baseline_base),
        np.max(pw_baseline_case1),
        np.max(pw_v0g_max),
        np.max(pw_v1g),
    )
    ymin = min(
        np.min(pw_opt_da),
        np.min(pw_opt_t0),
        np.min(pw_opt_imp_base),
        np.min(pw_opt_imp_case1),
        np.min(pw_baseline_base),
        np.min(pw_baseline_case1),
        np.min(pw_v0g_max),
        np.min(pw_v1g),
    )
    marker_level_base = ymax * 0.96
    marker_level_case1 = ymax * 0.92
    ax.scatter(t_core[event_base > 0.5], np.full(np.sum(event_base > 0.5), marker_level_base), marker="x", color="#d00000", s=28, label="Event hour (100%)")
    ax.scatter(t_core[event_case1 > 0.5], np.full(np.sum(event_case1 > 0.5), marker_level_case1), marker="+", color="#2b9348", s=28, label="Event hour (eta%)")
    ax.plot(t_core[:5], np.full(5, D_Th_NCD_96[0][0]), color="#ff9f1c", marker="8", linestyle="", markersize=6, label="NCD start (100%)")
    ax.plot(t_core[:5], np.full(5, D_Th_NCD_96[1][0]), color="#2a9d8f", marker=">", linestyle="", markersize=5, label="NCD start (eta%)")
    ax.plot(t_core[:5], np.full(5, D_Th_PD_96[0][0]), color="#ff9f1c", marker="s", linestyle="", markersize=6, label="PD start (100%)")
    ax.plot(t_core[:5], np.full(5, D_Th_PD_96[1][0]), color="#2a9d8f", marker="<", linestyle="", markersize=5, label="PD start (eta%)")
    ax.set_title(f"Implemented EV + BESS Dispatch on {H_Start_RT.strftime('%Y/%m/%d')}", fontsize=18, fontweight="bold")
    if cycle_label is not None:
        ax.text(
            0.01,
            0.98,
            f"Cycle count: {cycle_label}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=11,
            fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="lightgrey", boxstyle="round,pad=0.25"),
        )
    ax.set_xlabel("Time", fontsize=14)
    ax.set_ylabel("Power Demand (kW)", fontsize=14)
    ax.set_xlim(pd.to_datetime(t_edges[0]), pd.to_datetime(t_edges[-1]))
    ax.set_ylim(ymin - 0.08 * (ymax - ymin + 1e-9), ymax + 0.12 * (ymax - ymin + 1e-9))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.grid(axis="x", linestyle=":", alpha=0.20)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=11, frameon=False)
    fig.tight_layout()
    out_dir = Path("Results/Plots/Imp_Stair") / f"{H_Start_RT.strftime('%Y')}_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{WM_Mode}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{H_Start_RT.strftime('%Y%m%d')}_Implemented_Pw_stair.png"
    fig.savefig(out_path, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)
    return str(out_path)

Main Loop


In [5]:
# clean all csv files presaved 
base_dir = Path('Results/Dispatch')
RUN_MAIN_LOOP_DIRECT = False  # False: define/load only; True: run this block directly
for f in (base_dir.glob('[01]*') if RUN_MAIN_LOOP_DIRECT else []):
    f.unlink(missing_ok=True)
prefix = f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{WM_Mode}'
_clear_impl_at_start = bool(globals().get('CLEAR_IMPLEMENTATION_AT_RUN_START', True))
if RUN_MAIN_LOOP_DIRECT and _clear_impl_at_start:
    for suffix in ['implementation.csv', 'daily_summary.csv', 'daily_cost.csv']:
        (base_dir / f'{prefix}_{suffix}').unlink(missing_ok=True)
if RUN_MAIN_LOOP_DIRECT:
    if _clear_impl_at_start:
        print("All dispatch files from last implementation have been deleted.\n")
    else:
        print("Existing implementation files preserved for continuous baseline history.\n")
else:
    print("RUN_MAIN_LOOP_DIRECT=False: main loop block loaded without execution.")
# When preserving implementation history, keep only rows before the first requested
# simulation day. This allows a standalone July rerun to learn from June dispatch
# without accidentally reading stale July rows as historical baseline data.
def _trim_existing_csv_before_run_start(out_path, first_timestamp):
    out_path = Path(out_path)
    if (not RUN_MAIN_LOOP_DIRECT) or _clear_impl_at_start or (not out_path.exists()):
        return
    try:
        _df_existing = pd.read_csv(out_path, low_memory=False)
        if 'Interval start' in _df_existing.columns:
            _row_time = pd.to_datetime(_df_existing['Interval start'], errors='coerce')
        elif 'Date' in _df_existing.columns:
            _row_time = pd.to_datetime(_df_existing['Date'].astype(str), format='%Y%m%d', errors='coerce')
        else:
            return
        _rows_before = len(_df_existing)
        _df_existing = _df_existing[_row_time < pd.Timestamp(first_timestamp)]
        _rows_after = len(_df_existing)
        _df_existing.to_csv(out_path, index=False)
    except Exception as _trim_err:
        print(f"Warning: could not trim existing implementation history in {out_path}: {_trim_err}")
# Original helper for incremental CSV writes. It is re-defined later in the notebook
# to enforce two-decimal rounding and a consistent float format.
def append_df_to_csv(df, out_path, index=False):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        df.to_csv(out_path, mode='a', index=index, header=False)
    else:
        df.to_csv(out_path, mode='w', index=index, header=True)
# Read and parse AS files once to avoid repeated daily I/O.
dir_AS_DA = os.path.join("2025Data/AS_DAM/")
dir_AS_RT = os.path.join("2025Data/AS_RTM/")
Data_AS_DA_all = pd.read_csv(dir_AS_DA + 'AS_price_2025_clear.csv')
Data_AS_RT_all = pd.read_csv(dir_AS_RT + 'AS_price_2025_clear.csv')
# AS timestamps span daylight-saving transitions; parse via UTC to keep .dt usable.
Data_AS_DA_all["datetime"] = pd.to_datetime(Data_AS_DA_all["datetime"], utc=True).dt.tz_convert("America/Los_Angeles")
Data_AS_RT_all["datetime"] = pd.to_datetime(Data_AS_RT_all["datetime"], utc=True).dt.tz_convert("America/Los_Angeles")
if Fc_AtArrival == 'MLatArrival':
    # Cache this lookup table once to avoid repeated per-car file reads.
    UserBess = pd.read_csv("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Data/PowerFlex input for EV statistics to be plugged into master sheet.csv")
# Main loop for monthly optimization with implementation
Cost_All = pd.DataFrame()
start_time = time.time()  # record start time
RUN_MONTHS_CONFIG = list(globals().get('RUN_MONTHS_CONFIG', [7]))
test_month = int(RUN_MONTHS_CONFIG[0])
SOC_BESS_last_step = np.ones(len(Cases)) * 0.5  # initial SOC at 50%
SOC_BESS_daily_end = np.ones(len(Cases)) * 0.5
for month in (RUN_MONTHS_CONFIG if RUN_MAIN_LOOP_DIRECT else []):
    print("Test month:", month)
    num_days = calendar.monthrange(year,month)[1] 
    run_days = list(RUN_DAYS_CONFIG)  # manually configured at block start
    run_days = [d for d in run_days if 1 <= d <= num_days]
    if len(run_days) == 0:
        raise ValueError('run_days is empty after month/day bounds check.')
    # Each study month is evaluated as an independent billing/control period.
    # Reset BESS carry-over state so running [6, 7] gives the same July initial
    # condition as running [7] alone.
    SOC_BESS_daily_end = np.ones(len(Cases)) * 0.5
    Data_AS_DA_month = Data_AS_DA_all[(Data_AS_DA_all["datetime"].dt.year == year) & (Data_AS_DA_all["datetime"].dt.month == month)]
    Data_AS_RT_month = Data_AS_RT_all[(Data_AS_RT_all["datetime"].dt.year == year) & (Data_AS_RT_all["datetime"].dt.month == month)]
    # NOTE: Initialize M_Th to 0 at the start of each month.
    # SDGE demand charge resets each month; start from 0 and let rolling max accumulate.
    _init_ncd = 0.0
    _init_pd = 0.0
    M_Th_NCD = [_init_ncd for _ in range(len(Cases)+1)]
    M_Th_PD = [_init_pd for _ in range(len(Cases)+1)]
    print(f'Month {month}: M_Th_NCD init={_init_ncd:.1f} kW, M_Th_PD init={_init_pd:.1f} kW (initialized to 0)')
    M_Th_NCD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_NCD_list_96 = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list_96 = [[] for _ in range(len(Cases)+1)]
    M_V0G = pd.DataFrame()
    M_V1G = pd.DataFrame()
    M_Opt_Offline = pd.DataFrame()
    M_Opt_Imp = [pd.DataFrame() for _ in range(len(Cases))] #pd.DataFrame()
    M_t = pd.DataFrame()
    M_ED_V0G = []
    M_ED_V1G = []
    M_ED_Opt_DA = []
    M_ED_Opt_t0 = []
    M_ED_Opt_base = []
    M_ED_Opt_case1 = []
    Status_list_DA = []
    for Day in tqdm(run_days, desc="Processing days"):
        revenue_WM_daily = np.zeros(2)
        revenue_DR_daily = np.zeros(2)
        is_last_run_day = (Day == run_days[-1])
        TheDate_Day0 = datetime(year, month, Day)   
        TheDate_Day0_Start = TheDate_Day0 + timedelta(seconds=0)
        TheDate_Day0_End = TheDate_Day0 + timedelta(hours=24)
        if len(Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0])>0: 
            Order = Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekdaysWOH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekdaysWOH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekdaysWOH[Order-3].iloc[0]
        else:
            Order = Y_WeekendsWH[Y_WeekendsWH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekendsWH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekendsWH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekendsWH[Order-3].iloc[0]
        TheDate_Dayb1_Start = TheDate_Dayb1 + timedelta(seconds=0)
        TheDate_Dayb1_End   = TheDate_Dayb1 + timedelta(hours=24)
        TheDate_Dayb2_Start = TheDate_Dayb2 + timedelta(seconds=0)
        TheDate_Dayb2_End   = TheDate_Dayb2 + timedelta(hours=24)
        TheDate_Dayb3_Start = TheDate_Dayb3 + timedelta(seconds=0)
        TheDate_Dayb3_End   = TheDate_Dayb3 + timedelta(hours=24)
        TheDate        = [TheDate_Day0, TheDate_Dayb1, TheDate_Dayb2, TheDate_Dayb3]
        TheDates_Start = [TheDate_Day0_Start, TheDate_Dayb1_Start, TheDate_Dayb2_Start, TheDate_Dayb3_Start]
        TheDates_End   = [TheDate_Day0_End, TheDate_Dayb1_End, TheDate_Dayb2_End, TheDate_Dayb3_End]
        if DAM == 0:
            Bid_Pr_DA = np.zeros(96)
            Bid_Pr_RT = np.zeros(96)
            Baseline_Opt_avg = np.empty((3, 3, 24))
        # Read the LMP data
        dir_Input_DA = os.path.join("2025Data/LMP/2025/DA/")
        dir_Input_RT = os.path.join("2025Data/LMP/2025/FM/")
        filename_Input = glob.glob(dir_Input_DA + TheDate_Day0_Start.strftime('%Y%m%d') + '*.csv')
        Data_LMP_DA = pd.read_csv(''.join(filename_Input))
        filename_Input = glob.glob(dir_Input_RT + TheDate_Day0_Start.strftime('%Y%m%d') + '*.csv')
        Data_LMP_RT = pd.read_csv(''.join(filename_Input))
        Data_LMP_DA = Data_LMP_DA[Data_LMP_DA['LMP_TYPE']=='LMP']
        Data_LMP_RT = Data_LMP_RT[Data_LMP_RT['LMP_TYPE']=='LMP']
        # sort the 'INTERVALSTARTTIME_GMT'
        LMP_IntervalStartGMT_DA = Data_LMP_DA['INTERVALSTARTTIME_GMT']
        LMP_IntervalStartGMT_RT = Data_LMP_RT['INTERVALSTARTTIME_GMT']
        # sort LMP data based on the 'INTERVALSTARTTIME_GMT'    
        A = pd.to_datetime(LMP_IntervalStartGMT_DA)
        B = [k for k in range(len(A))]
        LMPOrder = [x for _, x in sorted(zip(A, B))]
        Data_LMP_DA_ = Data_LMP_DA.iloc[ LMPOrder, :]
        A = pd.to_datetime(LMP_IntervalStartGMT_RT)
        B = [k for k in range(len(A))]
        LMPOrder = [x for _, x in sorted(zip(A, B))]
        Data_LMP_RT_ = Data_LMP_RT.iloc[ LMPOrder, :]
        Bid_Pr_DA = np.array(Data_LMP_DA_['MW'].repeat(4)*0.001) #repeat 4 times of the 24 x 1 series, $/MWh ---> $/kWh   
        Bid_Pr_RT = (np.average(np.array(Data_LMP_RT_['VALUE']).reshape(-1, 3), axis=1)*0.001) #ave every 3 entries of the 288  x 1 series, $/MWh ---> $/kWh
        # Read the Ancillary Service prices data
        Data_AS_DA = Data_AS_DA_month[Data_AS_DA_month["datetime"].dt.day == Day].reset_index(drop=True)
        Data_AS_RT = Data_AS_RT_month[Data_AS_RT_month["datetime"].dt.day == Day].reset_index(drop=True)
        # Convert DA hourly AS prices to 15-min interval prices
        AS_Pr_RU_DA = np.repeat(Data_AS_DA["RegUp"].values, 4)
        AS_Pr_RD_DA = np.repeat(Data_AS_DA["RegDown"].values, 4)
        AS_Pr_SP_DA = np.repeat(Data_AS_DA["Spin"].values, 4)
        AS_Pr_NSP_DA = np.repeat(Data_AS_DA["NonSpin"].values, 4)
        # RT values does not need to be converted
        AS_Pr_RU_RT = Data_AS_RT["RegUp"].values
        AS_Pr_RD_RT = Data_AS_RT["RegDown"].values
        AS_Pr_SP_RT = Data_AS_RT["Spin"].values
        AS_Pr_NSP_RT = Data_AS_RT["NonSpin"].values
        # divide all by 1000 to convert $/MWh to $/kWh
        AS_Pr_RU_DA = AS_Pr_RU_DA * 0.001 # NOTE: only for test
        AS_Pr_RD_DA = AS_Pr_RD_DA * 0.001
        AS_Pr_SP_DA = AS_Pr_SP_DA * 0.001
        AS_Pr_NSP_DA = AS_Pr_NSP_DA * 0.001
        AS_Pr_RU_RT = AS_Pr_RU_RT * 0.001
        AS_Pr_RD_RT = AS_Pr_RD_RT * 0.001
        AS_Pr_SP_RT = AS_Pr_SP_RT * 0.001
        AS_Pr_NSP_RT = AS_Pr_NSP_RT * 0.001
        # concat last loop's implementation results with baseline
        dir_Output = os.path.join('Results/Dispatch/' +
                                    str(year) +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode  +'_implementation.csv')
        _trim_existing_csv_before_run_start(dir_Output, TheDate_Day0)
        path = Path(dir_Output)
        if path.exists():
            Dispatch_implemented_lastloop = pd.read_csv(dir_Output, low_memory=False, header = 0)
            Dispatch_implemented_lastloop['Interval start'] = pd.to_datetime(Dispatch_implemented_lastloop['Interval start'])
            Dispatch_2025 = pd.concat([Dispatch_2025_baseline, Dispatch_implemented_lastloop], axis=0)
            Dispatch_2025 = Dispatch_2025.drop_duplicates('Interval start',keep='last')
        else:
            if Day == run_days[0]:
                Dispatch_2025 = Dispatch_2025_baseline
            else:
                print("No previous implementation file, exit!\n")
                sys.exit()
        dispatch_day_cache = {}
        dispatch_day_hour_cache = {}
        def _get_dispatch_day(this_day):
            day_key = pd.Timestamp(this_day).normalize()
            if day_key not in dispatch_day_cache:
                start = day_key
                end = day_key + timedelta(days=1)
                dispatch_day_cache[day_key] = Dispatch_2025[(Dispatch_2025['Interval start']>=start)&(Dispatch_2025['Interval start']<end)].reset_index()
            return dispatch_day_cache[day_key]
        def _get_dispatch_day_hour(this_day, i_hour):
            key = (pd.Timestamp(this_day).normalize(), int(i_hour))
            if key not in dispatch_day_hour_cache:
                day_df = _get_dispatch_day(this_day)
                hour_df = day_df[day_df['Interval start'].dt.hour == i_hour]
                dispatch_day_hour_cache[key] = hour_df.drop_duplicates('Interval start',keep='last')
            return dispatch_day_hour_cache[key]
        # Read baseline data
        days = 3
        Baseline_cases = [f'{case} [kWh]' for case in Cases] + ['V0G [kWh]']  # active RT cases plus DA/V0G reference
        Baseline_Opt_TheDates = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)]  # Base case (100% service level), Case1 (eta%), and V0G for Day0. Day-1. and Day-2
        Baseline_days = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)] # Base case, Case1, and V0G for Day0. Day-1. and Day-2
        Baseline_Opt_all = [[pd.Series() for _ in range(len(Baseline_cases))] for _ in range(days)] 
        Baseline_Opt_avg = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)] 
        for i_d in range(days): #Day0. Day-1. and Day-2
            # if TheDate is a weekend day, or a holiday---> previous 4 weekend days                
            if TheDate[i_d] in list(Y_WeekendsWH) :
                NumbBaselineDays = 4
                CatagoryBaselineDays = Y_WeekendsWH
            else: # TheDate[i_d] in list(Y_WeekdaysWOH)
                NumbBaselineDays = 10
                CatagoryBaselineDays = Y_WeekdaysWOH
            for i_case in range(len(Baseline_cases)): # Base case (100% service level), Case1 (eta%), and V0G
                if i_case < len(Cases): # RT cases with market participation; event days can differ by hour
                    Baseline_days[i_d][i_case] = [[] for _ in range(24)]
                    Baseline_Opt_all[i_d][i_case] = [[] for _ in range(24)]
                    for i_hour in range(24):
                        for j in range(45):
                            ThisDay = TheDate[i_d] - timedelta(days=j+1)
                            if ThisDay in list(CatagoryBaselineDays):
                                Dispatch_2025_ThisDate_ThisHour = _get_dispatch_day_hour(ThisDay, i_hour)
                                # check if this hour this day is an event hour, if NOT an event hour, count the value
                                # TODO: 2 cases baseline always same with DA results, but should be different using implementation as baseline after 45 days running
                                if i_case == 0:
                                    Event = 'event hour_Base'
                                else:
                                    Event = 'event hour_Case1'
                                if Dispatch_2025_ThisDate_ThisHour.empty or (Event not in Dispatch_2025_ThisDate_ThisHour.columns):
                                    continue
                                if Dispatch_2025_ThisDate_ThisHour[Event].iloc[0] == 0:
                                    Baseline_days[i_d][i_case][i_hour] = Baseline_days[i_d][i_case][i_hour] + [ThisDay]
                                    avg = sum(Dispatch_2025_ThisDate_ThisHour[Baseline_cases[i_case]])/4
                                    Baseline_Opt_all[i_d][i_case][i_hour] = Baseline_Opt_all[i_d][i_case][i_hour] + [avg]
                            if len(Baseline_Opt_all[i_d][i_case][i_hour]) == NumbBaselineDays:
                                Baseline_Opt_avg[i_d][i_case] = Baseline_Opt_avg[i_d][i_case] + [float(np.mean(Baseline_Opt_all[i_d][i_case][i_hour])) if len(Baseline_Opt_all[i_d][i_case][i_hour]) > 0 else 0.0]
                                break
                            # if after going thru 45 days still cannot find the 'non event hour day', 
                            # just average howmany hour days 'Baseline_Opt_all[i_d][i_case][i_hour]' has
                            if j==44:
                                Baseline_Opt_avg[i_d][i_case] = Baseline_Opt_avg[i_d][i_case] + [float(np.mean(Baseline_Opt_all[i_d][i_case][i_hour])) if len(Baseline_Opt_all[i_d][i_case][i_hour]) > 0 else 0.0]
                else: # for i_case==2 no market participation
                    for j in range(45):
                        ThisDay = TheDate[i_d] - timedelta(days=j+1)
                        if ThisDay in list(CatagoryBaselineDays):
                            Dispatch_2025_ThisDate = _get_dispatch_day(ThisDay)
                            Baseline_days[i_d][i_case] = Baseline_days[i_d][i_case] + [ThisDay]
                            Baseline_Opt_all[i_d][i_case] = pd.concat([Baseline_Opt_all[i_d][i_case],Dispatch_2025_ThisDate[Baseline_cases[i_case]]],axis = 1,ignore_index = True)
                        if len(Baseline_days[i_d][i_case]) == NumbBaselineDays:
                            break
                    _baseline_samples = len(Baseline_days[i_d][i_case])
                    _baseline_matrix = Baseline_Opt_all[i_d][i_case]
                    if _baseline_samples > 0 and not _baseline_matrix.empty:
                        _baseline_matrix = _baseline_matrix.drop(0, axis=1, errors='ignore')
                        if _baseline_matrix.shape[1] > 0:
                            _interval_avg = np.array(_baseline_matrix.sum(axis=1)) / _baseline_matrix.shape[1]
                            Baseline_Opt_avg[i_d][i_case] = np.nan_to_num(np.average(_interval_avg.reshape(-1, 4), axis=1), nan=0.0, posinf=0.0, neginf=0.0) # ave every 4 entries of the (96, ) array
                        else:
                            Baseline_Opt_avg[i_d][i_case] = np.zeros(24)
                    else:
                        Baseline_Opt_avg[i_d][i_case] = np.zeros(24)
        # Choose data for Day0, Dayb1, and Dayb2
        ThisDates = []
        Data_TheDates = []       
        TimeSeries_TheDates = []
        Time_table_TheDates = []
        EnergyDemand_Table_V0G_TheDates = []
        EnergyDemand_Table_V1G_TheDates = [] 
        EnergyDemand_Step_V0G_TheDates = []
        EnergyDemand_Step_V1G_TheDates = []
        ArrivalTime_table_TheDates = []
        ArrivalTime_table_1_TheDates = []
        ArrivalTime_table_2_TheDates = []
        Car_table_TheDates = []
        Car_table_1_TheDates = []
        Car_table_2_TheDates = []
        Eta_min_table_TheDates = []
        Eta_min_table_1_TheDates = []
        Eta_min_table_2_TheDates = []
        IntervalkWh_max_TheDates = []
        IntervalkWh_max_1_TheDates = []
        IntervalkWh_max_2_TheDates = []
        SessionkWh_table_TheDates = []
        SessionkWh_table_1_TheDates = []
        SessionkWh_table_2_TheDates = []        
        Type_table_TheDates = []
        Type_table_1_TheDates = []
        Type_table_2_TheDates = []
        D_Th_NCD_96 = [[],[]]
        D_Th_PD_96  = [[],[]]
        for i_d in range(3):
            # Create Time Series for the Day                  
            start_ind = TheDates_Start[i_d]
            end_ind = TheDates_End[i_d]
            TimeSeries_ThisDate = []
            while start_ind < end_ind:
                TimeSeries_ThisDate.append(start_ind)
                start_ind += interval
            # Filter the data based on the Date you choose
            Data_ThisDate = Data[(TheDates_Start[i_d] <= Data['Interval start']) & (Data['Interval start'] < TheDates_End[i_d])]
            # if no data on the day, walk backward through the available lookback days.
            if len(Data_ThisDate) == 0:
                print("No data for {0}, search earlier lookback days".format(TheDates_Start[i_d]))
                for _fallback_i in range(i_d + 1, len(TheDates_Start)):
                    _fallback_data = Data[(TheDates_Start[_fallback_i] <= Data['Interval start']) & (Data['Interval start'] < TheDates_End[_fallback_i])]
                    if len(_fallback_data) > 0:
                        start_ind = TheDates_Start[_fallback_i]
                        end_ind = TheDates_End[_fallback_i]
                        TimeSeries_ThisDate = []
                        while start_ind < end_ind:
                            TimeSeries_ThisDate.append(start_ind)
                            start_ind += interval 
                        Data_ThisDate = _fallback_data
                        print("  using fallback data from {0}".format(TheDates_Start[_fallback_i]))
                        break
            Data_TheDates = Data_TheDates + [Data_ThisDate]
            TimeSeries_TheDates = TimeSeries_TheDates + [TimeSeries_ThisDate]    
            Time_table_TheDates = Time_table_TheDates + [pd.DataFrame(TimeSeries_ThisDate, columns=["Interval start"]) ]  
            #################################################################################
            # Availability table (Time series x Vehicles)
            ################################################################################# 
            AllCar_table_max_charger = pd.DataFrame(columns=["Interval start"])
            AllCar_table_float_V0G = pd.DataFrame(columns=["Interval start"])
            AllCar_table_float_V1G = pd.DataFrame(columns=["Interval start"])
            Type_list = []
            SessionkWh_list = []
            ArrivalTime_list = []
            Eta_min_list = []
            Cars = Data_ThisDate["Car_"].unique()
            Cars2 = []
            if Fc_AtArrival == 'MLatArrival' :   
                #store Day 0, real session data for ML forecast inputs
                if i_d == 0:               
                    List = ['Car', 'User','Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']
                    Sess_Car = []
                    Sess_User = []
                    Sess_SessionStart = []
                    Sess_ArrivalHr = []
                    Sess_BESS = []
                    Sess_Weekday = []
                    Sess_MaxPw = []
            for i in range(len(Cars)):
                CarName = Cars[i]      
                #Check for Users with no sessions creating empty dataframes 
                data_Merge_ThisDate_ThisCar_Test = Data_ThisDate[Data_ThisDate["Car_"] == CarName]
                rows, columns =  data_Merge_ThisDate_ThisCar_Test.shape
                if rows > 0: # 2
                    data_Merge_ThisDate_ThisCar = Data_ThisDate[Data_ThisDate["Car_"] == CarName]
                    data_Merge_ThisDate_ThisCar = data_Merge_ThisDate_ThisCar.sort_values('Interval start')
                    # Round up/down the arrival/departure time for V1G, V0G, and Opt
                    # NOTE: if there are > 1 sessions for this car?
                    if data_Merge_ThisDate_ThisCar["Session start"].nunique() == 1:
                        filtered = data_Merge_ThisDate_ThisCar[
                            (data_Merge_ThisDate_ThisCar["Interval start"] >= data_Merge_ThisDate_ThisCar["Session start"]) &
                            (data_Merge_ThisDate_ThisCar["Interval end"] <= data_Merge_ThisDate_ThisCar["Session end"])
                        ]
                        data_Merge_ThisDate_ThisCar = filtered
                        AvailableSlots = len(data_Merge_ThisDate_ThisCar)
                    elif data_Merge_ThisDate_ThisCar["Session start"].nunique() >= 2:
                        for sess in range(data_Merge_ThisDate_ThisCar["Session start"].nunique()):
                            filtered = data_Merge_ThisDate_ThisCar[
                                (data_Merge_ThisDate_ThisCar["Interval start"] >= data_Merge_ThisDate_ThisCar["Session start"].unique()[sess]) &
                                (data_Merge_ThisDate_ThisCar["Interval end"] <= data_Merge_ThisDate_ThisCar["Session end"].unique()[sess])
                            ]
                            if sess == 0:
                                data_Merge_ThisDate_ThisCar_tmp = filtered
                            else:
                                data_Merge_ThisDate_ThisCar_tmp = pd.concat([data_Merge_ThisDate_ThisCar_tmp, filtered], axis=0)
                        data_Merge_ThisDate_ThisCar = data_Merge_ThisDate_ThisCar_tmp.sort_values('Interval start')
                        AvailableSlots = len(data_Merge_ThisDate_ThisCar)
                    if AvailableSlots == 0:
                        continue # if no available slots after filtering, skip this car
                    Cars2 = Cars2 + [CarName]
                    # NOTE: 2025 data doesnt have Tesla Type, but still have Tesla cars with higher charging power
                    if (data_Merge_ThisDate_ThisCar['Interval kWh'] > 1.664).any() or (data_Merge_ThisDate_ThisCar['Interval max demand kW'] > 1.664*4).any():
                        Type = 'Tesla'
                    elif (data_Merge_ThisDate_ThisCar['Interval kWh'] <= 1.664).all() and (data_Merge_ThisDate_ThisCar['Interval max demand kW'] <= 1.664*4).all():
                        Type = 'NonTesla'
                    else:
                        print("Error: not sure if Tesla or NonTesla, exit!")
                        sys.exit()
                    Type_list = Type_list + [Type]        
                    if Type == 'Tesla':
                        IntervalkWh_CH_max = 4.16
                    else:
                        IntervalkWh_CH_max = 1.664
                    # adjust intervalkWh_V1G if SessionkWh_V1G > numb of available intervals * IntervalkWh_CH_max 
                    SessionkWh = sum(data_Merge_ThisDate_ThisCar['Interval kWh'])
                    if SessionkWh > AvailableSlots*IntervalkWh_CH_max:
                        ratio = AvailableSlots*IntervalkWh_CH_max/SessionkWh
                        # modify interval data (decrease V1G)
                        data_Merge_ThisDate_ThisCar['Interval kWh'] = data_Merge_ThisDate_ThisCar['Interval kWh']*ratio
                        #also need to modify session data (decrease SessionkWh)
                        SessionkWh = SessionkWh*ratio
                    SessionkWh_list = SessionkWh_list + [SessionkWh]
                    # there might be two session start!!
                    ArrivalTime = data_Merge_ThisDate_ThisCar["Session start"].sort_values(ascending=True).iloc[0]
                    ArrivalTime_list = ArrivalTime_list + [ArrivalTime]
                    # Eta_min
                    Eta_min = data_Merge_ThisDate_ThisCar['Eta'].iloc[0]
                    Eta_min_list = Eta_min_list + [Eta_min]
                    # Interval time series during plug-in time
                    IntervalStart_ThisCar_round = pd.to_datetime(data_Merge_ThisDate_ThisCar["Interval start"])
                    # CHarger capacity: IntervalkWh_ThisCar_CH
                    IntervalkWh_ThisCar_CH = pd.Series(range(AvailableSlots))*0 + IntervalkWh_CH_max            
                    ThisCar_table_float_max_charger = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_CH)), columns=["Interval start", CarName])            
                    ThisCar_table_float_max_charger_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_float_max_charger, how="left", on=["Interval start"])
                    AllCar_table_max_charger = AllCar_table_max_charger.merge(ThisCar_table_float_max_charger_,  on="Interval start", how='outer')
                    if Fc_AtArrival == 'MLatArrival' : 
                        #store Day 0, real session data for ML forecast inputs
                        if i_d == 0:
                            Sess_Car = Sess_Car + [CarName]
                            Sess_User = Sess_User + [data_Merge_ThisDate_ThisCar['User'].iloc[0]]
                            ArrivalTime = data_Merge_ThisDate_ThisCar['Session start'].iloc[0]
                            Sess_SessionStart = Sess_SessionStart + [ArrivalTime] 
                            Sess_ArrivalHr = Sess_ArrivalHr + [ArrivalTime.hour + ArrivalTime.minute/60]
                            # based on 'User', import BESS info from Byron's master sheet
                            BESS_kWh = UserBess['Battery (kWh)'][UserBess['Doe Id']==int(Sess_User[0])]
                            if len(BESS_kWh)==0:
                                BESS_kWh = 48 #which is the avg BESS kWh of the master sheet
                                UserNoBess = UserNoBess + [data_Merge_ThisDate_ThisCar['User'].iloc[0]]
                            Sess_BESS = Sess_BESS + [BESS_kWh]
                            Sess_Weekday = Sess_Weekday + [ArrivalTime.weekday()] #monday is 0
                            Sess_MaxPw = Sess_MaxPw + [IntervalkWh_CH_max*4]
                    # V0G
                    IntervalkWh_ThisCar_V0G = IntervalkWh_ThisCar_CH
                    IntervalkWh_left = SessionkWh + 0
                    for j in range(len(IntervalkWh_ThisCar_V0G)):
                        # j = 0
                        if IntervalkWh_left >= IntervalkWh_ThisCar_CH.iloc[j]:
                            IntervalkWh_left = IntervalkWh_left - IntervalkWh_ThisCar_CH.iloc[j]
                        else:
                            IntervalkWh_ThisCar_V0G.iloc[j] = IntervalkWh_left
                            IntervalkWh_ThisCar_V0G.iloc[j+1:] = 0
                            break
                    ThisCar_table_V0G = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_V0G)), columns=["Interval start", CarName])
                    ThisCar_table_V0G_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_V0G, how="left", on=["Interval start"])
                    AllCar_table_float_V0G = AllCar_table_float_V0G.merge(ThisCar_table_V0G_,  on="Interval start", how='outer')
                    # V1G_real which is the real charging data
                    IntervalkWh_ThisCar_float_V1G = data_Merge_ThisDate_ThisCar["Interval kWh"]
                    ThisCar_table_float = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_float_V1G)), columns=["Interval start", CarName])
                    ThisCar_table_float_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_float, how="left", on=["Interval start"])
                    AllCar_table_float_V1G = AllCar_table_float_V1G.merge(ThisCar_table_float_,  on="Interval start", how='outer')
            # Availability matrix with Max CHarger capacity
            AllCar_table_max_charger = AllCar_table_max_charger.fillna(0)
            IntervalkWh_max = AllCar_table_max_charger.drop("Interval start", axis=1)            
            # V0G
            AllCar_table_float_V0G = AllCar_table_float_V0G.fillna(0)
            EnergyDemand_Table_V0G = AllCar_table_float_V0G.drop("Interval start", axis=1)
            EnergyDemand_Step_V0G = EnergyDemand_Table_V0G.sum(axis=1)
            # V1G_real
            AllCar_table_float_V1G = AllCar_table_float_V1G.fillna(0)
            EnergyDemand_Table_V1G =  AllCar_table_float_V1G.drop("Interval start", axis=1)  # tables without time series
            EnergyDemand_Step_V1G = EnergyDemand_Table_V1G.sum(axis=1)
            #################################################################################
            # Sort all the list based on the car arrival time
            #################################################################################       
            Cars2 = np.array(Cars2)
            # Create dataframes for SessionkWh, Eta_min, Arrival time, Types, and Cars
            ArrivalTime_table = pd.DataFrame(ArrivalTime_list).T
            ArrivalTime_table.columns = Cars2
            Car_table = pd.DataFrame(columns=Cars2)
            Eta_min_table = pd.DataFrame(Eta_min_list).T
            Eta_min_table.columns = Cars2
            SessionkWh_table = pd.DataFrame(SessionkWh_list).T
            SessionkWh_table.columns = Cars2
            Type_table = pd.DataFrame(Type_list).T
            Type_table.columns = Cars2
            # sort all tables based on the arrival time (first come first serve)        
            A = pd.to_datetime(ArrivalTime_list)
            B = [k for k in range(len(A))]
            CarOrder = [x for _, x in sorted(zip(A, B))]
            # Sort the order                
            ArrivalTime_table = ArrivalTime_table.iloc[:, CarOrder]
            Car_table         = Car_table.iloc[:, CarOrder]
            Eta_min_table     = Eta_min_table.iloc[:, CarOrder]
            SessionkWh_table  = SessionkWh_table.iloc[:, CarOrder]
            Type_table        = Type_table.iloc[:, CarOrder]
            IntervalkWh_max   = IntervalkWh_max.iloc[:, CarOrder]
            # Final feasibility reconciliation for optimizer input.
            # Rare PowerFlex records can have session kWh slightly above the 15-min
            # interval max-demand envelope after rounding/merging. The MILP cannot
            # satisfy both equality service and interval upper bounds unless we cap
            # only these data-quality mismatches before constructing forecasts.
            if SessionkWh_table.shape[1] > 0:
                _max_feasible_session_kwh = IntervalkWh_max.sum(axis=0).reindex(SessionkWh_table.columns).fillna(0.0).astype(float)
                _session_kwh_raw = SessionkWh_table.iloc[0].astype(float)
                _excess_kwh = _session_kwh_raw - _max_feasible_session_kwh
                _needs_cap = _excess_kwh > 1e-6
                _large_mismatch = _excess_kwh > 0.10
                if bool(_large_mismatch.any()):
                    print("Warning: EV session energy exceeds interval envelope by more than 0.10 kWh; capping for feasibility:")
                    print(pd.DataFrame({
                        'session_kWh': _session_kwh_raw[_large_mismatch],
                        'interval_envelope_kWh': _max_feasible_session_kwh[_large_mismatch],
                        'excess_kWh': _excess_kwh[_large_mismatch],
                    }).round(4))
                if bool(_needs_cap.any()):
                    print(f"Adjusted {int(_needs_cap.sum())} session energy value(s) to interval max-feasible energy on {TheDates_Start[i_d].date()}")
                    SessionkWh_table.iloc[0, :] = np.minimum(_session_kwh_raw, _max_feasible_session_kwh).values
            #################################################################################
            # Seperate two types of cars: Tesla & non-Tesla
            #################################################################################                 
            if Type_table.empty or Type_table.shape[1] == 0:
                Type_table_1 = Type_table.iloc[:, 0:0].copy()
                Type_table_2 = Type_table.iloc[:, 0:0].copy()
            else:
                Type_table_1 = Type_table.loc[:,Type_table.iloc[0, :] == 'Tesla']
                Type_table_2 = Type_table.loc[:,Type_table.iloc[0, :] != 'Tesla']            
            Logic_table_select_car_1 = Car_table.columns.intersection(list(Type_table_1.columns))
            Logic_table_select_car_2 = Car_table.columns.intersection(list(Type_table_2.columns))
            ArrivalTime_table_1 = ArrivalTime_table[Logic_table_select_car_1]
            ArrivalTime_table_2 = ArrivalTime_table[Logic_table_select_car_2]
            Car_table_1 = Car_table[Logic_table_select_car_1]
            Car_table_2 = Car_table[Logic_table_select_car_2]
            Eta_min_table_1 = Eta_min_table[Logic_table_select_car_1]
            Eta_min_table_2 = Eta_min_table[Logic_table_select_car_2]
            IntervalkWh_max_1 = IntervalkWh_max[Logic_table_select_car_1]
            IntervalkWh_max_2 = IntervalkWh_max[Logic_table_select_car_2]
            SessionkWh_table_1 = SessionkWh_table[Logic_table_select_car_1]
            SessionkWh_table_2 = SessionkWh_table[Logic_table_select_car_2]
            #Put all variables of Day0, Dayb1, and Dayb2 in lists
            ArrivalTime_table_TheDates = ArrivalTime_table_TheDates + [ArrivalTime_table]
            ArrivalTime_table_1_TheDates = ArrivalTime_table_1_TheDates + [ArrivalTime_table_1]            
            ArrivalTime_table_2_TheDates = ArrivalTime_table_2_TheDates + [ArrivalTime_table_2]
            ArrivalTime_table_TheDates_ = [ArrivalTime_table_TheDates,ArrivalTime_table_1_TheDates,ArrivalTime_table_2_TheDates]
            Car_table_TheDates = Car_table_TheDates + [Car_table]
            Car_table_1_TheDates = Car_table_1_TheDates + [Car_table_1]
            Car_table_2_TheDates = Car_table_2_TheDates + [Car_table_2]
            Car_table_TheDates_ = [Car_table_TheDates,Car_table_1_TheDates,Car_table_2_TheDates]
            Eta_min_table_TheDates = Eta_min_table_TheDates + [Eta_min_table]
            Eta_min_table_1_TheDates = Eta_min_table_1_TheDates + [Eta_min_table_1]
            Eta_min_table_2_TheDates = Eta_min_table_2_TheDates + [Eta_min_table_2]  
            Eta_min_table_TheDates_ = [Eta_min_table_TheDates,Eta_min_table_1_TheDates,Eta_min_table_2_TheDates]
            IntervalkWh_max_TheDates = IntervalkWh_max_TheDates + [IntervalkWh_max]
            IntervalkWh_max_1_TheDates = IntervalkWh_max_1_TheDates + [IntervalkWh_max_1]
            IntervalkWh_max_2_TheDates = IntervalkWh_max_2_TheDates + [IntervalkWh_max_2]
            IntervalkWh_max_TheDates_ = [IntervalkWh_max_TheDates,IntervalkWh_max_1_TheDates,IntervalkWh_max_2_TheDates]
            SessionkWh_table_TheDates = SessionkWh_table_TheDates + [SessionkWh_table]
            SessionkWh_table_1_TheDates = SessionkWh_table_1_TheDates + [SessionkWh_table_1]
            SessionkWh_table_2_TheDates = SessionkWh_table_2_TheDates + [SessionkWh_table_2]
            SessionkWh_table_TheDates_ = [SessionkWh_table_TheDates,SessionkWh_table_1_TheDates,SessionkWh_table_2_TheDates]
            Type_table_TheDates = Type_table_TheDates + [Type_table]
            Type_table_1_TheDates = Type_table_1_TheDates + [Type_table_1]
            Type_table_2_TheDates = Type_table_2_TheDates + [Type_table_2]
            EnergyDemand_Table_V0G_TheDates = EnergyDemand_Table_V0G_TheDates + [EnergyDemand_Table_V0G] 
            EnergyDemand_Step_V0G_TheDates = EnergyDemand_Step_V0G_TheDates + [EnergyDemand_Step_V0G]
            EnergyDemand_Table_V1G_TheDates = EnergyDemand_Table_V1G_TheDates + [EnergyDemand_Table_V1G] 
            EnergyDemand_Step_V1G_TheDates = EnergyDemand_Step_V1G_TheDates + [EnergyDemand_Step_V1G]    
        if Fc_AtArrival == 'MLatArrival' : 
            #List ['Car', 'User', 'Session start', 'Arrival Hour', 'Battery (kWh)', 'Weekday', 'Max Charging Power']
            All_Sess = pd.concat([pd.Series(Sess_Car),  pd.Series(Sess_User),    pd.Series(Sess_SessionStart), pd.Series(Sess_ArrivalHr),\
                                    pd.Series(Sess_BESS), pd.Series(Sess_Weekday), pd.Series(Sess_MaxPw)], axis=1)
            All_Sess.columns = List
        else:
            All_Sess = pd.DataFrame()
        # Choose the Real and forecasted variables from Day0, or Day 1 based on the forecast
        i_ = len(Cases)+1
        EmptyList = [[[] for _ in range(i_)] for _ in range(3)] # for All EVS, Tesla, non-Tesla, each has three lists 
        ArrivalTime_table_real =  [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy() 
        ArrivalTime_table_fc   = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Car_table_real         = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Car_table_fc           = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Eta_min_table_real     = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Eta_table_RT           = [[[] for _ in range(i_)] for _ in range(3)] 
        IntervalkWh_max_real   = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        IntervalkWh_max_fc     = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        SessionkWh_table_real  = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        SessionkWh_table_fc    = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()    
        Time_table_real        = [[] for _ in range(i_)]
        Time_table_fc          = [[] for _ in range(i_)]
        TimeSeries_real        = [[] for _ in range(i_)]
        TimeSeries_fc          = [[] for _ in range(i_)]
        Baseline_Opt_real = [[] for _ in range(i_)] #Base, Case1, and V0G
        Baseline_Opt_fc   = [[] for _ in range(i_)]
        # unlike other variables, Base and Case1 have diff baseline values
        for i in range(len(Cases)+1): #Base, Case1, V0G
            Baseline_Opt_real[i] = np.array(Baseline_Opt_avg[0][i])
            if Fc_SessionkWh == 'PerfectSessionkWh'  :
                Baseline_Opt_fc[i] = np.array(Baseline_Opt_avg[0][i]) 
            elif Fc_SessionkWh == 'PersistenceSessionkWh'  :
                Baseline_Opt_fc[i] = np.array(Baseline_Opt_avg[1][i]) 
        #Assign real values for RT (2 cases) amd DA
        for i in range(len(Cases)+1):
            if i < len(Cases):
                i_ = 0 # Day0 assigned to active RT cases
            else:
                i_ = 1 # Dayb1 assigned to DA/V0G reference
            for j in range(3):      #All EVs, Tela, and Non-Tesla
                ArrivalTime_table_real[j][i] = (ArrivalTime_table_TheDates_[j][i_]).copy()
                Car_table_real[j][i] = Car_table_TheDates_[j][i_].copy()
                Eta_min_table_real[j][i] = Eta_min_table_TheDates_[j][i_].copy() 
                if i == 0: #base case has Eta=100*
                    Eta_min_table_real[j][i][Eta_min_table_real[j][i]>0]=1
                Eta_table_RT[j][i] = Eta_min_table_real[j][i].copy()
                #Eta_table_RT[j][i][Eta_table_RT[j][i]>0] = 1
                IntervalkWh_max_real[j][i] = IntervalkWh_max_TheDates_[j][i_].copy()
                SessionkWh_table_real[j][i]   =  SessionkWh_table_TheDates_[j][i_].copy()    
            Time_table_real[i] = Time_table_TheDates[i_].copy() 
            TimeSeries_real[i] = TimeSeries_TheDates[i_].copy()            
            if Fc_SessionkWh == 'PerfectSessionkWh':
                i_ = 0
            elif Fc_SessionkWh == 'PersistenceSessionkWh':
                if i < len(Cases):
                    i_ = 1 # Dayb1 assigned to active RT cases
                else:
                    i_ = 2 # Dayb2 assigned to DA/V0G reference
            for j in range(3): #All EVs, Tela, and Non-Tesla
                ArrivalTime_table_fc[j][i] = ArrivalTime_table_TheDates_[j][i_].copy()
                Car_table_fc[j][i] = Car_table_TheDates_[j][i_].copy()
                IntervalkWh_max_fc[j][i] = IntervalkWh_max_TheDates_[j][i_].copy()
                SessionkWh_table_fc[j][i]   =  SessionkWh_table_TheDates_[j][i_].copy()                
            Time_table_fc[i] = Time_table_TheDates[i_].copy()                
            TimeSeries_fc[i] = TimeSeries_TheDates[i_].copy() 
        #################################################################################
        # make the forecasted SessionkWh and IntervalkWh_max the same length as the real ones
        #################################################################################
        #For Real-time and Day-ahead optimization, and Real-time optimization has two cases: with Eta= 100% and Eta_min
        IntervalkWh_max_fc_fix = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_table_fc_fix = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        #SessionkWh_left           =  [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        Car_table_fc_fix          =  [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        for i in range(len(Cases)+1):
            if (Fc_SessionkWh == 'PersistenceSessionkWh') & (Fc_NumbEV == 'PersistenceNumbEV'):
                for j in np.array(range(2))+1:
                    IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                    SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                    Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
            elif Fc_NumbEV == 'PerfectNumbEV':
                # in this version, with perfect forecasted SessionkWh, Day-1 uses Day0 data, 
                # Careful! Don't trim the fc_fix variables with Day-1 real data
                # instead, just make '_fc_fix' = '_fc' given that '_fc' already equals to '_real'
                if Fc_SessionkWh == 'PerfectSessionkWh'  :
                    for j in np.array(range(2))+1:
                        IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                        SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                        Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
                else:            
                    for j in np.array(range(2))+1: # Tela, and Non-Tesla
                        # trim the forecast tables
                        if len(Car_table_fc[j][i].columns) > len(Car_table_real[j][i].columns):  
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                            Car_table_fc_fix[j][i] = Car_table_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                        # repeat the last n1 rows
                        # note that after repeating the last n rows, these n column names are repeated 
                            # so when calling columns, donnot call the names, but call the order
                        # note that if there was no cars, ie len(Car_table_1_fc)==0, can't repeat 
                        # note if n1_=0
                        elif (len(Car_table_fc[j][i].columns) < len(Car_table_real[j][i].columns)) & (len(Car_table_fc[j][i].columns) > 0):  
                            n1 = len(Car_table_real[j][i].columns)//len(Car_table_fc[j][i].columns)  # qoutient
                            n1_ = len(Car_table_real[j][i].columns)%len(Car_table_fc[j][i].columns)  # remainder
                            Aux1 = pd.concat([SessionkWh_table_fc[j][i]]*n1,  axis=1)
                            Aux2 = pd.concat([IntervalkWh_max_fc[j][i]]*n1,  axis=1)
                            if n1_ == 0:
                                SessionkWh_table_fc_fix[j][i] = Aux1
                                IntervalkWh_max_fc_fix[j][i] = Aux2
                            else:
                                SessionkWh_table_fc_fix[j][i] = pd.concat([Aux1, SessionkWh_table_fc[j][i].iloc[:, -n1_:]], axis=1) 
                                IntervalkWh_max_fc_fix[j][i] = pd.concat([Aux2, IntervalkWh_max_fc[j][i].iloc[:, -n1_:]], axis=1)
                            Car_table_fc_fix[j][i] = pd.DataFrame(columns=IntervalkWh_max_fc_fix[j][i].columns)
                        # have to use perfect fc if no data from yesterday!!!   
                        # ex: on Dec 27th, no Tesla arrived yesterday(Dec. 26th)
                        elif len(Car_table_fc[j][i]) == 0:
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_real[j][i].copy()
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_real[j][i].copy()
                            Car_table_fc_fix[j][i] = Car_table_real[j][i].copy()
                        else:
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                            Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
            # Combine Type1 (Tesla) and Type2 (Non-Tesla) for optimization    
            SessionkWh_table_fc_fix[0][i] = pd.concat([SessionkWh_table_fc_fix[1][i], SessionkWh_table_fc_fix[2][i]], axis=1)   
            IntervalkWh_max_fc_fix[0][i] = pd.concat([IntervalkWh_max_fc_fix[1][i], IntervalkWh_max_fc_fix[2][i]], axis=1)
            Car_table_fc_fix[0][i] = pd.concat([Car_table_fc_fix[1][i], Car_table_fc_fix[2][i]], axis=1)
            Eta_table_RT[0][i] = pd.concat([Eta_table_RT[1][i], Eta_table_RT[2][i]], axis=1)
        #################################################################################
        # VERSION: DA/Offline CVX Optimization
        #################################################################################
        H = 96
        _n_ev_physical_da = Car_table_fc_fix[0][DA_CASE_IDX].shape[1]
        if _n_ev_physical_da == 0:
            _dummy_ev_col = '__dummy_ev_zero__'
            Car_table_fc_fix[0][DA_CASE_IDX] = pd.DataFrame(columns=[_dummy_ev_col])
            SessionkWh_table_fc_fix[0][DA_CASE_IDX] = pd.DataFrame([[0.0]], columns=[_dummy_ev_col])
            IntervalkWh_max_fc_fix[0][DA_CASE_IDX] = pd.DataFrame(0.0, index=range(H), columns=[_dummy_ev_col])
        H_Start_DA = TheDate_Day0 + timedelta(minutes=0)
        Operator_SumColumn = np.ones((Car_table_fc_fix[0][DA_CASE_IDX].shape[1], 1))  # size: num_cars x 1
        Operator_SumRow = np.ones((1, H)) # size: 1 x num_step
        c_e_TOU_AL_DA = c_e_TOU_AL[:] # $/kWh
        Bid_Pr_DA_i = Bid_Pr_DA[:]
        Bid_Pr_RT_i = Bid_Pr_RT[:]
        AS_Pr_RU_DA_i = AS_Pr_RU_DA[:]
        AS_Pr_RD_DA_i = AS_Pr_RD_DA[:]
        AS_Pr_RU_RT_i = AS_Pr_RU_RT[:]
        AS_Pr_RD_RT_i = AS_Pr_RD_RT[:]
        AS_Pr_SP_DA_i = AS_Pr_SP_DA[:]
        AS_Pr_NSP_DA_i = AS_Pr_NSP_DA[:]
        AS_Pr_SP_RT_i = AS_Pr_SP_RT[:]
        AS_Pr_NSP_RT_i = AS_Pr_NSP_RT[:]
        # EV service revenue with a EV charging rate
        # https://transportation.ucsd.edu/commute/ev-stations.html
        c_EV_service = np.ones(H) * 0.4 # $/kWh
        # Filter for Peak demand Period
        PP_start = 16 * 4
        PP_end = 21 * 4
        Filter = 1
        # output variables for All EVs, Tesla, and Non-Tesla, each for RT (2 cases) and DA optimization
        EnergyDemand_Table_Opt_cap = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        EnergyDemand_Step_Opt_cap = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        # Construct the problem
        Cost_Opt = cp.Variable()
        penalty_BESS = 0
        # EV Variables
        EnergyDemand_Table_Opt = cp.Variable((H, Car_table_fc_fix[0][DA_CASE_IDX].shape[1]))
        EnergyDemand_Step_Opt = cp.Variable(H)
        p_EV = cp.Variable(H, nonneg=True)  # total EV charging power, kW
        p_GI = cp.Variable(H)  # grid import power, kW
        # BESS variables
        p_BESS = cp.Variable(H)
        p_ch_BESS = cp.Variable(H, nonneg=True)
        p_dch_BESS = cp.Variable(H, nonneg=True)
        p_ch_BESS_WM = cp.Variable(H, nonneg=True)
        p_dch_BESS_WM = cp.Variable(H, nonneg=True)
        p_ch_BESS_NWM = cp.Variable(H, nonneg=True)
        p_dch_BESS_NWM = cp.Variable(H, nonneg=True)
        soc_BESS = cp.Variable(H+1) # SOC from 0 to H
        if Day == run_days[0]:
            SOC_BESS_initial = 0.5
        else:
            SOC_BESS_initial = SOC_BESS_daily_end[0]  # use the final SOC of previous day in 100% eta as initial SOC of current day
        # BESS constraints
        constraints_bess = [
            p_BESS == p_ch_BESS - p_dch_BESS, # either positive or negative
            p_ch_BESS == p_ch_BESS_WM + p_ch_BESS_NWM,
            p_dch_BESS == p_dch_BESS_WM + p_dch_BESS_NWM,
            p_BESS <= P_BESS_max,
            p_BESS >= -P_BESS_max,
            soc_BESS[0] == SOC_BESS_initial,
            soc_BESS <= SOC_BESS_max,
            soc_BESS >= SOC_BESS_min,
            soc_BESS[1:] == soc_BESS[:-1] + (dt_h / C_BESS) * (cp.multiply(p_ch_BESS, np.sqrt(gamma)) - cp.multiply(p_dch_BESS, 1 / np.sqrt(gamma))),
            dt_h * cp.sum(p_ch_BESS + p_dch_BESS) <= 2 * C_BESS * (SOC_BESS_max - SOC_BESS_min)  # limit total throughput in one day
        ]
        if is_last_run_day:
            constraints_bess += [soc_BESS[-1] == 0.5]
            penalty_BESS_target = 0.5
            penalty_BESS = (cp.norm(soc_BESS[-1] - penalty_BESS_target, 2)) * c_BESS_penalty # penalty coefficient
        else:
            penalty_BESS = cp.Constant(0.0)
        # Power balance
        constraints_balance = [
            p_GI == p_EV + p_BESS
        ]
        Baseline_Opt_i = np.nan_to_num(Baseline_Opt_fc[0].repeat(4), nan=0.0, posinf=0.0, neginf=0.0) # TODO: current Baseline use the 100% perfect, need to update from DRAM
        B_EV = Baseline_Opt_i / dt_h  # convert kWh to kW for constraints and cost calculation
        P_EV_max = 6.6 * _n_ev_physical_da  # max charging power per real car * number of real cars, kW
        M_EV = 1.05 * max(float(P_EV_max), float(np.nanmax(B_EV)), 1e-6)  # big-M must cover EV deviation from baseline, especially low-EV days
        # WM constraints
        p_DA = cp.Variable(H)
        p_RT = cp.Variable(H)
        c_RU_DA = cp.Variable(H, nonneg=True)
        c_RD_DA = cp.Variable(H, nonneg=True)
        c_SP_DA = cp.Variable(H, nonneg=True)
        c_NSP_DA = cp.Variable(H, nonneg=True)
        # Signed RT AS adjustments: actual delivered capacity is c_DA + c_RT.
        c_RU_RT = cp.Variable(H)
        c_RD_RT = cp.Variable(H)
        c_SP_RT = cp.Variable(H)
        c_NSP_RT = cp.Variable(H)
        p_up = P_BESS_max + B_EV
        p_down = P_BESS_max - B_EV + P_EV_max
        p_ch_EV_WM = cp.Variable(H, nonneg=True)      # EV baseline deviation used for WM products
        p_dch_EV_WM = cp.Variable(H, nonneg=True)     # EV baseline deviation used for WM products
        p_ch_EV_NWM = cp.Variable(H, nonneg=True)  # EV baseline deviation needed for non-WM service feasibility
        p_dch_EV_NWM = cp.Variable(H, nonneg=True)
        p_ch_net = cp.Variable(H, nonneg=True)
        p_dch_net = cp.Variable(H, nonneg=True)
        b_ch_BESS = cp.Variable(H, boolean=True)
        b_dch_BESS = cp.Variable(H, boolean=True)
        b_ch_EV = cp.Variable(H, boolean=True)
        b_dch_EV = cp.Variable(H, boolean=True)
        b_ch_net = cp.Variable(H, boolean=True)
        b_dch_net = cp.Variable(H, boolean=True)
        alpha_RU = 0.7
        alpha_RD = 0.7
        alpha_SP = 0.2
        alpha_NSP = 0.2
        p_wm_actual_DA = p_DA + p_RT
        if Enable_WM:
            constraints_WM = [
                c_RU_DA + c_RU_RT >= 0,
                c_RD_DA + c_RD_RT >= 0,
                c_SP_DA + c_SP_RT >= 0,
                c_NSP_DA + c_NSP_RT >= 0,
                c_RU_DA + c_SP_DA + c_NSP_DA <= p_up,
                c_RD_DA <= p_down,
                # Capacity
                # p_DA + p_RT + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                # -(p_DA + p_RT) + c_RD_DA + c_RD_RT <= p_down,
                cp.pos(p_wm_actual_DA) + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                cp.pos(-p_wm_actual_DA) + c_RD_DA + c_RD_RT <= p_down,
                p_DA <= p_up,
                p_RT <= p_up,
                p_DA >= -p_down,
                p_RT >= -p_down,
                # Energy
                b_ch_BESS + b_dch_BESS <= 1,
                b_ch_EV + b_dch_EV <= 1,
                b_ch_net + b_dch_net <= 1,
                p_EV - B_EV == (p_ch_EV_WM - p_dch_EV_WM) + (p_ch_EV_NWM - p_dch_EV_NWM),
                # p_dch_EV_WM == (B_EV - p_EV) * b_dch_EV,
                p_dch_EV_WM + p_dch_EV_NWM <= M_EV * b_dch_EV,
                # p_ch_EV_WM == -(B_EV - p_EV) * b_ch_EV,
                p_ch_EV_WM + p_ch_EV_NWM <= M_EV * b_ch_EV,
                p_dch_net <= p_DA + p_RT + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT) + cp.multiply(p_up, b_ch_net),
                p_dch_net >= p_DA + p_RT + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT),
                p_dch_BESS <= P_BESS_max * b_dch_BESS,
                p_ch_net <= -(p_DA + p_RT + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT)) + cp.multiply(p_down, b_dch_net),
                p_ch_net >= -(p_DA + p_RT + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT)),
                p_ch_BESS <= P_BESS_max * b_ch_BESS,
                p_ch_net <= cp.multiply(p_down, b_ch_net),
                p_dch_net <= cp.multiply(p_up, b_dch_net),
                p_ch_net - p_dch_net == p_ch_BESS_WM - p_dch_BESS_WM + p_ch_EV_WM - p_dch_EV_WM,
                # Duration
                soc_BESS[1:] >= SOC_BESS_min + (2 * dt_h) / (C_BESS * np.sqrt(gamma)) * (c_RU_DA + c_RU_RT + c_SP_DA + c_SP_RT + c_NSP_DA + c_NSP_RT),
                soc_BESS[1:] <= SOC_BESS_max - (2 * dt_h * np.sqrt(gamma)) / C_BESS * (c_RD_DA + c_RD_RT)
            ]
            # VERSION: Add-ons when direction aligned
            if Direction_Aligned_AddOn:
                constraints_WM += [
                    c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= cp.multiply(p_up, b_dch_net),
                    c_RD_DA + c_RD_RT <= cp.multiply(p_down, b_ch_net),
                ]
            else:
                print(f"--> Current Direction_Aligned_AddOn Status: {Direction_Aligned_AddOn}")
            # VERSION: WM only mode (disable non-WM BESS channel)
            if WM_Mode == 'wm_only':
                constraints_WM += [
                    p_ch_BESS_NWM == 0,
                    p_dch_BESS_NWM == 0,
                ]
            # WM Revenue calculation
            Revenue_WM = (
                dt_h * (
                    Bid_Pr_RT_i @ (alpha_RU * (c_RU_DA + c_RU_RT) - alpha_RD * (c_RD_DA + c_RD_RT)
                                   + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT))
                    + Bid_Pr_DA_i @ p_DA + Bid_Pr_RT_i @ p_RT
                    + AS_Pr_RU_DA_i @ c_RU_DA + AS_Pr_RU_RT_i @ c_RU_RT + AS_Pr_RD_DA_i @ c_RD_DA + AS_Pr_RD_RT_i @ c_RD_RT
                    + AS_Pr_SP_DA_i @ c_SP_DA + AS_Pr_SP_RT_i @ c_SP_RT + AS_Pr_NSP_DA_i @ c_NSP_DA + AS_Pr_NSP_RT_i @ c_NSP_RT
                )
            )
        else:
            constraints_WM = [
                p_DA == 0, p_RT == 0,
                c_RU_DA == 0, c_RD_DA == 0, c_SP_DA == 0, c_NSP_DA == 0,
                c_RU_RT == 0, c_RD_RT == 0, c_SP_RT == 0, c_NSP_RT == 0,
                p_ch_BESS_WM == 0, p_dch_BESS_WM == 0,
                p_ch_EV_WM == 0, p_dch_EV_WM == 0, p_ch_net == 0, p_dch_net == 0,
            ]
            Revenue_WM = 0
        # Cost-related constraints
        constraints_cost = [
            Cost_Opt >= (
                c_NCD * cp.maximum(cp.max(p_GI) - M_Th_NCD[DA_CASE_IDX], 0)
                + c_PD * cp.maximum(cp.max(Filter * p_GI[PP_start:PP_end]) - M_Th_PD[DA_CASE_IDX], 0)
                + c_e_TOU_AL_DA @ p_GI * dt_h
                - c_EV_service @ p_EV * dt_h
                - Revenue_WM
            )
        ]
        # EV constraints
        constraints_EV = [
            p_EV == EnergyDemand_Step_Opt / dt_h,  # kW
            EnergyDemand_Table_Opt @ Operator_SumColumn == cp.reshape(EnergyDemand_Step_Opt, (H, 1), order='C'),
            Operator_SumRow @ EnergyDemand_Table_Opt == SessionkWh_table_fc_fix[0][DA_CASE_IDX],
            EnergyDemand_Table_Opt >= 0,
            EnergyDemand_Table_Opt <= IntervalkWh_max_fc_fix[0][DA_CASE_IDX].iloc[:, :]
        ]
        # NOTE: Combine all constraints
        constraints = constraints_cost + constraints_EV + constraints_bess + constraints_WM + constraints_balance
        # Solve the problem            
        # solver_opts = {"Method": 2, "Seed": 0, "Threads": 1}
        objective = cp.Minimize(Cost_Opt + penalty_BESS)
        prob = cp.Problem(objective, constraints)
        result = prob.solve(solver=cp.GUROBI, verbose=False, MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1)
        status = prob.status
        if status == 'infeasible_or_unbounded':
            try:
                result = prob.solve(solver=cp.GUROBI, verbose=False, MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1, reoptimize=True)
                status = prob.status
            except Exception as _da_reopt_err:
                print(f"DA reoptimize failed while diagnosing infeasible_or_unbounded: {_da_reopt_err}")
        Status_list_DA = Status_list_DA + [status]
        if status != 'optimal':
            _b_ev_arr = np.asarray(B_EV, dtype=float)
            _p_down_arr = np.asarray(p_down, dtype=float)
            _p_up_arr = np.asarray(p_up, dtype=float)
            try:
                _da_energy_target = float(np.nansum(np.asarray(SessionkWh_table_fc_fix[0][DA_CASE_IDX]).astype(float)))
                _da_energy_envelope = float(np.nansum(np.asarray(IntervalkWh_max_fc_fix[0][DA_CASE_IDX]).astype(float)))
            except Exception:
                _da_energy_target = np.nan
                _da_energy_envelope = np.nan
            print("DA optimization does not have a usable solution!")
            print(f"  status={status}, month={month}, day={Day}, date={pd.Timestamp(TheDate_Day0).strftime('%Y-%m-%d')}, WM_Mode={WM_Mode}, Enable_WM={Enable_WM}")
            print(f"  forecast={Fc_SessionkWh}|{Fc_NumbEV}|{Fc_AtArrival}, SOC_initial={float(SOC_BESS_initial):.6f}, P_EV_max={float(P_EV_max):.6f}, M_EV={float(M_EV):.6f}")
            print(f"  B_EV[kW]: min={float(np.nanmin(_b_ev_arr)):.6f}, max={float(np.nanmax(_b_ev_arr)):.6f}, sum={float(np.nansum(_b_ev_arr)):.6f}, nan={int(np.isnan(_b_ev_arr).sum())}")
            print(f"  p_up[kW]: min={float(np.nanmin(_p_up_arr)):.6f}, max={float(np.nanmax(_p_up_arr)):.6f}; p_down[kW]: min={float(np.nanmin(_p_down_arr)):.6f}, max={float(np.nanmax(_p_down_arr)):.6f}, negative_count={int(np.sum(_p_down_arr < -1e-7))}")
            print(f"  DA EV session target kWh={_da_energy_target:.6f}, interval envelope kWh={_da_energy_envelope:.6f}, target-minus-envelope={_da_energy_target - _da_energy_envelope:.6f}")
            sys.exit()
        # Save DA results            
        Solver_Outputs_DA = {
            'Cost_Opt': [],
            'Cost_PD': [],
            'Cost_NCD': [],
            'Cost_TOU': [],
            'Revenue_EV': [],
            'penalty_BESS': [],
            'Revenue_WM': [],
            'p_GI': [],
            'p_EV': [],
            'p_ch_BESS': [],
            'p_dch_BESS': [],
            'p_ch_BESS_WM': [],
            'p_dch_BESS_WM': [],
            'p_ch_BESS_NWM': [],
            'p_dch_BESS_NWM': [],
            'p_ch_EV': [],
            'p_dch_EV': [],
            'p_ch_EV_WM': [],
            'p_dch_EV_WM': [],
            'p_ch_EV_NWM': [],
            'p_dch_EV_NWM': [],
            'p_ch_net': [],
            'p_dch_net': [],
            'soc_BESS': [],
            'p_BESS': [],
            'c_RU_DA': [],
            'c_RD_DA': [],
            'c_SP_DA': [],
            'c_NSP_DA': [],
            'p_DA': [],
            'c_RU_RT': [],
            'c_RD_RT': [],
            'c_SP_RT': [],
            'c_NSP_RT': [],
            'p_RT': [],
            'LMP_DA': Bid_Pr_DA,
            'LMP_RT': Bid_Pr_RT,
            'AS_Pr_RU_DA': AS_Pr_RU_DA,
            'AS_Pr_RU_RT': AS_Pr_RU_RT,
            'AS_Pr_RD_DA': AS_Pr_RD_DA,
            'AS_Pr_RD_RT': AS_Pr_RD_RT,
            'AS_Pr_SP_DA': AS_Pr_SP_DA,
            'AS_Pr_NSP_DA': AS_Pr_NSP_DA,
            'AS_Pr_SP_RT': AS_Pr_SP_RT,
            'AS_Pr_NSP_RT': AS_Pr_NSP_RT,
            'Baseline(kW)': Baseline_Opt_i / dt_h,
            'P_EV_max': P_EV_max,
        }
        c_RU_DA_value_DA = c_RU_DA.value
        c_RD_DA_value_DA = c_RD_DA.value
        c_SP_DA_value_DA = c_SP_DA.value
        c_NSP_DA_value_DA = c_NSP_DA.value
        p_DA_value_DA = p_DA.value
        p_RT_value_DA = p_RT.value
        EnergyDemand_Table_Opt_value = EnergyDemand_Table_Opt.value
        EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX] = pd.DataFrame(EnergyDemand_Table_Opt_value, columns=list(Car_table_fc_fix[0][DA_CASE_IDX].columns))
        EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX][EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX] <= 0.0001] = 0 
        EnergyDemand_Table_Opt_cap[1][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].iloc[:, :len(Car_table_fc_fix[1][DA_CASE_IDX])]
        EnergyDemand_Table_Opt_cap[2][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].iloc[:, len(Car_table_fc_fix[2][DA_CASE_IDX]):]
        EnergyDemand_Step_Opt_cap[0][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].sum(axis=1)
        # dispatch_Offline = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX]
        Solver_Outputs_DA['Cost_Opt'] = Cost_Opt.value.item()
        Solver_Outputs_DA['Cost_PD'] = (c_PD * np.maximum(p_GI.value[PP_start:PP_end].max() - M_Th_PD[DA_CASE_IDX], 0)).item()
        Solver_Outputs_DA['Cost_NCD'] = (c_NCD * np.maximum(np.max(p_GI.value) - M_Th_NCD[DA_CASE_IDX], 0)).item()
        Solver_Outputs_DA['Cost_TOU'] = (np.sum(c_e_TOU_AL_DA * p_GI.value) * dt_h).item()
        Solver_Outputs_DA['Revenue_EV'] = (np.sum(c_EV_service * p_EV.value) * dt_h).item()
        Solver_Outputs_DA['penalty_BESS'] = penalty_BESS.value.item()
        Solver_Outputs_DA['Revenue_WM'] = Revenue_WM.value.item() if Enable_WM else 0.0
        Solver_Outputs_DA['p_GI'] = p_GI.value.flatten()
        Solver_Outputs_DA['p_EV'] = p_EV.value.flatten()
        Solver_Outputs_DA['p_ch_BESS'] = p_ch_BESS.value.flatten()
        Solver_Outputs_DA['p_dch_BESS'] = p_dch_BESS.value.flatten()
        Solver_Outputs_DA['p_ch_BESS_WM'] = p_ch_BESS_WM.value.flatten()
        Solver_Outputs_DA['p_dch_BESS_WM'] = p_dch_BESS_WM.value.flatten()
        Solver_Outputs_DA['p_ch_BESS_NWM'] = p_ch_BESS_NWM.value.flatten()
        Solver_Outputs_DA['p_dch_BESS_NWM'] = p_dch_BESS_NWM.value.flatten()
        # Backward-compatible output keys: p_ch_EV/p_dch_EV now store the EV WM channel.
        Solver_Outputs_DA['p_ch_EV'] = p_ch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_dch_EV'] = p_dch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_ch_EV_WM'] = p_ch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_dch_EV_WM'] = p_dch_EV_WM.value.flatten()
        _ev_wm_dev_DA = np.asarray(p_ch_EV_WM.value).flatten() - np.asarray(p_dch_EV_WM.value).flatten()
        _ev_resid_DA = np.asarray(p_EV.value).flatten() - np.asarray(B_EV).flatten() - _ev_wm_dev_DA
        if p_ch_EV_NWM.value is None or p_dch_EV_NWM.value is None:
            _p_ch_EV_NWM_DA = np.maximum(_ev_resid_DA, 0.0)
            _p_dch_EV_NWM_DA = np.maximum(-_ev_resid_DA, 0.0)
        else:
            _p_ch_EV_NWM_DA = p_ch_EV_NWM.value.flatten()
            _p_dch_EV_NWM_DA = p_dch_EV_NWM.value.flatten()
        Solver_Outputs_DA['p_ch_EV_NWM'] = _p_ch_EV_NWM_DA
        Solver_Outputs_DA['p_dch_EV_NWM'] = _p_dch_EV_NWM_DA
        Solver_Outputs_DA['p_ch_net'] = p_ch_net.value.flatten()
        Solver_Outputs_DA['p_dch_net'] = p_dch_net.value.flatten()
        Solver_Outputs_DA['soc_BESS'] = soc_BESS.value.flatten()[1:]  # exclude the initial SOC
        Solver_Outputs_DA['p_BESS'] = p_BESS.value.flatten()
        Solver_Outputs_DA['c_RU_DA'] = c_RU_DA.value.flatten()
        Solver_Outputs_DA['c_RD_DA'] = c_RD_DA.value.flatten()
        Solver_Outputs_DA['c_SP_DA'] = c_SP_DA.value.flatten()
        Solver_Outputs_DA['c_NSP_DA'] = c_NSP_DA.value.flatten()
        Solver_Outputs_DA['p_DA'] = p_DA.value.flatten()
        Solver_Outputs_DA['c_RU_RT'] = c_RU_RT.value.flatten()
        Solver_Outputs_DA['c_RD_RT'] = c_RD_RT.value.flatten()
        Solver_Outputs_DA['c_SP_RT'] = c_SP_RT.value.flatten()
        Solver_Outputs_DA['c_NSP_RT'] = c_NSP_RT.value.flatten()
        Solver_Outputs_DA['p_RT'] = p_RT.value.flatten()
        Solver_Outputs_DA['P_EV_max'] = P_EV_max
        # before accepting DA market commitments. This is a pre-submission screen,
        # not an RT ex-post reversal.
        # DA per-timestep WM profit breakdown for each c/p product
        bid_pr_da_i = np.array(Bid_Pr_DA).flatten()
        bid_pr_rt_i = np.array(Bid_Pr_RT).flatten()
        as_ru_da_i = np.array(AS_Pr_RU_DA_i).flatten()
        as_ru_rt_i = np.array(AS_Pr_RU_RT_i).flatten()
        as_rd_da_i = np.array(AS_Pr_RD_DA_i).flatten()
        as_rd_rt_i = np.array(AS_Pr_RD_RT_i).flatten()
        as_sp_da_i = np.array(AS_Pr_SP_DA_i).flatten()
        as_sp_rt_i = np.array(AS_Pr_SP_RT_i).flatten()
        as_nsp_da_i = np.array(AS_Pr_NSP_DA_i).flatten()
        as_nsp_rt_i = np.array(AS_Pr_NSP_RT_i).flatten()
        if Enable_WM:
            wm_profit_table = pd.DataFrame({
                'p_DA_profit': dt_h * bid_pr_da_i * Solver_Outputs_DA['p_DA'], # 注意这里，DA处是 Solver_Outputs_DA，RT处是 Solver_Outputs_RT_Base
                'p_RT_profit': dt_h * bid_pr_rt_i * Solver_Outputs_DA['p_RT'],
                # 向上调节 RU
                'c_RU_DA_energy': dt_h * (alpha_RU * bid_pr_rt_i) * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_DA_capacity': dt_h * as_ru_da_i * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_RT_energy': dt_h * (alpha_RU * bid_pr_rt_i) * Solver_Outputs_DA['c_RU_RT'],
                'c_RU_RT_capacity': dt_h * as_ru_rt_i * Solver_Outputs_DA['c_RU_RT'],
                # 向下调节 RD (注意公式中的负号)
                'c_RD_DA_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_DA_capacity': dt_h * as_rd_da_i * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_RT_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * Solver_Outputs_DA['c_RD_RT'],
                'c_RD_RT_capacity': dt_h * as_rd_rt_i * Solver_Outputs_DA['c_RD_RT'],
                # 旋转备用 SP
                'c_SP_DA_energy': dt_h * (alpha_SP * bid_pr_rt_i) * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_DA_capacity': dt_h * as_sp_da_i * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_RT_energy': dt_h * (alpha_SP * bid_pr_rt_i) * Solver_Outputs_DA['c_SP_RT'],
                'c_SP_RT_capacity': dt_h * as_sp_rt_i * Solver_Outputs_DA['c_SP_RT'],
                # 非旋转备用 NSP
                'c_NSP_DA_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_DA_capacity': dt_h * as_nsp_da_i * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_RT_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * Solver_Outputs_DA['c_NSP_RT'],
                'c_NSP_RT_capacity': dt_h * as_nsp_rt_i * Solver_Outputs_DA['c_NSP_RT'],
            })
            wm_tou_table = pd.DataFrame({
                'p_DA_TOU_Cost': -dt_h * c_e_TOU_AL * Solver_Outputs_DA['p_DA'],
                'p_RT_TOU_Cost': -dt_h * c_e_TOU_AL * Solver_Outputs_DA['p_RT'],
                'c_RU_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * Solver_Outputs_DA['c_RU_RT'],
                'c_RD_DA_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_RT_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * Solver_Outputs_DA['c_RD_RT'],
                'c_SP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * Solver_Outputs_DA['c_SP_RT'],
                'c_NSP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * Solver_Outputs_DA['c_NSP_RT'],
            })
        else:
            cols = [
                'p_DA_profit', 'p_RT_profit',
                'c_RU_DA_energy', 'c_RU_DA_capacity', 'c_RU_RT_energy', 'c_RU_RT_capacity',
                'c_RD_DA_energy', 'c_RD_DA_capacity', 'c_RD_RT_energy', 'c_RD_RT_capacity',
                'c_SP_DA_energy', 'c_SP_DA_capacity', 'c_SP_RT_energy', 'c_SP_RT_capacity',
                'c_NSP_DA_energy', 'c_NSP_DA_capacity', 'c_NSP_RT_energy', 'c_NSP_RT_capacity'
            ]
            wm_profit_table = pd.DataFrame(np.zeros((H, len(cols))), columns=cols)
            wm_tou_table = pd.DataFrame(np.zeros((H, 10)), columns=[
                'p_DA_TOU_Cost', 'p_RT_TOU_Cost',
                'c_RU_DA_TOU_Cost', 'c_RU_RT_TOU_Cost',
                'c_RD_DA_TOU_Cost', 'c_RD_RT_TOU_Cost',
                'c_SP_DA_TOU_Cost', 'c_SP_RT_TOU_Cost',
                'c_NSP_DA_TOU_Cost', 'c_NSP_RT_TOU_Cost'
            ])
        wm_profit_table['profit_sum_all_products'] = wm_profit_table.sum(axis=1)
        wm_tou_table['TOU_Cost_sum_all_products'] = wm_tou_table.sum(axis=1)
        wm_profit_total_from_table = float(wm_profit_table['profit_sum_all_products'].sum())
        wm_profit_check_diff = float(abs(wm_profit_total_from_table - Solver_Outputs_DA['Revenue_WM']))
        Solver_Outputs_DA['WM_Profit_Table'] = wm_profit_table
        Solver_Outputs_DA['WM_Profit_Total_From_Table'] = wm_profit_total_from_table
        Solver_Outputs_DA['WM_TOU_Table'] = wm_tou_table
        Solver_Outputs_DA['WM_TOU_Total_From_Table'] = float(wm_tou_table['TOU_Cost_sum_all_products'].sum())
        Solver_Outputs_DA['WM_Profit_Check_Diff'] = wm_profit_check_diff
        if wm_profit_check_diff > 1e-3:
            print(f"DA WM profit check -> table_sum={wm_profit_total_from_table:.6f}, Revenue_WM={Solver_Outputs_DA['Revenue_WM']:.6f}, abs_diff={wm_profit_check_diff:.6e}")
        p_GI_DA = Solver_Outputs_DA['p_GI']
        # Save threshold BEFORE updating, so fig(f) shows previous day's threshold
        M_Th_NCD_prev = list(M_Th_NCD)  # snapshot of previous day's threshold
        M_Th_PD_prev = list(M_Th_PD)
        # DA threshold is 2, RT 100% and eta% are 0 and 1, respectively
        M_Th_NCD[DA_CASE_IDX] = np.maximum(p_GI_DA.max(), M_Th_NCD[DA_CASE_IDX])
        M_Th_PD[DA_CASE_IDX] = np.maximum(p_GI_DA[PP_start:PP_end].max(), M_Th_PD[DA_CASE_IDX])
        # M_Th_NCD[DA_CASE_IDX] = np.maximum(max((dispatch_Offline.sum(axis=1))/dt_h),M_Th_NCD[DA_CASE_IDX])
        # M_Th_PD[DA_CASE_IDX] = np.maximum(max((dispatch_Offline.sum(axis=1)[PP_start:PP_end])/dt_h),M_Th_PD[DA_CASE_IDX])
        M_Th_NCD_list[DA_CASE_IDX] = M_Th_NCD_list[DA_CASE_IDX] + [M_Th_NCD[DA_CASE_IDX]]
        M_Th_PD_list[DA_CASE_IDX] = M_Th_PD_list[DA_CASE_IDX] + [M_Th_PD[DA_CASE_IDX]]
        # bidding event hours:
        Baseline_hr  = [[] for _ in range(len(Cases))] # Base, Case1
        Baseline_96  = [[] for _ in range(len(Cases))]
        EventHour    = [[] for _ in range(len(Cases))]   # Base and Case1 (no marlet participation for V0G)
        EventHour_96 = [[] for _ in range(len(Cases))] 
        Offline_hr   = np.average(np.array(p_GI_DA * dt_h).reshape(-1, 4), axis=1)
        # Offline_hr   = np.average(np.array(dispatch_Offline.sum(axis=1)).reshape(-1, 4), axis=1) #ave every 4 entries of the 96  x 1 series
        for i in range(len(Cases)): # Base, Case1
            Baseline_hr[i] = Baseline_Opt_fc[i].copy()
            Baseline_96[i] = Baseline_Opt_fc[i].repeat(4).reshape(96, 1) #repeat 4 times of the 24 x 1 series
            EventHour[i] = Baseline_hr[i] - Offline_hr
            EventHour[i][EventHour[i]<0] = 0
            EventHour[i][EventHour[i]>0] = 1
            EventHour_96[i] = EventHour[i].repeat(4).reshape(96, 1) #repeat 4 times of the 24 x 1 series
        #################################################################################
        # VERSION: Real Time Optimization Implementation
        #################################################################################
        # Inputs for RT optimization
        Dispatch = [[[] for _ in range(len(Cases))] for _ in range(3)]
        IntervalkWh_table_Opt_list  = [[] for _ in range(len(Cases))] 
        SessionkWh_nEta_list = [[] for _ in range(len(Cases))]
        dispatch_t0 = [[],[]]
        # Results saving variables
        Cost_Opt_value_RT = np.zeros((len(Cases), 96))
        Cost_PD_value_RT = np.zeros((len(Cases), 96))
        Cost_NCD_value_RT = np.zeros((len(Cases), 96))
        Cost_TOU_value_RT = np.zeros((len(Cases), 96))
        Revenue_EV_value_RT = np.zeros((len(Cases), 96))
        Revenue_WM_value_RT = np.zeros((len(Cases), 96))
        penalty_BESS_value_RT = np.zeros((len(Cases), 96))
        penalty_WM_value_RT = np.zeros((len(Cases), 96))
        p_GI_value_RT = np.zeros((len(Cases), 96))
        p_EV_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_WM_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_WM_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_NWM_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_NWM_value_RT = np.zeros((len(Cases), 96))
        p_BESS_value_RT = np.zeros((len(Cases), 96))
        p_ch_EV_value_RT = np.zeros((len(Cases), 96))
        p_dch_EV_value_RT = np.zeros((len(Cases), 96))
        p_ch_EV_NWM_value_RT = np.zeros((len(Cases), 96))
        p_dch_EV_NWM_value_RT = np.zeros((len(Cases), 96))
        p_ch_net_value_RT = np.zeros((len(Cases), 96))
        p_dch_net_value_RT = np.zeros((len(Cases), 96))
        p_EV_max_value_RT = np.zeros((len(Cases), 96))
        soc_BESS_value_RT = np.zeros((len(Cases), 96))
        c_RU_DA_value_RT = np.zeros((len(Cases), 96))
        c_RD_DA_value_RT = np.zeros((len(Cases), 96))
        c_SP_DA_value_RT = np.zeros((len(Cases), 96))
        c_NSP_DA_value_RT = np.zeros((len(Cases), 96))
        p_DA_value_RT = np.zeros((len(Cases), 96))
        c_RU_RT_value_RT = np.zeros((len(Cases), 96))
        c_RD_RT_value_RT = np.zeros((len(Cases), 96))
        c_SP_RT_value_RT = np.zeros((len(Cases), 96))
        c_NSP_RT_value_RT = np.zeros((len(Cases), 96))
        p_RT_value_RT = np.zeros((len(Cases), 96))
        E_BESS_throughput_RT = np.zeros((len(Cases), 97)) # store energy throughput, like SOC from 0 to 97
        Th_NCD = [0 for _ in range(len(Cases))]
        Th_PD = [0 for _ in range(len(Cases))]
        #############################################################################
        Count = [[0 for _ in range(len(Cases))] for _ in range(3)]  # All EVs, Tesla, and Non-Tesla for active RT cases
        def _append_aligned_column(frame, column_name, values):
            # New EVs arrive one at a time. Rebuild the small 96-row table with concat
            # so pandas does not accumulate hundreds of fragmented column blocks.
            if isinstance(values, pd.DataFrame):
                if values.shape[1] != 1:
                    raise ValueError(f'Expected one source column for {column_name}, got {values.shape[1]}')
                column = values.iloc[:, 0].copy()
            else:
                column = values.copy() if isinstance(values, pd.Series) else pd.Series(values, index=frame.index)
            column.name = column_name
            return pd.concat([frame, column], axis=1, copy=False).copy()
        for i in range(len(Cases)):
            for j in np.array(range(3)): # All EVs, Tela, and Non-Tesla 
                Dispatch[j][i] = pd.DataFrame(np.zeros(shape=(96, (Car_table_real[j][i]).shape[1])),columns=Car_table_real[j][i].columns)   
            IntervalkWh_table_Opt_list[i]  = pd.DataFrame(columns=["Interval start"])             
        # create real-time variables
        IntervalkWh_max_RT = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        Car_table_RT = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_RT= [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_nEta = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        UpperBound = [[] for _ in range(3)] # all, Tesla, and Non-Tesla
        for i in range(len(Cases)+1):      # active RT cases and DA/V0G reference
            for j in range(3):  # all, Tesla, and Non-Tesla
                IntervalkWh_max_RT[j][i] = IntervalkWh_max_fc_fix[j][i].copy()
                Car_table_RT[j][i] = Car_table_fc_fix[j][i].copy()  
                SessionkWh_RT[j][i] = SessionkWh_table_fc_fix[j][i].copy()
                SessionkWh_nEta[j][i] = SessionkWh_table_fc_fix[j][i].copy()
        # If Case1 is enabled, apply Eta_min logic for sessions whose layover does not overlap event hours.
        if RUN_CASE1 and Fc_SessionkWh == 'PerfectSessionkWh':    
            for j in np.array(range(2))+1: #Tesla, non-Tesla                   
                overlap = pd.DataFrame((IntervalkWh_max_RT[j][1]*EventHour_96[1]).sum())
                for car in range(len(overlap)):
                    if overlap.iloc[car,0] == 0:
                        Eta_table_RT[j][1].iloc[:,car] = 1 
                SessionkWh_nEta[j][1] = SessionkWh_table_fc_fix[j][1]*Eta_table_RT[j][1]
            Eta_table_RT[0][1] = pd.concat([Eta_table_RT[1][1], Eta_table_RT[2][1]], axis=1)        
            SessionkWh_nEta[0][1] = pd.concat([SessionkWh_nEta[1][1], SessionkWh_nEta[2][1]], axis=1)   
        for j in range(3):
            UpperBound[j] = SessionkWh_nEta[j][0].copy() #100% base case
        SessionkWh_RT_ML = [[[] for _ in range(len(Cases))] for _ in range(3)] #All EVs, Tesla, and Non-Tesla
        # Shrinking horizon implementation
        Status_list_imp = [[] for _ in range(len(Cases))]  # two cases
        Solver_Outputs_RT = [[] for _ in range(len(Cases))] # two cases
        for i in range(len(Cases)):
            Solver_Outputs_RT[i] = {
                'Cost_Opt': [],
                'Cost_PD': [],
                'Cost_NCD': [],
                'Cost_TOU': [],
                'Revenue_EV': [],
                'Revenue_WM': [],
                'penalty_BESS': [],
                'penalty_WM': [],
                'p_GI': [],
                'p_EV': [],
                'p_ch_BESS': [],
                'p_dch_BESS': [],
                'p_ch_BESS_WM': [],
                'p_dch_BESS_WM': [],
                'p_ch_BESS_NWM': [],
                'p_dch_BESS_NWM': [],
                'p_ch_EV': [],
                'p_dch_EV': [],
            'p_ch_EV_WM': [],
            'p_dch_EV_WM': [],
            'p_ch_EV_NWM': [],
            'p_dch_EV_NWM': [],
                'p_ch_net': [],
                'p_dch_net': [],
                'soc_BESS': [],
                'c_RU_DA': [],
                'c_RD_DA': [],
                'c_SP_DA': [],
                'c_NSP_DA': [],
                'p_DA': [],
                'c_RU_RT': [],
                'c_RD_RT': [],
                'c_SP_RT': [],
                'c_NSP_RT': [],
                'p_RT': [],
                'Baseline(kW)': [],
                'P_EV_max': []
            }
        user_sessions_cache = {}
        for i_t in range(int(24/dt_h)):
            H = int(24/dt_h - i_t)  # Horizon, numb of time slots
            # RT start and end of the receding horizon
            H_Start_RT = TheDate_Day0 + timedelta(minutes=i_t*dt_m_EV)
            H_End_RT = TheDate_Day0 + timedelta(hours=24)
            # real start and end of the following time interval: [t, t+1)
            t_start_interval = H_Start_RT
            t_end_interval = H_Start_RT + timedelta(minutes=dt_m_EV)
            # create time series for H
            Time_table_RT = Time_table_real[0][(H_Start_RT <= Time_table_real[0]['Interval start'])]
            #################################################################################
            # Realization: Update inputs with arrived cars
            #################################################################################
            # at t>0, based on the EventHour (if at least one hour is an event hour), the aggregator decides Eta = Eta_min, or else 100%       
            #if i_t > 0:               
                # Cars arrived in the previous interval: [t-1, t)
            Cars_arrived_LastInterval = np.array(list(Car_table_real[0][i]))[np.array(ArrivalTime_table_real[0][i].iloc[0, :] > t_start_interval-timedelta(minutes=dt_m_EV)) & np.array(ArrivalTime_table_real[0][i].iloc[0, :] <= t_start_interval)]
            if len(Cars_arrived_LastInterval) > 0:
                #the arrival car for loop is outside of the cases loop so the ML forecast is run only once for each car    
                for car_i in Cars_arrived_LastInterval:                            
                    for i in range(len(Cases)):
                        if car_i in Car_table_real[1][0]:
                            j = 1
                            IntervalkWh_CH_max = 4.16
                        elif car_i in Car_table_real[2][0]:
                            j = 2
                            IntervalkWh_CH_max = 1.664
                        else:
                            j = 0
                        # how many columns of sessionkWh_1_real/IntervalkWh_1_real are realized at t
                        Count[j][i] = Count[j][i] + 1
                        # Update column name
                        # note that after updating the RT column name, there might be repeated column names
                        # so always replace the order of the column instead of using column names
                        # if the arrived car number is NOT larger than the current matrix width
                        if Count[j][i] <= len(Car_table_RT[j][i].columns):
                            ColumnName_RT = Car_table_RT[j][i].columns[Count[j][i]-1]
                            ColumnName_real = Car_table_real[j][i].columns[Count[j][i]-1]
                            Car_table_RT[j][i].rename(columns={ColumnName_RT:ColumnName_real}, inplace=True)
                            IntervalkWh_max_RT[j][i].rename(columns={ColumnName_RT:ColumnName_real}, inplace=True)
                            SessionkWh_RT[j][i].rename(columns={ColumnName_RT:ColumnName_real}, inplace=True)
                            SessionkWh_nEta[j][i].rename(columns={ColumnName_RT:ColumnName_real}, inplace=True)
                            if i == 0:
                                UpperBound[j].rename(columns={ColumnName_RT:ColumnName_real}, inplace=True)
                            IntervalkWh_max_RT[j][i].iloc[:,Count[j][i]-1]= IntervalkWh_max_real[j][i].iloc[:,Count[j][i]-1]
                            SessionkWh_RT[j][i].iloc[:,Count[j][i]-1]      = SessionkWh_table_real[j][i].iloc[:,Count[j][i]-1]
                        # if the arrived car number is larger than the current matrix width    
                        else:
                            ColumnName_real = Car_table_real[j][i].columns[Count[j][i]-1]
                            Car_table_RT[j][i] = _append_aligned_column(Car_table_RT[j][i], ColumnName_real, Car_table_real[j][i][ColumnName_real])
                            IntervalkWh_max_RT[j][i] = _append_aligned_column(IntervalkWh_max_RT[j][i], ColumnName_real, IntervalkWh_max_real[j][i][ColumnName_real])
                            SessionkWh_RT[j][i] = _append_aligned_column(SessionkWh_RT[j][i], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                            SessionkWh_nEta[j][i] = _append_aligned_column(SessionkWh_nEta[j][i], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                            if i == 0:
                                UpperBound[j] = _append_aligned_column(UpperBound[j], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                        # if ML forecast upon EV arrival, rewrite sessionkWh_RT and IntervalkWh_max_RT
                        if Fc_AtArrival == 'MLatArrival':
                            # shouod NOT run the ML forecast for two cases separately. Run for 100% base case and copy that to eta% case!!  
                            if i == 0:    
                                SessInfo_ThisUser = All_Sess[All_Sess['Car']==car_i]
                                ThisUser = int(SessInfo_ThisUser['User'])
                                ThisUser_AT = SessInfo_ThisUser['Session start'].iloc[0]
                                if ThisUser in list(User_known):                                   
                                    # find the individual driver data file
                                    if ThisUser not in user_sessions_cache:
                                        Data_ThisUser = pd.read_csv('Sessions_Data/Sessions_Data_'+str(int(ThisUser))+'.csv')
                                        Data_ThisUser['Session start'] = pd.to_datetime(Data_ThisUser['Session start'])
                                        user_sessions_cache[ThisUser] = Data_ThisUser
                                    Data_ThisUser = user_sessions_cache[ThisUser]
                                    Data_ThisUser_Before = Data_ThisUser[Data_ThisUser['Session start']<ThisUser_AT]
                                    # need to have at least 10 sessions BEFORE current session to be counted as a KnownUser                                    
                                    if len(Data_ThisUser_Before)>=10:
                                        # User- type: int, size: 1
                                        # test_sessions_data- type: DataFrmae, size: (1,2), comprised of 'Session start' and 'Arrival Hour'                                        
                                        ThisUser_PD_ED = KnownUser(ThisUser,SessInfo_ThisUser[['Session start', 'Arrival Hour']])
                                    else:
                                        # test_sessions_data- type: DataFrmae, size: (1,5), comprised of 'Session start' and 'Arrival Hour' , 'Battery (kWh)','Weekday', 'Max Charging Power' 
                                        ThisUser_PD_ED = UnKnownUser(SessInfo_ThisUser[['Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']])
                                else:
                                    ThisUser_PD_ED = UnKnownUser(SessInfo_ThisUser[['Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']])
                                # create IntervalkWh_max_RT based on PD  
                                ThisUser_DT = ThisUser_AT + timedelta(minutes=ThisUser_PD_ED[0]) 
                                Logic_ThisUser_Interval = (Time_table_real[0].iloc[:,0]>=ThisUser_AT)&(Time_table_real[0].iloc[:,0]<=ThisUser_DT)
                                ThisUser_Interval = np.array(Logic_ThisUser_Interval.astype(int))*IntervalkWh_CH_max
                            # rewrite current IntervalkWh_max_RT and SessionkWh_RT
                            SessionkWh_RT[j][i].iloc[:,Count[j][i]-1] = ThisUser_PD_ED[1] 
                            IntervalkWh_max_RT[j][i].iloc[:,Count[j][i]-1]= ThisUser_Interval
                            # keep record of the ML forecasted SessionkWh_RT
                            SessionkWh_RT_ML[j][i] =  SessionkWh_RT_ML[j][i] + [ThisUser_PD_ED[1]]
                        # if layover time and event hours don't overlap, Eta_min_i = 100%                         
                        if sum(np.array(IntervalkWh_max_RT[j][i].iloc[:,Count[j][i]-1]).reshape(96,1)*EventHour_96[i] ) == 0: 
                            Eta_table_RT[j][i].iloc[:,Count[j][i]-1] = 1 
                        #SessionkWh_nEta[j][i][ColumnName_real] = SessionkWh_RT[j][i][ColumnName_real]*Eta_table_RT[j][i][ColumnName_real]  
                        SessionkWh_nEta[j][i].iloc[0, Count[j][i]-1] = SessionkWh_RT[j][i].iloc[0, Count[j][i]-1]*Eta_table_RT[j][i].iloc[0, Count[j][i]-1]
                        if i == 1:
                            UpperBound[j].iloc[0, Count[j][i]-1] = SessionkWh_nEta[j][0].iloc[0, Count[j][i]-1].copy() 
                #After updating all the arrival cars for Tesla and non-Tesla of case 100% and eta%, concat Tesla and non-Tesla                    
                for i in range(len(Cases)):                          
                    IntervalkWh_max_RT[0][i] = pd.concat( [IntervalkWh_max_RT[1][i], IntervalkWh_max_RT[2][i]], axis=1)
                    Car_table_RT[0][i] = pd.DataFrame(columns=IntervalkWh_max_RT[0][i].columns)
                    Eta_table_RT[0][i] = pd.concat([Eta_table_RT[1][i], Eta_table_RT[2][i]], axis=1)
                    SessionkWh_RT[0][i] = pd.concat([SessionkWh_RT[1][i], SessionkWh_RT[2][i]], axis=1)
                    SessionkWh_nEta[0][i] = pd.concat([SessionkWh_nEta[1][i], SessionkWh_nEta[2][i]], axis=1)
                    if i==1:
                        UpperBound[0] = pd.concat([UpperBound[1], UpperBound[2]], axis=1)
            IntervalkWh_max_RT_opt = [None for _ in range(len(Cases))]
            for i in range(len(Cases)):   
                # The first MPC action is executed immediately and may only use EVs
                # that have physically arrived. Persistence forecasts remain active
                # for future rows, but forecast-only EVs are forced to zero at row 0.
                _rt_ub_opt = IntervalkWh_max_RT[0][i].iloc[i_t:, :].copy()
                _arrived_ids = {str(c) for c in list(Car_table_RT[1][i].columns[:Count[1][i]]) + list(Car_table_RT[2][i].columns[:Count[2][i]])}
                if len(_rt_ub_opt) > 0:
                    _not_arrived = [k for k, c in enumerate(_rt_ub_opt.columns) if str(c) not in _arrived_ids]
                    if _not_arrived:
                        _rt_ub_opt.iloc[0, _not_arrived] = 0.0
                IntervalkWh_max_RT_opt[i] = _rt_ub_opt
                # Always make sure required energy does not exceed the corrected remaining envelope.
                SessionkWh_nEta[0][i] = pd.DataFrame(pd.concat([SessionkWh_nEta[0][i], pd.DataFrame(_rt_ub_opt.sum()).transpose()]).min(axis=0)).transpose()
            UpperBound[0] = pd.DataFrame(pd.concat([UpperBound[0], pd.DataFrame(IntervalkWh_max_RT[0][0].iloc[i_t:].sum()).transpose()]).min(axis=0)).transpose()
            if RUN_CASE1:
                var_list = [UpperBound[0], SessionkWh_nEta[0][1]]
                name_list = ['UpperBound', 'Session_nEta']
                for var in range(len(var_list)):
                    dir_Output = os.path.join('Results/Dispatch/0_'+name_list[var]+'_'+WM_Mode+'.csv')
                    append_df_to_csv(var_list[var], dir_Output, index=False)
            # Define RT optimization inputs that are time-varying
            Operator_SumColumn = np.ones((Car_table_RT[0][i].shape[1], 1)) # size: num_cars x 1
            Operator_SumRow = np.ones((1, H)) # size: 1 x num_steps
            c_e_TOU_AL_RT = c_e_TOU_AL[i_t:] # $/kWh
            Bid_Pr_DA_i = Bid_Pr_DA[i_t:]
            Bid_Pr_RT_i = Bid_Pr_RT[i_t:]
            AS_Pr_RU_DA_i = AS_Pr_RU_DA[i_t:]
            AS_Pr_RD_DA_i = AS_Pr_RD_DA[i_t:]
            AS_Pr_RU_RT_i = AS_Pr_RU_RT[i_t:]
            AS_Pr_RD_RT_i = AS_Pr_RD_RT[i_t:]
            AS_Pr_SP_DA_i = AS_Pr_SP_DA[i_t:]
            AS_Pr_NSP_DA_i = AS_Pr_NSP_DA[i_t:]
            AS_Pr_SP_RT_i = AS_Pr_SP_RT[i_t:]
            AS_Pr_NSP_RT_i = AS_Pr_NSP_RT[i_t:]
            # EV service revenue with a EV charging rate
            c_EV_service_Day0 = np.ones(96)*0.4 # for daily cost calculation
            c_EV_service = np.ones(H)*0.4 # $/kWh
            # Filter for Peak demand Period
            if i_t < 16*4:
                PP_start = 16*4-i_t
                PP_end = 21*4-i_t
                Filter = 1
            # from 4-9 pm
            elif (i_t >= 16*4) & (i_t < 21*4):
                PP_start = 0
                PP_end = 21*4 - i_t
                Filter = 1
            else:
                PP_start = 0
                PP_end = 1
                Filter = 0
            track = np.zeros(H)
            for i in range(len(Cases)):
                # 1-D array with length H = 96 - i_t
                EventHour_96_i = EventHour_96[i][i_t:].flatten() # TODO: event need update for new DR
                Bid_Pr_cap = Bid_Pr_DA_i * EventHour_96_i
                Baseline_Opt_i = np.nan_to_num(Baseline_96[i][i_t:].flatten(), nan=0.0, posinf=0.0, neginf=0.0) # TODO: current Baseline use the 100% perfect, need to update from DRAM
                NonEventHour_96_i = 1 - EventHour_96_i
                B_EV = Baseline_Opt_i / dt_h
                P_EV_max = 6.6 * Car_table_RT[0][i].shape[1]  # max charging power per car * number of cars, kW
                _ev_agg_ub_kw = np.asarray(IntervalkWh_max_RT_opt[i], dtype=float).sum(axis=1) / dt_h
                M_EV = 1.05 * np.maximum.reduce([_ev_agg_ub_kw, np.abs(np.asarray(B_EV, dtype=float)), np.full(H, 1e-6)])  # interval-specific tight EV big-M
                # Construct the problem
                Cost_Opt = cp.Variable() # scalar
                penalty_WM = cp.Constant(0.0)
                penalty_BESS = 0
                penalty_eta = 0
                # EV Variables (cannot prebuild bc they depend on the number of cars at each time step)
                EnergyDemand_Table_Opt = cp.Variable(shape=(H, Car_table_RT[0][i].shape[1])) # H x num_cars
                EnergyDemand_Step_Opt = cp.Variable(H)
                p_EV = cp.Variable(H, nonneg=True)  # EV charging power
                # Power variables
                p_GI = cp.Variable(H)  # grid import power, kW
                # BESS variables
                p_BESS = cp.Variable(H)
                p_ch_BESS = cp.Variable(H, nonneg = True)
                p_dch_BESS = cp.Variable(H, nonneg = True)
                p_ch_BESS_WM = cp.Variable(H, nonneg = True)
                p_dch_BESS_WM = cp.Variable(H, nonneg = True)
                p_ch_BESS_NWM = cp.Variable(H, nonneg = True)
                p_dch_BESS_NWM = cp.Variable(H, nonneg = True)
                soc_BESS = cp.Variable(H+1) # SOC from 0 to H                
                # SOC initialization
                if i_t == 0:
                    if Day == run_days[0]:
                        SOC_BESS_initial = 0.5
                    else:
                        SOC_BESS_initial = SOC_BESS_daily_end[i]
                else:
                    SOC_BESS_initial = SOC_BESS_last_step[i]
                # BESS constraints
                constraints_bess = [
                    p_BESS == p_ch_BESS - p_dch_BESS, # either positive or negative
                    p_ch_BESS == p_ch_BESS_WM + p_ch_BESS_NWM,
                    p_dch_BESS == p_dch_BESS_WM + p_dch_BESS_NWM,
                    p_BESS <= P_BESS_max,
                    p_BESS >= -P_BESS_max,
                    soc_BESS[0] == SOC_BESS_initial,
                    soc_BESS <= SOC_BESS_max,
                    soc_BESS >= SOC_BESS_min,
                    soc_BESS[1:] == soc_BESS[:-1] + (dt_h / C_BESS) * (cp.multiply(p_ch_BESS, np.sqrt(gamma)) - cp.multiply(p_dch_BESS, 1 / np.sqrt(gamma))),
                    dt_h * cp.sum(p_ch_BESS + p_dch_BESS) + E_BESS_throughput_RT[i, i_t] <= 2 * C_BESS * (SOC_BESS_max - SOC_BESS_min)  # limit total throughput in one day
                ]
                if is_last_run_day:
                    constraints_bess += [soc_BESS[-1] == 0.5]
                    penalty_BESS_target = 0.5
                    penalty_BESS = (cp.norm(soc_BESS[-1] - penalty_BESS_target, 2)) * c_BESS_penalty # penalty coefficient
                else:
                    penalty_BESS = cp.Constant(0.0)
                # Power balance
                constraints_balance = [
                    p_GI == p_EV + p_BESS
                ]
                # WM constraints
                p_DA = cp.Variable(H)
                p_RT = cp.Variable(H)
                c_RU_DA = cp.Variable(H, nonneg=True)
                c_RD_DA = cp.Variable(H, nonneg=True)
                c_SP_DA = cp.Variable(H, nonneg=True)
                c_NSP_DA = cp.Variable(H, nonneg=True)
                # Signed RT AS adjustments: actual delivered capacity is c_DA + c_RT.
                c_RU_RT = cp.Variable(H)
                c_RD_RT = cp.Variable(H)
                c_SP_RT = cp.Variable(H)
                c_NSP_RT = cp.Variable(H)
                p_up = P_BESS_max + B_EV
                p_down = P_BESS_max - B_EV + P_EV_max
                p_ch_EV_WM = cp.Variable(H, nonneg=True)      # EV baseline deviation used for WM products
                p_dch_EV_WM = cp.Variable(H, nonneg=True)     # EV baseline deviation used for WM products
                p_ch_EV_NWM = cp.Variable(H, nonneg=True)  # EV baseline deviation needed for non-WM service feasibility
                p_dch_EV_NWM = cp.Variable(H, nonneg=True)
                p_ch_net = cp.Variable(H, nonneg=True)
                p_dch_net = cp.Variable(H, nonneg=True)
                b_ch_BESS = cp.Variable(H, boolean=True)
                b_dch_BESS = cp.Variable(H, boolean=True)
                b_ch_EV = cp.Variable(H, boolean=True)
                b_dch_EV = cp.Variable(H, boolean=True)
                b_ch_net = cp.Variable(H, boolean=True)
                b_dch_net = cp.Variable(H, boolean=True)
                alpha_RU = 0.7
                alpha_RD = 0.7
                alpha_SP = 0.2
                alpha_NSP = 0.2
                p_da_committed = np.asarray(p_DA_value_DA[i_t:], dtype=float)
                # Historical settlement parameterization (fd0c8c3): internal p_RT is the actual RT position.
                # The paper/settlement p_RT is the increment or decrement from the fixed DA commitment.
                p_wm_actual = p_RT if RT_DA_COMMITMENT_MODE == 'settlement' else p_DA + p_RT
                p_rt_settled = p_RT - p_da_committed if RT_DA_COMMITMENT_MODE == 'settlement' else p_RT
                if Enable_WM:
                    constraints_WM = [
                        c_RU_DA + c_RU_RT >= 0,
                        c_RD_DA + c_RD_RT >= 0,
                        c_SP_DA + c_SP_RT >= 0,
                        c_NSP_DA + c_NSP_RT >= 0,
                        # Capacity
                        # p_DA + p_RT + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                        # -(p_DA + p_RT) + c_RD_DA + c_RD_RT <= p_down,
                        # CAISO two-settlement logic: physical capability follows the net DA+RT position.
                        cp.pos(p_wm_actual) + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                        cp.pos(-p_wm_actual) + c_RD_DA + c_RD_RT <= p_down,
                        # Energy
                        b_ch_BESS + b_dch_BESS <= 1,
                        b_ch_EV + b_dch_EV <= 1,
                        b_ch_net + b_dch_net <= 1,
                        p_EV - B_EV == (p_ch_EV_WM - p_dch_EV_WM) + (p_ch_EV_NWM - p_dch_EV_NWM),
                        # p_dch_EV_WM == (B_EV - p_EV) * b_dch_EV,
                        p_dch_EV_WM + p_dch_EV_NWM <= cp.multiply(M_EV, b_dch_EV),
                        # p_ch_EV_WM == -(B_EV - p_EV) * b_ch_EV,
                        p_ch_EV_WM + p_ch_EV_NWM <= cp.multiply(M_EV, b_ch_EV),
                        p_dch_net <= p_wm_actual + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT) + cp.multiply(p_up, b_ch_net),
                        p_dch_net >= p_wm_actual + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT),
                        p_dch_BESS <= P_BESS_max * b_dch_BESS,
                        p_ch_net <= -(p_wm_actual + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT)) + cp.multiply(p_down, b_dch_net),
                        p_ch_net >= -(p_wm_actual + alpha_RU * (c_RU_DA + c_RU_RT) + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT) - alpha_RD * (c_RD_DA + c_RD_RT)),
                        p_ch_BESS <= P_BESS_max * b_ch_BESS,
                        p_ch_net <= cp.multiply(p_down, b_ch_net),
                        p_dch_net <= cp.multiply(p_up, b_dch_net),
                        p_ch_net - p_dch_net == p_ch_BESS_WM - p_dch_BESS_WM + p_ch_EV_WM - p_dch_EV_WM,
                        # Duration
                        soc_BESS[1:] >= SOC_BESS_min + (2 * dt_h) / (C_BESS * np.sqrt(gamma)) * (c_RU_DA + c_RU_RT + c_SP_DA + c_SP_RT + c_NSP_DA + c_NSP_RT),
                        soc_BESS[1:] <= SOC_BESS_max - (2 * dt_h * np.sqrt(gamma)) / C_BESS * (c_RD_DA + c_RD_RT)
                    ]
                    # VERSION: Add-ons when direction aligned
                    if Direction_Aligned_AddOn:
                        constraints_WM += [
                            c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= cp.multiply(p_up, b_dch_net),
                            c_RD_DA + c_RD_RT <= cp.multiply(p_down, b_ch_net),
                        ]
                    # VERSION: WM only mode (disable non-WM BESS channel)
                    if WM_Mode == 'wm_only':
                        constraints_WM += [
                            p_ch_BESS_NWM == 0,
                            p_dch_BESS_NWM == 0,
                        ]
                else:
                    constraints_WM = [
                        p_DA == 0, p_RT == 0,
                        c_RU_DA == 0, c_RD_DA == 0, c_SP_DA == 0, c_NSP_DA == 0,
                        c_RU_RT == 0, c_RD_RT == 0, c_SP_RT == 0, c_NSP_RT == 0,
                        p_ch_BESS_WM == 0, p_dch_BESS_WM == 0,
                        p_ch_EV_WM == 0, p_dch_EV_WM == 0, p_ch_net == 0, p_dch_net == 0,
                    ]
                if Enable_WM:
                    if RT_DA_COMMITMENT_MODE == 'hard':
                        constraints_WM += [
                            c_RU_DA == c_RU_DA_value_DA[i_t:],
                            c_RD_DA == c_RD_DA_value_DA[i_t:],
                            c_SP_DA == c_SP_DA_value_DA[i_t:],
                            c_NSP_DA == c_NSP_DA_value_DA[i_t:],
                            p_DA == p_DA_value_DA[i_t:],
                        ]
                        penalty_WM = cp.Constant(0.0)
                    elif RT_DA_COMMITMENT_MODE == 'settlement':
                        # DA energy and DA AS awards are immutable RT obligations.
                        # This simplified case assumes the resource remains available; CAISO
                        # forced-buyback/no-pay settlement for outages is outside model scope.
                        constraints_WM += [
                            p_DA == 0,
                            c_RU_DA == np.maximum(np.asarray(c_RU_DA_value_DA[i_t:], dtype=float), 0.0),
                            c_RD_DA == np.maximum(np.asarray(c_RD_DA_value_DA[i_t:], dtype=float), 0.0),
                            c_SP_DA == np.maximum(np.asarray(c_SP_DA_value_DA[i_t:], dtype=float), 0.0),
                            c_NSP_DA == np.maximum(np.asarray(c_NSP_DA_value_DA[i_t:], dtype=float), 0.0),
                        ]
                        penalty_WM = cp.Constant(0.0)
                    elif RT_DA_COMMITMENT_MODE == 'penalty':
                        penalty_WM = M * cp.sum(cp.multiply(
                            np.abs(Bid_Pr_RT_i),
                            cp.abs(c_RU_DA - c_RU_DA_value_DA[i_t:])
                            + cp.abs(c_RD_DA - c_RD_DA_value_DA[i_t:])
                            + cp.abs(c_SP_DA - c_SP_DA_value_DA[i_t:])
                            + cp.abs(c_NSP_DA - c_NSP_DA_value_DA[i_t:])
                            + cp.abs(p_DA - p_DA_value_DA[i_t:])
                        ))
                    else:
                        raise ValueError("RT_DA_COMMITMENT_MODE must be 'settlement', 'hard', or 'penalty'")
                    # WM Revenue calculation # TODO: for i_t each 4 times the P values should be the same that bid is hourly
                    Revenue_WM = (
                        dt_h * (
                            Bid_Pr_RT_i @ (alpha_RU * (c_RU_DA + c_RU_RT) - alpha_RD * (c_RD_DA + c_RD_RT)
                                        + alpha_SP * (c_SP_DA + c_SP_RT) + alpha_NSP * (c_NSP_DA + c_NSP_RT))
                            + Bid_Pr_DA_i @ (p_da_committed if RT_DA_COMMITMENT_MODE == 'settlement' else p_DA) + Bid_Pr_RT_i @ p_rt_settled
                            + AS_Pr_RU_DA_i @ c_RU_DA + AS_Pr_RU_RT_i @ c_RU_RT + AS_Pr_RD_DA_i @ c_RD_DA + AS_Pr_RD_RT_i @ c_RD_RT
                            + AS_Pr_SP_DA_i @ c_SP_DA + AS_Pr_SP_RT_i @ c_SP_RT + AS_Pr_NSP_DA_i @ c_NSP_DA + AS_Pr_NSP_RT_i @ c_NSP_RT
                        )
                    )
                else:
                    penalty_WM = cp.Constant(0.0)
                    Revenue_WM = 0
                # Cost-related constraints
                constraints_cost = [
                    Cost_Opt >= (
                        c_NCD * cp.maximum(cp.max(p_GI) - M_Th_NCD[i], 0)
                        + c_PD * cp.maximum(cp.max(Filter * p_GI[PP_start:PP_end]) - M_Th_PD[i], 0)
                        + c_e_TOU_AL_RT @ p_GI * dt_h
                        - c_EV_service @ p_EV * dt_h
                        - Revenue_WM
                    )
                ]
                # EV constraints
                if i == 0:
                    constraints_EV = [
                        p_EV == EnergyDemand_Step_Opt / dt_h,  # kW
                        EnergyDemand_Table_Opt @ Operator_SumColumn == cp.reshape(EnergyDemand_Step_Opt, (H, 1), order = 'C'),
                        Operator_SumRow @ EnergyDemand_Table_Opt == SessionkWh_nEta[0][0],
                        EnergyDemand_Table_Opt >= 0,
                        EnergyDemand_Table_Opt <= IntervalkWh_max_RT_opt[i]
                    ]
                elif i == 1:
                    constraints_EV = [
                        p_EV == EnergyDemand_Step_Opt / dt_h,  # kW
                        EnergyDemand_Table_Opt @ Operator_SumColumn == cp.reshape(EnergyDemand_Step_Opt, (H, 1), order = 'C'),
                        Operator_SumRow @ EnergyDemand_Table_Opt >= SessionkWh_nEta[0][1],
                        Operator_SumRow @ EnergyDemand_Table_Opt <= UpperBound[0],
                        EnergyDemand_Table_Opt >= 0,
                        EnergyDemand_Table_Opt <= IntervalkWh_max_RT_opt[i]
                    ]
                    # soft tracking penalty
                    # penalty_eta += M * cp.sum(cp.multiply(NonEventHour_96_i, cp.abs((EnergyDemand_Step_Opt - track))))
                # NOTE: combine all constraints
                constraints = constraints_cost + constraints_EV + constraints_bess + constraints_WM + constraints_balance
                # solver_opts = {"Method": 2, "Seed": 0, "Threads": 1} # ensures deterministic results
                objective = cp.Minimize(Cost_Opt + penalty_BESS + penalty_WM + penalty_eta)
                prob = cp.Problem(objective, constraints)
                rt_solver_opts = dict(MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1)
                _rt_watchdog_limit = RT_SOLVER_TIME_LIMIT if RT_SOLVER_TIME_LIMIT is not None else RT_SOLVER_EMERGENCY_TIME_LIMIT
                if _rt_watchdog_limit is not None:
                    rt_solver_opts['TimeLimit'] = _rt_watchdog_limit
                result = prob.solve(solver=cp.GUROBI, verbose=False, **rt_solver_opts)
                status = prob.status
                if status == 'infeasible_or_unbounded':
                    rt_solver_opts_reopt = dict(rt_solver_opts)
                    rt_solver_opts_reopt['reoptimize'] = True
                    try:
                        result = prob.solve(solver=cp.GUROBI, verbose=False, **rt_solver_opts_reopt)
                        status = prob.status
                    except Exception as _reopt_err:
                        print(f"RT reoptimize failed while diagnosing infeasible_or_unbounded: {_reopt_err}")
                _record_rt_solver_status(prob, H_Start_RT, i_t, Cases[i])
                Status_list_imp[i] = Status_list_imp[i] + [status]
                if (REQUIRE_RT_OPTIMAL and status != 'optimal') or status not in ['optimal', 'optimal_inaccurate', 'user_limit'] or p_GI.value is None or EnergyDemand_Table_Opt.value is None:
                    try:
                        _remaining_energy = float(np.nansum(np.asarray(SessionkWh_nEta[0][i]).astype(float)))
                    except Exception:
                        _remaining_energy = np.nan
                    try:
                        _remaining_envelope = float(np.nansum(np.asarray(IntervalkWh_max_RT_opt[i]).astype(float)))
                    except Exception:
                        _remaining_envelope = np.nan
                    _b_ev_arr = np.asarray(B_EV, dtype=float)
                    _p_down_arr = np.asarray(p_down, dtype=float)
                    _p_up_arr = np.asarray(p_up, dtype=float)
                    print("RT optimization does not have a usable solution!")
                    print(f"  status={status}, month={month}, day={Day}, date={pd.Timestamp(TheDate_Day0).strftime('%Y-%m-%d')}, step={i_t}, time={H_Start_RT}, case={Cases[i]}, WM_Mode={WM_Mode}, Enable_WM={Enable_WM}")
                    print(f"  forecast={Fc_SessionkWh}|{Fc_NumbEV}|{Fc_AtArrival}, SOC_initial={float(SOC_BESS_initial):.6f}, P_EV_max={float(P_EV_max):.6f}, M_EV={float(M_EV):.6f}")
                    print(f"  B_EV[kW]: min={float(np.nanmin(_b_ev_arr)):.6f}, max={float(np.nanmax(_b_ev_arr)):.6f}, sum={float(np.nansum(_b_ev_arr)):.6f}, nan={int(np.isnan(_b_ev_arr).sum())}")
                    print(f"  p_up[kW]: min={float(np.nanmin(_p_up_arr)):.6f}, max={float(np.nanmax(_p_up_arr)):.6f}; p_down[kW]: min={float(np.nanmin(_p_down_arr)):.6f}, max={float(np.nanmax(_p_down_arr)):.6f}, negative_count={int(np.sum(_p_down_arr < -1e-7))}")
                    try:
                        _throughput_used = float(E_BESS_throughput_RT[i, i_t])
                        _throughput_cap = float(2 * C_BESS * (SOC_BESS_max - SOC_BESS_min))
                        _min_charge_to_soc_min = max(0.0, float((SOC_BESS_min - SOC_BESS_initial) * C_BESS / (dt_h * np.sqrt(gamma))))
                        _max_discharge_to_soc_min = max(0.0, float((SOC_BESS_initial - SOC_BESS_min) * C_BESS * np.sqrt(gamma) / dt_h))
                        print(f"  BESS throughput: used={_throughput_used:.6f} kWh, cap={_throughput_cap:.6f} kWh, remaining={_throughput_cap - _throughput_used:.6f} kWh, max_discharge_to_min_now={_max_discharge_to_soc_min:.6f} kW")
                    except Exception as _bess_diag_err:
                        print(f"  BESS throughput diagnostic failed: {_bess_diag_err}")
                    print(f"  EV remaining session target kWh={_remaining_energy:.6f}, remaining interval envelope kWh={_remaining_envelope:.6f}, target-minus-envelope={_remaining_energy - _remaining_envelope:.6f}")
                    try:
                        _target_by_car = pd.to_numeric(SessionkWh_nEta[0][i].iloc[0], errors='coerce').fillna(0.0)
                        _envelope_by_car = IntervalkWh_max_RT_opt[i].sum(axis=0)
                        _envelope_by_car = pd.to_numeric(_envelope_by_car.reindex(_target_by_car.index).fillna(0.0), errors='coerce').fillna(0.0)
                        _excess_by_car = _target_by_car - _envelope_by_car
                        _bad_excess = _excess_by_car[_excess_by_car > 1e-5].sort_values(ascending=False)
                        print(f"  EV per-car feasibility: max_excess={float(_excess_by_car.max()):.6f} kWh, bad_cars={len(_bad_excess)}")
                        if len(_bad_excess) > 0:
                            _worst = _bad_excess.head(5)
                            print("  Worst EV per-car excess kWh:", {str(k): float(v) for k, v in _worst.items()})
                    except Exception as _ev_diag_err:
                        print(f"  EV per-car diagnostic failed: {_ev_diag_err}")
                    try:
                        _rd_da = np.asarray(c_RD_DA_value_DA[i_t:], dtype=float)
                        _ru_da = np.asarray(c_RU_DA_value_DA[i_t:], dtype=float)
                        _p_da = np.asarray(p_DA_value_DA[i_t:], dtype=float)
                        print(f"  DA commitment remaining: p_DA sum={float(np.nansum(_p_da)):.6f}, min={float(np.nanmin(_p_da)):.6f}, max={float(np.nanmax(_p_da)):.6f}; RU sum={float(np.nansum(_ru_da)):.6f}, max={float(np.nanmax(_ru_da)):.6f}; RD sum={float(np.nansum(_rd_da)):.6f}, max={float(np.nanmax(_rd_da)):.6f}, SP sum={float(np.nansum(c_SP_DA_value_DA[i_t:])):.6f}, NSP sum={float(np.nansum(c_NSP_DA_value_DA[i_t:])):.6f}")
                    except Exception as _diag_err:
                        print(f"  DA commitment diagnostic failed: {_diag_err}")
                    sys.exit()
                # Save each step RT result
                _p_da_commit_0 = float(p_da_committed[0]) if RT_DA_COMMITMENT_MODE == 'settlement' else float(p_DA.value[0])
                _p_rt_deviation_0 = float(p_RT.value[0] - _p_da_commit_0) if RT_DA_COMMITMENT_MODE == 'settlement' else float(p_RT.value[0])
                revenue_WM_i_t = (
                    dt_h * (
                        Bid_Pr_RT_i[0] * (alpha_RU * (c_RU_DA.value[0] + c_RU_RT.value[0]) - alpha_RD * (c_RD_DA.value[0] + c_RD_RT.value[0]) + alpha_SP * (c_SP_DA.value[0] + c_SP_RT.value[0]) + alpha_NSP * (c_NSP_DA.value[0] + c_NSP_RT.value[0]))
                        + Bid_Pr_DA_i[0] * _p_da_commit_0 + Bid_Pr_RT_i[0] * _p_rt_deviation_0
                        + AS_Pr_RU_DA_i[0] * c_RU_DA.value[0] + AS_Pr_RU_RT_i[0] * c_RU_RT.value[0] + AS_Pr_RD_DA_i[0] * c_RD_DA.value[0] + AS_Pr_RD_RT_i[0] * c_RD_RT.value[0]
                        + AS_Pr_SP_DA_i[0] * c_SP_DA.value[0] + AS_Pr_SP_RT_i[0] * c_SP_RT.value[0] + AS_Pr_NSP_DA_i[0] * c_NSP_DA.value[0] + AS_Pr_NSP_RT_i[0] * c_NSP_RT.value[0]
                    )
                )
                SOC_BESS_last_step[i] = soc_BESS.value[1]
                Revenue_WM_value_RT[i, i_t] = revenue_WM_i_t.item()
                Revenue_EV_value_RT[i, i_t] = c_EV_service[0] * p_EV.value[0] * dt_h
                p_ch_BESS_value_RT[i, i_t] = p_ch_BESS.value[0]
                p_dch_BESS_value_RT[i, i_t] = p_dch_BESS.value[0]
                p_ch_BESS_WM_value_RT[i, i_t] = p_ch_BESS_WM.value[0]
                p_dch_BESS_WM_value_RT[i, i_t] = p_dch_BESS_WM.value[0]
                p_ch_BESS_NWM_value_RT[i, i_t] = p_ch_BESS_NWM.value[0]
                p_dch_BESS_NWM_value_RT[i, i_t] = p_dch_BESS_NWM.value[0]
                p_ch_EV_value_RT[i, i_t] = p_ch_EV_WM.value[0]
                p_dch_EV_value_RT[i, i_t] = p_dch_EV_WM.value[0]
                if p_ch_EV_NWM.value is None or p_dch_EV_NWM.value is None:
                    _b_ev0 = float(prm_b_ev.value[0]) if 'prm_b_ev' in locals() else float(B_EV[0])
                    _ev_resid0 = float(p_EV.value[0]) - _b_ev0 - (float(p_ch_EV_WM.value[0]) - float(p_dch_EV_WM.value[0]))
                    p_ch_EV_NWM_value_RT[i, i_t] = max(_ev_resid0, 0.0)
                    p_dch_EV_NWM_value_RT[i, i_t] = max(-_ev_resid0, 0.0)
                else:
                    p_ch_EV_NWM_value_RT[i, i_t] = p_ch_EV_NWM.value[0]
                    p_dch_EV_NWM_value_RT[i, i_t] = p_dch_EV_NWM.value[0]
                p_ch_net_value_RT[i, i_t] = p_ch_net.value[0]
                p_dch_net_value_RT[i, i_t] = p_dch_net.value[0]
                soc_BESS_value_RT[i, i_t] = soc_BESS.value[1]
                c_RU_DA_value_RT[i, i_t] = c_RU_DA.value[0]
                c_RD_DA_value_RT[i, i_t] = c_RD_DA.value[0]
                c_SP_DA_value_RT[i, i_t] = c_SP_DA.value[0]
                c_NSP_DA_value_RT[i, i_t] = c_NSP_DA.value[0]
                c_RU_RT_value_RT[i, i_t] = c_RU_RT.value[0]
                c_RD_RT_value_RT[i, i_t] = c_RD_RT.value[0]
                c_SP_RT_value_RT[i, i_t] = c_SP_RT.value[0]
                c_NSP_RT_value_RT[i, i_t] = c_NSP_RT.value[0]
                p_DA_value_RT[i, i_t] = _p_da_commit_0
                p_RT_value_RT[i, i_t] = _p_rt_deviation_0
                p_GI_value_RT[i, i_t] = p_GI.value[0]
                p_EV_value_RT[i, i_t] = p_EV.value[0]
                p_BESS_value_RT[i, i_t] = p_BESS.value[0]
                p_EV_max_value_RT[i, i_t] = P_EV_max
                penalty_BESS_value_RT[i, i_t] = penalty_BESS.value.item()
                penalty_WM_value_RT[i, i_t] = penalty_WM.value.item() if Enable_WM else 0.0
                E_BESS_throughput_RT[i, i_t+1] = E_BESS_throughput_RT[i, i_t] + dt_h * (p_ch_BESS.value[0] + p_dch_BESS.value[0])     
                # only store the current time step cost/revenue values
                daily_pd = c_PD * np.maximum(np.max(p_GI.value[PP_start:PP_end]) - M_Th_PD[i], 0)
                Cost_PD_value_RT[i, i_t] = daily_pd
                daily_ncd = c_NCD * np.maximum(np.max(p_GI.value) - M_Th_NCD[i], 0)
                Cost_NCD_value_RT[i, i_t] = daily_ncd
                Cost_TOU_value_RT[i, i_t] = c_e_TOU_AL_RT[0] * p_GI.value[0] * dt_h
                Cost_Opt_value_RT[i, i_t] = Cost_PD_value_RT[i, i_t] + Cost_NCD_value_RT[i, i_t] + Cost_TOU_value_RT[i, i_t] - Revenue_WM_value_RT[i, i_t] - Revenue_EV_value_RT[i, i_t]
                # Map optimized EV schedules to physically arrived vehicles by EV ID.
                # Positional slicing is invalid for Persistence forecasts because the
                # combined forecast column order need not remain [arrived type 1, arrived type 2].
                EnergyDemand_Table_Opt_value = np.asarray(EnergyDemand_Table_Opt.value, dtype=float)
                EnergyDemand_Table_Opt_value[EnergyDemand_Table_Opt_value <= 1e-4] = 0.0
                model_ev_cols = [str(c) for c in Car_table_RT[0][i].columns]
                df_opt_full = pd.DataFrame(EnergyDemand_Table_Opt_value, columns=model_ev_cols)
                arrived_type1 = [str(c) for c in Car_table_RT[1][i].columns[:Count[1][i]]]
                arrived_type2 = [str(c) for c in Car_table_RT[2][i].columns[:Count[2][i]]]
                EnergyDemand_Table_Opt_cap[0][i] = df_opt_full
                EnergyDemand_Table_Opt_cap[1][i] = df_opt_full.reindex(columns=arrived_type1, fill_value=0.0)
                EnergyDemand_Table_Opt_cap[2][i] = df_opt_full.reindex(columns=arrived_type2, fill_value=0.0)
                EnergyDemand_Step_Opt_cap[0][i] = EnergyDemand_Table_Opt_cap[0][i].sum(axis=1)
                if i == 0:
                    track = np.array(EnergyDemand_Step_Opt_cap[0][0]) # kWh
                IntervalkWh_Opt = pd.DataFrame(list(zip(Time_table_RT['Interval start'], EnergyDemand_Step_Opt_cap[0][i])), columns=["Interval start", i_t])
                IntervalkWh_Opt_ = pd.merge(Time_table_real[0], IntervalkWh_Opt, how="left", on=["Interval start"])
                IntervalkWh_table_Opt_list[i] = IntervalkWh_table_Opt_list[i].merge(IntervalkWh_Opt_, on="Interval start", how='outer')
                # record the schedule at t=0, will be used for the aggregator to decide if Eta = Eta_min or 100%
                if i_t == 0:
                    dispatch_t0[i] = p_GI.value
            # end of for i in range(len(Cases))
            #################################################################################
            # EXECUTION
            #################################################################################
            # Dispatch to all the arrived cars for both Type 1 and 2
            for i in range(len(Cases)):
                for j in np.array(range(2))+1:
                    if i == 0:
                        Dispatch[j][i].iloc[i_t, :Count[j][i]] = np.minimum(np.array(EnergyDemand_Table_Opt_cap[j][i].iloc[0, :Count[j][i]]), np.array(SessionkWh_nEta[j][0].iloc[0, :Count[j][i]]))
                    else:
                        Dispatch[j][i].iloc[i_t, :Count[j][i]] = np.minimum(np.array(EnergyDemand_Table_Opt_cap[j][i].iloc[0, :Count[j][i]]), np.array(UpperBound[j].iloc[0, :Count[j][i]]))  
                    # Update the Session left
                    SessionkWh_nEta[j][i].iloc[0, :Count[j][i]] = np.array(SessionkWh_nEta[j][i].iloc[0, :Count[j][i]]) - np.array(Dispatch[j][i].iloc[i_t, :Count[j][i]])
                    #sometimes SessionkWh_nEta = -1*10^-8 because MPC output is capped with 0.0001                                     
                    #SessionkWh_nEta[j][i].iloc[0, :Count[j][i]] = np.maximum((SessionkWh_nEta[j][i].iloc[0, :Count[j][i]]),0) 
                    SessionkWh_nEta[j][i][SessionkWh_nEta[j][i] <= 0.0001] = 0                            
                    if i==1:
                            UpperBound[j].iloc[0, :Count[j][0]] = np.array(UpperBound[j].iloc[0, :Count[j][0]]) - np.array(Dispatch[j][1].iloc[i_t, :Count[j][0]])
                            UpperBound[j][UpperBound[j] <= 0.0001] = 0
                Dispatch[0][i] = pd.concat([Dispatch[1][i], Dispatch[2][i]], axis=1)      
                SessionkWh_nEta[0][i] = pd.concat([SessionkWh_nEta[1][i], SessionkWh_nEta[2][i]], axis=1)
                if i==1:
                    UpperBound[0] = pd.concat([UpperBound[1], UpperBound[2]], axis=1)                         
                Th_NCD[i] = max((Dispatch[0][i].sum(axis=1))/dt_h + p_BESS_value_RT[i, :]) # TODO: can i use P_GI_value_RT here?
                Th_PD[i] = max((Dispatch[0][i].sum(axis=1)[16*4:21*4])/dt_h + p_BESS_value_RT[i, 16*4:21*4])
                M_Th_NCD[i] = np.maximum(Th_NCD[i],M_Th_NCD[i])
                M_Th_PD[i] = np.maximum(Th_PD[i],M_Th_PD[i])
                M_Th_NCD_list_96[i] = M_Th_NCD_list_96[i] + [M_Th_NCD[i]]
                M_Th_PD_list_96[i] = M_Th_PD_list_96[i] + [M_Th_PD[i]]
                D_Th_NCD_96[i] = D_Th_NCD_96[i] + [M_Th_NCD[i]]
                D_Th_PD_96[i] = D_Th_PD_96[i] + [M_Th_PD[i]]
                if i_t == 95:
                    M_Th_NCD_list[i] = M_Th_NCD_list[i] + [M_Th_NCD[i]]
                    M_Th_PD_list[i] = M_Th_PD_list[i] + [M_Th_PD[i]]
            if RUN_CASE1:
                var_list = [pd.DataFrame(EnergyDemand_Table_Opt_cap[0][1].iloc[0, :]).T,
                            UpperBound[0], 
                            SessionkWh_nEta[0][1], 
                            pd.DataFrame(Dispatch[0][1].iloc[i_t, :]).T]
                name_list = ['MPC', 'UpperBound', 'Session_nEta', 'Dispatch_withoutBESS']
                for var in range(len(var_list)):
                    dir_Output = os.path.join('Results/Dispatch/1_'+name_list[var]+'_'+WM_Mode+'.csv')
                    append_df_to_csv(var_list[var], dir_Output, index=True)
            # end of for i in range(len(Cases)) 
        # end of for i_t in range(96)
        #################################################################################
        executed_ev_kW = np.vstack([
            np.asarray(Dispatch[0][i].sum(axis=1), dtype=float) / dt_h
            for i in range(len(Cases))
        ])
        execution_meter_error_kW = float(np.max(np.abs(
            p_GI_value_RT - (executed_ev_kW + p_BESS_value_RT)
        )))
        if execution_meter_error_kW > 1e-3:
            raise ValueError(
                f'Executed-dispatch/meter mismatch: {execution_meter_error_kW:.6g} kW. '
                'Check EV-ID mapping between the forecast optimization table and arrived vehicles.'
            )
        #################################################################################
        # Save daily results
        #################################################################################
        for i in range(len(Cases)):
            Solver_Outputs_RT[i]['Cost_Opt'] = Cost_Opt_value_RT[i].flatten()
            Solver_Outputs_RT[i]['penalty_BESS'] = penalty_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['penalty_WM'] = penalty_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Revenue_WM'] = Revenue_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS'] = p_ch_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS'] = p_dch_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS_WM'] = p_ch_BESS_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS_WM'] = p_dch_BESS_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS_NWM'] = p_ch_BESS_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS_NWM'] = p_dch_BESS_NWM_value_RT[i].flatten()
            # Backward-compatible output keys: p_ch_EV/p_dch_EV now store the EV WM channel.
            Solver_Outputs_RT[i]['p_ch_EV'] = p_ch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV'] = p_dch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_EV_WM'] = p_ch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV_WM'] = p_dch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_EV_NWM'] = p_ch_EV_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV_NWM'] = p_dch_EV_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_net'] = p_ch_net_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_net'] = p_dch_net_value_RT[i].flatten()
            Solver_Outputs_RT[i]['soc_BESS'] = soc_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_BESS'] = p_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RU_DA'] = c_RU_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RD_DA'] = c_RD_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_SP_DA'] = c_SP_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_NSP_DA'] = c_NSP_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_DA'] = p_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RU_RT'] = c_RU_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RD_RT'] = c_RD_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_SP_RT'] = c_SP_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_NSP_RT'] = c_NSP_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_RT'] = p_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_GI'] = p_GI_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_EV'] = p_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_PD'] = Cost_PD_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_NCD'] = Cost_NCD_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_TOU'] = Cost_TOU_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Revenue_EV'] = Revenue_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Baseline(kW)'] = Baseline_96[i].flatten() / dt_h
            Solver_Outputs_RT[i]['P_EV_max'] = p_EV_max_value_RT[i].flatten()
        Solver_Outputs_RT_Base = Solver_Outputs_RT[0]  # 100% eta for implementation
        Solver_Outputs_RT_Case1 = Solver_Outputs_RT[1] if len(Cases) > 1 else None
        # Update the SOC for the next day
        for i in range(len(Cases)):
            SOC_BESS_daily_end[i] = Solver_Outputs_RT[i]['soc_BESS'][-1]
            if not np.isclose(SOC_BESS_daily_end[i], SOC_BESS_last_step[i], atol=1e-5):
                print("SOC continuity error")
                sys.exit()
            if is_last_run_day and (not np.isclose(SOC_BESS_daily_end[i], 0.5, atol=1e-4)):
                print("Final-day SOC terminal error: SOC must return to 0.5")
                sys.exit()
        # print(f"Day {Day} end SOC by case: {[float(SOC_BESS_daily_end[k]) for k in range(len(Cases))]}")
        M_ED_V0G = M_ED_V0G + [sum(EnergyDemand_Step_V0G_TheDates[0])]       
        M_ED_V1G = M_ED_V1G + [sum(EnergyDemand_Step_V1G_TheDates[0])] 
        M_ED_Opt_DA = M_ED_Opt_DA + [sum(p_GI_DA * dt_h)]
        M_ED_Opt_t0 = M_ED_Opt_t0 + [sum(dispatch_t0[0] * dt_h)] 
        M_ED_Opt_base = M_ED_Opt_base + [sum(Dispatch[0][0].sum(axis=1) + p_BESS_value_RT[0, :] * dt_h)]
        if RUN_CASE1:
            M_ED_Opt_case1 = M_ED_Opt_case1 + [sum(Dispatch[0][1].sum(axis=1) + p_BESS_value_RT[1, :] * dt_h)]
        #################################################################################
        # Save dispatch & baseline on a daily basis
        #################################################################################
        # TODO: do i need t0 dispatch and baseline DA here? this is just for next day initialization, and the results analysis is based on the Solver_Outputs df
        D_all = pd.DataFrame({'Interval start': pd.Series(TimeSeries_real[0]),
                              'V0G [kWh]': EnergyDemand_Step_V0G_TheDates[0],
                              'V1G_real [kWh]': EnergyDemand_Step_V1G_TheDates[0],
                              'Opt_DA [kWh]': p_GI_DA * dt_h,
                              'Opt_RT_t0 [kWh]': dispatch_t0[0] * dt_h}) # NOTE: V0G and V1G here are saved without BESS and DR/WM
        case_tables = []
        for i, case in enumerate(Cases):
            case_tables.append(
                pd.DataFrame({
                    # EV-only executed dispatch for baseline learning. Do not include BESS here;
                    # BESS/grid-meter effects are accounted in solver financial outputs.
                    f'{case} [kWh]': Dispatch[0][i].sum(axis=1),
                    f'{case}_withBESS [kWh]': Dispatch[0][i].sum(axis=1) + p_BESS_value_RT[i, :] * dt_h,
                    f'NCD_{case}': D_Th_NCD_96[i],
                    f'PD_{case}': D_Th_PD_96[i],
                    f'baseline_{case} [kWh]': Baseline_96[i].reshape(96),
                    f'event hour_{case}': EventHour_96[i].reshape(96)
                })
            )
        lmp_tables = [
            pd.DataFrame({'LMP_DA': Bid_Pr_DA}),
            pd.DataFrame({'LMP_RT': Bid_Pr_RT}),
        ]
        D_all = pd.concat([D_all, *case_tables, *lmp_tables], axis=1)
        dir_Output = os.path.join('Results/Dispatch/' + H_Start_RT.strftime("%Y") +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode  +'_implementation.csv')
        append_df_to_csv(D_all, dir_Output, index=False)
        #################################################################################
        # Save forecasted and real SessionkWh for validation
        #################################################################################
        validation_blocks = [
            ('Fc EVs', pd.DataFrame(Car_table_fc_fix[0][0].columns).T),
            ('Fc SessionkWh', SessionkWh_table_fc_fix[0][0].T.reset_index(drop=True).T),
        ]
        if Fc_AtArrival == 'MLatArrival':
            validation_blocks.append(('ML SessionkWh', pd.DataFrame(SessionkWh_RT_ML[0][0]).T))
        validation_blocks += [
            ('Real EVs', pd.DataFrame(pd.concat([Car_table_real[1][0], Car_table_real[2][0]], axis=1).columns).T),
            ('Real SessionkWh', pd.concat([SessionkWh_table_real[1][0], SessionkWh_table_real[2][0]], axis=1).T.reset_index(drop=True).T),
            ('Dispatched_base_withoutBESS', pd.DataFrame((Dispatch[0][0].sum(axis=0)).reset_index(drop=True)).T),
        ]
        if RUN_CASE1:
            validation_blocks += [
                ('Real Eta_min', pd.concat([Eta_min_table_real[1][1], Eta_min_table_real[2][1]], axis=1).T.reset_index(drop=True).T),
                ('Dispatched_Eta', Eta_table_RT[0][1].T.reset_index(drop=True).T),
                ('Dispatched_case1_withoutBESS', pd.DataFrame((Dispatch[0][1].sum(axis=0)).reset_index(drop=True)).T),
            ]
        List = [name for name, _ in validation_blocks]
        A = pd.concat([block for _, block in validation_blocks])
        A = pd.concat([A.reset_index(drop=True), pd.DataFrame(List, columns=[H_Start_RT.strftime("%Y%m%d")])], axis=1).set_index(H_Start_RT.strftime("%Y%m%d"))
        A = A.T
        A.index = [H_Start_RT.strftime("%Y%m%d")] * len(A) 
        A.index.name = "Date"
        dir_Output_summary = os.path.join('Results/Dispatch/' +\
                                    H_Start_RT.strftime("%Y") +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode +'_daily_summary.csv')
        _trim_existing_csv_before_run_start(dir_Output_summary, TheDate_Day0)
        append_df_to_csv(A, dir_Output_summary, index=True)
        #################################################################################
        # Save WM profit table and the reconciliation check in output folder
        #################################################################################
        # Save interval tables for Base case only
        if Enable_WM:
            p_DA_arr = np.array(Solver_Outputs_RT_Base['p_DA']).flatten() # 注意这里，DA处是 Solver_Outputs_DA，RT处是 Solver_Outputs_RT_Base
            p_RT_arr = np.array(Solver_Outputs_RT_Base['p_RT']).flatten()
            c_RU_DA_arr = np.array(Solver_Outputs_RT_Base['c_RU_DA']).flatten()
            c_RU_RT_arr = np.array(Solver_Outputs_RT_Base['c_RU_RT']).flatten()
            c_RD_DA_arr = np.array(Solver_Outputs_RT_Base['c_RD_DA']).flatten()
            c_RD_RT_arr = np.array(Solver_Outputs_RT_Base['c_RD_RT']).flatten()
            c_SP_DA_arr = np.array(Solver_Outputs_RT_Base['c_SP_DA']).flatten()
            c_SP_RT_arr = np.array(Solver_Outputs_RT_Base['c_SP_RT']).flatten()
            c_NSP_DA_arr = np.array(Solver_Outputs_RT_Base['c_NSP_DA']).flatten()
            c_NSP_RT_arr = np.array(Solver_Outputs_RT_Base['c_NSP_RT']).flatten()
            wm_profit_table = pd.DataFrame({
                'p_DA_profit': dt_h * bid_pr_da_i * p_DA_arr, 
                'p_RT_profit': dt_h * bid_pr_rt_i * p_RT_arr,
                # 向上调节 RU
                'c_RU_DA_energy': dt_h * (alpha_RU * bid_pr_rt_i) * c_RU_DA_arr,
                'c_RU_DA_capacity': dt_h * as_ru_da_i * c_RU_DA_arr,
                'c_RU_RT_energy': dt_h * (alpha_RU * bid_pr_rt_i) * c_RU_RT_arr,
                'c_RU_RT_capacity': dt_h * as_ru_rt_i * c_RU_RT_arr,
                # 向下调节 RD (注意公式中的负号)
                'c_RD_DA_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * c_RD_DA_arr,
                'c_RD_DA_capacity': dt_h * as_rd_da_i * c_RD_DA_arr,
                'c_RD_RT_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * c_RD_RT_arr,
                'c_RD_RT_capacity': dt_h * as_rd_rt_i * c_RD_RT_arr,
                # 旋转备用 SP
                'c_SP_DA_energy': dt_h * (alpha_SP * bid_pr_rt_i) * c_SP_DA_arr,
                'c_SP_DA_capacity': dt_h * as_sp_da_i * c_SP_DA_arr,
                'c_SP_RT_energy': dt_h * (alpha_SP * bid_pr_rt_i) * c_SP_RT_arr,
                'c_SP_RT_capacity': dt_h * as_sp_rt_i * c_SP_RT_arr,
                # 非旋转备用 NSP
                'c_NSP_DA_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * c_NSP_DA_arr,
                'c_NSP_DA_capacity': dt_h * as_nsp_da_i * c_NSP_DA_arr,
                'c_NSP_RT_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * c_NSP_RT_arr,
                'c_NSP_RT_capacity': dt_h * as_nsp_rt_i * c_NSP_RT_arr,
            })
            wm_tou_table = pd.DataFrame({
                'p_DA_TOU_Cost': -dt_h * c_e_TOU_AL * p_DA_arr,
                'p_RT_TOU_Cost': -dt_h * c_e_TOU_AL * p_RT_arr,
                'c_RU_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * c_RU_DA_arr,
                'c_RU_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * c_RU_RT_arr,
                'c_RD_DA_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * c_RD_DA_arr,
                'c_RD_RT_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * c_RD_RT_arr,
                'c_SP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * c_SP_DA_arr,
                'c_SP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * c_SP_RT_arr,
                'c_NSP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * c_NSP_DA_arr,
                'c_NSP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * c_NSP_RT_arr,
            })
        else:
            cols = [
                'p_DA_profit', 'p_RT_profit',
                'c_RU_DA_energy', 'c_RU_DA_capacity', 'c_RU_RT_energy', 'c_RU_RT_capacity',
                'c_RD_DA_energy', 'c_RD_DA_capacity', 'c_RD_RT_energy', 'c_RD_RT_capacity',
                'c_SP_DA_energy', 'c_SP_DA_capacity', 'c_SP_RT_energy', 'c_SP_RT_capacity',
                'c_NSP_DA_energy', 'c_NSP_DA_capacity', 'c_NSP_RT_energy', 'c_NSP_RT_capacity'
            ]
            wm_profit_table = pd.DataFrame(np.zeros((H, len(cols))), columns=cols)
            wm_tou_table = pd.DataFrame(np.zeros((H, 10)), columns=[
                'p_DA_TOU_Cost', 'p_RT_TOU_Cost',
                'c_RU_DA_TOU_Cost', 'c_RU_RT_TOU_Cost',
                'c_RD_DA_TOU_Cost', 'c_RD_RT_TOU_Cost',
                'c_SP_DA_TOU_Cost', 'c_SP_RT_TOU_Cost',
                'c_NSP_DA_TOU_Cost', 'c_NSP_RT_TOU_Cost'
            ])
        wm_profit_table['profit_sum_all_products'] = wm_profit_table.sum(axis=1)
        wm_tou_table['TOU_Cost_sum_all_products'] = wm_tou_table.sum(axis=1)
        wm_profit_total_from_table = float(wm_profit_table['profit_sum_all_products'].sum())
        wm_profit_check_diff = float(abs(wm_profit_total_from_table - np.sum(Solver_Outputs_RT_Base['Revenue_WM'])))
        Solver_Outputs_RT_Base['WM_Profit_Table'] = wm_profit_table
        Solver_Outputs_RT_Base['WM_Profit_Total_From_Table'] = wm_profit_total_from_table
        Solver_Outputs_RT_Base['WM_TOU_Table'] = wm_tou_table
        Solver_Outputs_RT_Base['WM_TOU_Total_From_Table'] = float(wm_tou_table['TOU_Cost_sum_all_products'].sum())
        if wm_profit_check_diff > 1e-3:
            print(f"DA WM profit check -> table_sum={wm_profit_total_from_table:.6f}, Revenue_WM={Solver_Outputs_RT_Base['Revenue_WM']:.6f}, abs_diff={wm_profit_check_diff:.6e}")
        # =========================================================================
        # Save daily WM profit table and the reconciliation check in output folder
        run_tag_daily = H_Start_RT.strftime("%Y") + '_' + Fc_SessionkWh + '_' + Fc_NumbEV + '_' + Fc_AtArrival + '_' + WM_Mode
        profit_dir = Path(f"Results/Plots/Solver_{TARGET_SAVE}_Choices") / run_tag_daily / pd.Timestamp(TheDate_Day0).strftime("%Y%m%d")
        profit_dir.mkdir(parents=True, exist_ok=True)
        if TARGET_SAVE == 'DA':
            Solver_Outputs_target = Solver_Outputs_DA
        else:
            Solver_Outputs_target = Solver_Outputs_RT_Base
        wm_profit_table_to_save = Solver_Outputs_target['WM_Profit_Table'].copy()
        wm_tou_table_to_save = Solver_Outputs_target['WM_TOU_Table'].copy()
        # =========================================================================
        # --- EV vs BESS 收益精确分配 (向上与向下严格解耦修正版) --- 
        # =========================================================================
        # 1. 动态获取当前表格的实际行数
        H_len = len(wm_profit_table_to_save)
        # 2. 提取物理出力，严格解耦放电(Up)与充电(Dn)，杜绝抵消漏洞
        p_up_bess = np.array(Solver_Outputs_target['p_dch_BESS_WM']).flatten()[:H_len]
        p_up_ev   = np.array(Solver_Outputs_target['p_dch_EV']).flatten()[:H_len]
        p_up_total = p_up_bess + p_up_ev
        p_dn_bess = np.array(Solver_Outputs_target['p_ch_BESS_WM']).flatten()[:H_len]
        p_dn_ev   = np.array(Solver_Outputs_target['p_ch_EV']).flatten()[:H_len]
        p_dn_total = p_dn_bess + p_dn_ev
        # 3. 动态容量边界（已由你在上一步正确计算）
        dynamic_P_BESS_max = np.full(H_len, P_BESS_max) 
        dynamic_B_EV = np.array(Solver_Outputs_target['Baseline(kW)']).flatten()[:H_len]
        dynamic_P_EV_max = np.array(Solver_Outputs_target['P_EV_max']).flatten()[:H_len]
        dynamic_up_max = dynamic_P_BESS_max + dynamic_B_EV
        dynamic_down_max = dynamic_P_BESS_max + dynamic_P_EV_max - dynamic_B_EV
        # =========================================================================
        # 4. 计算分配权重 (强制守恒修复版)
        # =========================================================================
        # 4b. 容量分配权重 (Capacity Weights) - 必须先算，作为保底
        w_cap_up_bess = np.divide(dynamic_P_BESS_max, dynamic_up_max, out=np.zeros_like(dynamic_up_max), where=dynamic_up_max>1e-6)
        w_cap_up_ev   = 1.0 - w_cap_up_bess
        w_cap_dn_bess = np.divide(dynamic_P_BESS_max, dynamic_down_max, out=np.zeros_like(dynamic_down_max), where=dynamic_down_max>1e-6)
        w_cap_dn_ev   = 1.0 - w_cap_dn_bess
        # =========================================================================
        # 4a. 能量权重：按代数净功率流占比 (支持对冲工况，守恒且无Warning)
        # =========================================================================
        # 分别计算 BESS 和 EV 的代数净功率流（放电为正，充电为负）
        p_net_bess = np.array(Solver_Outputs_target['p_dch_BESS_WM']).flatten()[:H_len] - np.array(Solver_Outputs_target['p_ch_BESS_WM']).flatten()[:H_len]
        p_net_ev   = np.array(Solver_Outputs_target['p_dch_EV']).flatten()[:H_len] - np.array(Solver_Outputs_target['p_ch_EV']).flatten()[:H_len]
        # 系统对外的总净功率流（等于两者代数相加，完美对应 p_dch_net - p_ch_net）
        p_net_system = p_net_bess + p_net_ev
        # 先计算原始的代数权重（允许破1或负数，用 where 规避系统不充不放时的除零错误）
        w_raw = np.divide(p_net_bess, p_net_system, out=w_cap_up_bess.copy(), where=np.abs(p_net_system) > 1e-3)
        # 核心：用你说的 min-max 逻辑将 BESS 强行锁死在 [0, 1] 之间
        w_energy_bess = np.clip(w_raw, 0.0, 1.0)
        w_energy_ev   = 1.0 - w_energy_bess
        # =========================================================================
        # 5. 执行拆分：将收益按【功能维度】与【物理方向】细分
        # =========================================================================
        cap_up_profit = (
            wm_profit_table_to_save['c_RU_DA_capacity'] + wm_profit_table_to_save['c_RU_RT_capacity'] + 
            wm_profit_table_to_save['c_SP_DA_capacity'] + wm_profit_table_to_save['c_SP_RT_capacity'] + 
            wm_profit_table_to_save['c_NSP_DA_capacity'] + wm_profit_table_to_save['c_NSP_RT_capacity']
        ).values
        cap_dn_profit = (
            wm_profit_table_to_save['c_RD_DA_capacity'] + wm_profit_table_to_save['c_RD_RT_capacity']
        ).values
        total_energy_profit = (
            wm_profit_table_to_save['p_DA_profit'] + wm_profit_table_to_save['p_RT_profit'] +
            wm_profit_table_to_save['c_RU_DA_energy'] + wm_profit_table_to_save['c_RU_RT_energy'] +
            wm_profit_table_to_save['c_SP_DA_energy'] + wm_profit_table_to_save['c_SP_RT_energy'] +
            wm_profit_table_to_save['c_NSP_DA_energy'] + wm_profit_table_to_save['c_NSP_RT_energy'] +
            wm_profit_table_to_save['c_RD_DA_energy'] + wm_profit_table_to_save['c_RD_RT_energy']
        ).values
        # 汇总所有容量收益
        total_cap_profit = (
            wm_profit_table_to_save['c_RU_DA_capacity'] + wm_profit_table_to_save['c_RU_RT_capacity'] +
            wm_profit_table_to_save['c_SP_DA_capacity'] + wm_profit_table_to_save['c_SP_RT_capacity'] +
            wm_profit_table_to_save['c_NSP_DA_capacity'] + wm_profit_table_to_save['c_NSP_RT_capacity'] +
            wm_profit_table_to_save['c_RD_DA_capacity'] + wm_profit_table_to_save['c_RD_RT_capacity']
        ).values
        # =========================================================================
        # 6. 分配并计算最终结果
        # =========================================================================
        wm_profit_table_to_save['BESS_Energy_Profit'] = total_energy_profit * w_energy_bess
        wm_profit_table_to_save['EV_Energy_Profit']   = total_energy_profit * w_energy_ev
        wm_profit_table_to_save['BESS_Capacity_Profit'] = (cap_up_profit * w_cap_up_bess) + (cap_dn_profit * w_cap_dn_bess)
        wm_profit_table_to_save['EV_Capacity_Profit']   = (cap_up_profit * (1.0 - w_cap_up_bess)) + (cap_dn_profit * (1.0 - w_cap_dn_bess))
        wm_profit_table_to_save['BESS_Total_WM_Profit'] = wm_profit_table_to_save['BESS_Energy_Profit'] + wm_profit_table_to_save['BESS_Capacity_Profit']
        wm_profit_table_to_save['EV_Total_WM_Profit']   = wm_profit_table_to_save['EV_Energy_Profit'] + wm_profit_table_to_save['EV_Capacity_Profit']
        # 7. 最后再保存到 CSV
        wm_profit_check_separation_diff = float(abs(wm_profit_table_to_save['BESS_Total_WM_Profit'].sum() + wm_profit_table_to_save['EV_Total_WM_Profit'].sum() - wm_profit_total_from_table))
        if wm_profit_check_separation_diff > 1e-3:
            print(f"WM profit separation check -> separated_sum={wm_profit_table_to_save['BESS_Total_WM_Profit'].sum() + wm_profit_table_to_save['EV_Total_WM_Profit'].sum():.6f}, original_total={wm_profit_total_from_table:.6f}, abs_diff={wm_profit_check_separation_diff:.6e}")
        wm_profit_table_to_save.insert(0, 'Interval start', pd.date_range(start=pd.Timestamp(TheDate_Day0), periods=len(wm_profit_table_to_save), freq=f"{max(1, int(round(dt_h * 60)))}min"))
        wm_profit_file = profit_dir / f"{TARGET_SAVE}_WM_profit_breakdown_{WM_Mode}.csv"
        wm_profit_table_to_save.to_csv(wm_profit_file, index=False, float_format='%.2f')
        wm_tou_table_to_save.insert(0, 'Interval start', pd.date_range(start=pd.Timestamp(TheDate_Day0), periods=len(wm_tou_table_to_save), freq=f"{max(1, int(round(dt_h * 60)))}min"))
        wm_tou_file = profit_dir / f"{TARGET_SAVE}_WM_TOU_breakdown_{WM_Mode}.csv"
        wm_tou_table_to_save.to_csv(wm_tou_file, index=False, float_format='%.2f')
        _save_daily_6panel = _date_key_allowed(TheDate_Day0, globals().get('PLOT_DAILY_6PANEL_DATES', []))
        if _save_daily_6panel:
            daily_fig_files = plot_daily_solver_choice_figures(
                TheDate_Day0=TheDate_Day0,
                dt_h=dt_h,
                Solver_Outputs=Solver_Outputs_target,
                c_e_TOU_AL=c_e_TOU_AL,
                Bid_Pr_DA=Bid_Pr_DA,
                Bid_Pr_RT=Bid_Pr_RT,
                AS_Pr_RU_DA=AS_Pr_RU_DA,
                AS_Pr_RU_RT=AS_Pr_RU_RT,
                AS_Pr_RD_DA=AS_Pr_RD_DA,
                AS_Pr_RD_RT=AS_Pr_RD_RT,
                AS_Pr_SP_DA=AS_Pr_SP_DA,
                AS_Pr_SP_RT=AS_Pr_SP_RT,
                AS_Pr_NSP_DA=AS_Pr_NSP_DA,
                AS_Pr_NSP_RT=AS_Pr_NSP_RT,
                alpha_RU=alpha_RU,
                alpha_RD=alpha_RD,
                alpha_SP=alpha_SP,
                alpha_NSP=alpha_NSP,
                run_tag=run_tag_daily,
                M_Th_NCD = M_Th_NCD_prev,  # use previous day's threshold for fig(f)
                M_Th_PD = M_Th_PD_prev,
                P_BESS_max = P_BESS_max,
                P_EV_max = dynamic_P_EV_max,
                mpc_version_label = MPC_VERSION_LABEL,
                forecast_label = ('Perfect forecast' if Fc_SessionkWh == 'PerfectSessionkWh' else 'Persistence forecast'),
                threshold_case_idx = (DA_CASE_IDX if TARGET_SAVE == 'DA' else 0),
            )
        else:
            daily_fig_files = []
            print(f"Skipping 6-panel figure for {pd.Timestamp(TheDate_Day0).strftime('%Y%m%d')} (PLOT_DAILY_6PANEL_DATES={globals().get('PLOT_DAILY_6PANEL_DATES', [])})")
        '''
        cycle_den_rt = 2.0 * float(C_BESS) * float(SOC_BESS_max - SOC_BESS_min)
        cycle_base_rt = dt_h * float(np.sum(p_ch_BESS_value_RT[0, :] + p_dch_BESS_value_RT[0, :])) / cycle_den_rt if cycle_den_rt > 0 else np.nan
        cycle_case1_rt = dt_h * float(np.sum(p_ch_BESS_value_RT[1, :] + p_dch_BESS_value_RT[1, :])) / cycle_den_rt if cycle_den_rt > 0 else np.nan
        cycle_label = f"Base {cycle_base_rt:.3f}, Case1 {cycle_case1_rt:.3f}"
        stairplot_file = save_old_stairplot_beautified(
            H_Start_RT=H_Start_RT,
            TimeSeries_real=TimeSeries_real,
            dt_m_EV=dt_m_EV,
            dt_h=dt_h,
            EnergyDemand_Step_V0G_TheDates=EnergyDemand_Step_V0G_TheDates,
            EnergyDemand_Step_V1G_TheDates=EnergyDemand_Step_V1G_TheDates,
            p_GI_DA=p_GI_DA,
            dispatch_t0=dispatch_t0,
            Dispatch=Dispatch,
            p_BESS_value_RT=p_BESS_value_RT,
            Baseline_96=Baseline_96,
            EventHour_96=EventHour_96,
            D_Th_NCD_96=D_Th_NCD_96,
            D_Th_PD_96=D_Th_PD_96,
            Fc_SessionkWh=Fc_SessionkWh,
            Fc_NumbEV=Fc_NumbEV,
            Fc_AtArrival=Fc_AtArrival,
            WM_Mode=WM_Mode,
            cycle_label=cycle_label,
        )
        print(' -', stairplot_file)
        '''
    # end of daily loop
# end of monthly loop
end_time = time.time()
print("Main loop runtime: {:.2f} seconds".format(end_time - start_time))
PLOT_DAILY_6PANEL_DATES = 'all'  # []: skip daily 6-panel; None/'all': all days; or use [1, 2] / ['20250701']
ANALYSIS_DAYS_CONFIG = None  # None uses RUN_DAYS_CONFIG for final financial plots
SAVE_FINAL_FINANCIAL_FIGURES = False  # main notebooks save CSVs; paper figures are made in paper_financial_comparison_plots.ipynb
def _date_key_allowed(date_like, selected_dates):
    """Return whether a daily expensive plot should be saved for date_like."""
    if selected_dates is None or selected_dates == 'all':
        return True
    if selected_dates == [] or selected_dates == () or selected_dates == set():
        return False
    target = pd.Timestamp(date_like).strftime('%Y%m%d')
    allowed = set()
    for item in selected_dates:
        s = str(item)
        if s.isdigit() and len(s) <= 2:
            allowed.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(s):02d}")
        else:
            allowed.add(pd.Timestamp(item).strftime('%Y%m%d'))
    return target in allowed


RUN_MAIN_LOOP_DIRECT=False: main loop block loaded without execution.


Main loop runtime: 0.01 seconds


In [6]:
# Run July-August 2x2 financial scenarios from the main notebook.
# Scenario grid per notebook: Perfect/Persistence forecast x full/retail_only market mode.
# Shrinking and rolling are run from their own notebooks.
import nbformat
import re
from pathlib import Path
# ===== User controls =====
TARGET_SAVE = 'RT'
RUN_MONTHS_CONFIG = [6]
# Clear old implementation files only when starting a fresh simulation period.
# For a standalone July run, keep the June implementation CSV so baseline learning
# continues from executed EV dispatch history.
CLEAR_IMPLEMENTATION_AT_RUN_START = min(RUN_MONTHS_CONFIG) <= 6
RUN_DAYS_CONFIG = list(range(1, 31))
RUN_MODES = ['full', 'retail_only']
RUN_FORECAST_CONFIGS = [
    ('PerfectSessionkWh', 'PerfectNumbEV', 'PerfectatArrival'),
    ('PersistenceSessionkWh', 'PersistenceNumbEV', 'PerfectatArrival'),
]
PLOT_DAILY_6PANEL_DATES = 'all'
ANALYSIS_DAYS_CONFIG = []  # [] means summarize all generated dates, including both July and August
# =========================
# Preflight guard: only run months covered by all required market data files.
# This keeps RUN_MONTHS_CONFIG=[7, 8] future-proof while avoiding empty-price CVXPY vectors.
def _market_file_has_month(path, month):
    import csv
    tag = f'2025-{int(month):02d}'
    path = Path(path)
    if not path.exists():
        return False
    with path.open(newline='') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row and str(row[0]).startswith(tag):
                return True
    return False
_required_market_files = [
    '2025Data/LMP/2025/LMP_DA_2025_clean.csv',
    '2025Data/LMP/2025/LMP_FM_2025_clean.csv',
    '2025Data/AS_DAM/AS_price_2025_clear.csv',
    '2025Data/AS_RTM/AS_price_2025_clear.csv',
]
_requested_months = list(RUN_MONTHS_CONFIG)
_available_months = []
_skipped_months = []
for _month in _requested_months:
    _missing = [str(_p) for _p in _required_market_files if not _market_file_has_month(_p, _month)]
    if _missing:
        _skipped_months.append((_month, _missing))
    else:
        _available_months.append(_month)
if _skipped_months:
    for _month, _missing in _skipped_months:
        print(f'Skipping 2025-{int(_month):02d}: missing market-data coverage in {_missing}')
RUN_MONTHS_CONFIG = _available_months
if not RUN_MONTHS_CONFIG:
    raise ValueError('No requested month has complete DA/RT LMP and AS market-data coverage.')
print(f'Validated run months: {RUN_MONTHS_CONFIG}')
notebook_path = Path('upscaledev_imp.ipynb')
nb = nbformat.read(notebook_path, as_version=4)
main_loop_source = None
financial_table_source = None
financial_summary_source = None
for cell in nb.cells:
    if cell.get('cell_type') != 'code':
        continue
    src = cell.get('source', '')
    src_text = ''.join(src) if isinstance(src, list) else str(src)
    is_runner_cell = 'Run July-August 2x2 financial scenarios from the main notebook' in src_text
    if (not is_runner_cell) and ('Main loop runtime' in src_text) and ('RUN_MAIN_LOOP_DIRECT' in src_text):
        main_loop_source = src_text
    if (not is_runner_cell) and ('Save WM profit breakdown summary CSV' in src_text):
        financial_table_source = src_text
    if (not is_runner_cell) and ('Run-Period Financial Summary CSVs Only' in src_text):
        financial_summary_source = src_text
if main_loop_source is None:
    raise ValueError('Could not locate the main loop code cell in the notebook.')
if financial_table_source is None:
    raise ValueError('Could not locate the daily financial table code cell in the notebook.')
if financial_summary_source is None:
    raise ValueError('Could not locate the financial summary code cell in the notebook.')
# CRITICAL: run the main-loop source when executed from this runner cell.
main_loop_source = re.sub(
    r"RUN_MAIN_LOOP_DIRECT\s*=\s*False",
    "RUN_MAIN_LOOP_DIRECT = True",
    main_loop_source,
)
for _fc_session, _fc_numbev, _fc_arrival in RUN_FORECAST_CONFIGS:
    Fc_SessionkWh = _fc_session
    Fc_NumbEV = _fc_numbev
    Fc_AtArrival = _fc_arrival
    print(
        f"\n=== Forecast scenario: {Fc_SessionkWh} | {Fc_NumbEV} | {Fc_AtArrival} "
        f"| months={RUN_MONTHS_CONFIG} | days={RUN_DAYS_CONFIG} ==="
    )
    for WM_Mode in RUN_MODES:
        run_days = RUN_DAYS_CONFIG
        Enable_WM = (WM_Mode in ['full'])
        print(
            f"\n=== Running WM_Mode: {WM_Mode} | Enable_WM={Enable_WM} "
            f"| months={RUN_MONTHS_CONFIG} | run_days={RUN_DAYS_CONFIG} | target_save={TARGET_SAVE} ==="
        )
        RT_SOLVER_STATUS_COUNTS = {}
        RT_SOLVER_LIMIT_EVENTS = []
        exec(main_loop_source, globals(), globals())
        _audit_counts = dict(RT_SOLVER_STATUS_COUNTS)
        _audit_events = list(RT_SOLVER_LIMIT_EVENTS)
        if _audit_events:
            print(f"RT TimeLimit audit: TRIGGERED {len(_audit_events)} time(s); status_counts={_audit_counts}")
            for _event in _audit_events[:10]:
                print('  ', _event)
            if len(_audit_events) > 10:
                print(f"  ... {len(_audit_events) - 10} additional TimeLimit event(s)")
        else:
            print(f"RT TimeLimit audit: PASS (not triggered); status_counts={_audit_counts}")
    # Build daily financial tables and summary for this forecast after both modes finish.
    ANALYSIS_DAYS_CONFIG = [f'{year}{int(_m):02d}{int(_d):02d}' for _m in RUN_MONTHS_CONFIG for _d in RUN_DAYS_CONFIG if 1 <= int(_d) <= calendar.monthrange(year, int(_m))[1]]
    exec(financial_table_source, globals(), globals())
    exec(financial_summary_source, globals(), globals())
print('\nAll 2x2 scenarios completed for this notebook.')


Validated run months: [6]

=== Forecast scenario: PerfectSessionkWh | PerfectNumbEV | PerfectatArrival | months=[6] | days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30] ===

=== Running WM_Mode: full | Enable_WM=True | months=[6] | run_days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30] | target_save=RT ===
All dispatch files from last implementation have been deleted.

Test month: 6
Month 6: M_Th_NCD init=0.0 kW, M_Th_PD init=0.0 kW (initialized to 0)


Processing days:   0%|          | 0/30 [00:00<?, ?it/s]

Set parameter Username


Academic license - for non-commercial use only - expires 2027-03-16


Processing days:   3%|▎         | 1/30 [00:08<04:02,  8.35s/it]

...

<string>:1296: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:1297: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:1298: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


Processing days: 100%|██████████| 30/30 [04:43<00:00,  8.68s/it]

Processing days: 100%|██████████| 30/30 [04:43<00:00,  9.46s/it]

Main loop runtime: 283.90 seconds


Saved 274 files across 61 days.
Financial summary restricted to run dates: ['20250601', '20250602', '20250603', '20250604', '20250605', '20250606', '20250607', '20250608', '20250609', '20250610', '20250611', '20250612', '20250613', '20250614', '20250615', '20250616', '20250617', '20250618', '20250619', '20250620', '20250621', '20250622', '20250623', '20250624', '20250625', '20250626', '20250627', '20250628', '20250629', '20250630']

  Run-Period Financial Summary  (US$)
             Total Revenue  WM Revenue  TOU Cost  PD Cost  NCD Cost  EV Revenue
Case / US$                                                                     
Retail only        5342.25        0.00 -11625.73  -492.94  -5932.99    23393.92
Both              10881.99     4273.53 -11498.63  -508.40  -4778.45    23393.92
Daily detail CSV saved to: Results/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv
Summary CSV saved to: Results/Plots/Cost/2025_PersistenceSessionkWh_Pe

In [7]:
# Save WM profit breakdown summary CSV for each day (full / retail_only)
# And build a 2x5 daily table without rerunning optimization:
# rows: Retail only, Both
# cols: Total Cost, WM Revenue, TOU Cost, Demand Cost (PD+NCD), EV Revenue
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
base_dir = Path(f'Results/Plots/Solver_{TARGET_SAVE}_Choices')
financial_dir = base_dir / f'{TARGET_SAVE}_financial_tables'
financial_dir.mkdir(parents=True, exist_ok=True)
dispatch_dir = Path('Results/Dispatch')
if 'RUN_MODES' not in globals():
    RUN_MODES = ['full', 'retail_only']
modes = [m for m in ['full', 'retail_only'] if m in RUN_MODES]
mode_label = {
    'retail_only': 'Retail only',
    'full': 'Both',
}
# Collect daily WM & TOU breakdown files by mode: {mode: {YYYYMMDD: file_path}}
daily_wm_files = {m: {} for m in modes}
for mode in modes:
    scenario_dir = base_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}'
    for fp in sorted(scenario_dir.glob(f'*/{TARGET_SAVE}_WM_profit_breakdown_{mode}.csv')):
        day_key = fp.parent.name
        daily_wm_files[mode][day_key] = fp
daily_tou_files = {m: {} for m in modes}
for mode in modes:
    scenario_dir = base_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}'
    for fp in sorted(scenario_dir.glob(f'*/{TARGET_SAVE}_WM_TOU_breakdown_{mode}.csv')):
        day_key = fp.parent.name
        daily_tou_files[mode][day_key] = fp
# Collect implementation and daily summary files by mode
impl_file_by_mode = {}
daily_summary_file_by_mode = {}
for mode in modes:
    impl_file = dispatch_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}_implementation.csv'
    if impl_file.exists():
        impl_file_by_mode[mode] = impl_file
    summary_file = dispatch_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}_daily_summary.csv'
    if summary_file.exists():
        daily_summary_file_by_mode[mode] = summary_file
# Summarize only the modes selected for the current test/run.
common_days = None
for mode in modes:
    mode_days = set(daily_wm_files[mode]).intersection(daily_tou_files[mode])
    common_days = mode_days if common_days is None else common_days.intersection(mode_days)
if not common_days:
    raise FileNotFoundError(f'No complete daily set found for active modes: {modes}.')
# Cache dispatch csvs once
impl_cache = {m: pd.read_csv(p) for m, p in impl_file_by_mode.items()}
for m in impl_cache:
    impl_cache[m]['Interval start'] = pd.to_datetime(impl_cache[m]['Interval start'])
summary_cache = {m: pd.read_csv(p) for m, p in daily_summary_file_by_mode.items()}
# Prefer notebook rates if available; otherwise reconstruct from current season settings
if 'c_e_TOU_AL' in globals() and isinstance(c_e_TOU_AL, np.ndarray) and len(c_e_TOU_AL) == 96:
    tou_rates_96 = np.array(c_e_TOU_AL, dtype=float)
else:
    # Fallback reconstruction using notebook tariff logic
    season_local = globals().get('season', 'Summer')
    if season_local == 'Summer':
        c_delivery_sofnof_local = 0.03921
        c_delivery_on_local = 0.335
        c_energy_on_local = c_delivery_on_local + 0.490
        c_energy_off_local = c_delivery_sofnof_local + 0.175
        c_energy_superoff_local = c_delivery_sofnof_local + 0.0815
    else:
        c_delivery_sofnof_local = 0.03978
        c_delivery_on_local = 0.384
        c_energy_on_local = c_delivery_on_local + 0.169
        c_energy_off_local = c_delivery_sofnof_local + 0.095
        c_energy_superoff_local = c_delivery_sofnof_local + 0.073
    onpeak_local = list(range(16 * 4, 21 * 4))
    offpeak_local = list(range(6 * 4, 16 * 4)) + list(range(21 * 4, 24 * 4))
    superoffpeak_local = list(range(0, 6 * 4))
    tou_rates_96 = np.ones(96)
    tou_rates_96[onpeak_local] = c_energy_on_local
    tou_rates_96[offpeak_local] = c_energy_off_local
    tou_rates_96[superoffpeak_local] = c_energy_superoff_local
c_ncd_local = float(globals().get('c_NCD', 15.38))
c_pd_local = float(globals().get('c_PD', 3.05))
ev_service_rate = 0.4
saved_paths = []
max_p_ncd_so_far = {m: 0.0 for m in modes}
max_p_pd_so_far = {m: 0.0 for m in modes}
current_billing_month = None
for day_key in sorted(common_days):
    day_ts_for_reset = pd.to_datetime(day_key, format='%Y%m%d')
    billing_month = day_ts_for_reset.to_period('M')
    if current_billing_month != billing_month:
        max_p_ncd_so_far = {m: 0.0 for m in modes}
        max_p_pd_so_far = {m: 0.0 for m in modes}
        current_billing_month = billing_month
    # 1) keep existing product breakdown summary
    profit_rows = []
    for mode in modes:
        mode_df = pd.read_csv(daily_wm_files[mode][day_key])
        breakdown_cols = [col for col in mode_df.columns if col not in {'Interval start', 'profit_sum_all_products'}]
        profit_rows.append(mode_df[breakdown_cols].sum(numeric_only=True))
    profit_sum_table = pd.DataFrame(profit_rows, index=modes)
    profit_sum_table.columns = [col.replace('_profit', '') for col in profit_sum_table.columns]
    profit_sum_table = profit_sum_table.round(2)
    profit_sum_table['wm_total'] = profit_sum_table.sum(axis=1).round(2)
    # save tou breakdown summary as well
    tou_rows = []
    for mode in modes:
        mode_tou_df = pd.read_csv(daily_tou_files[mode][day_key])
        breakdown_cols = [col for col in mode_tou_df.columns if col not in {'Interval start', 'cost_sum_all_products'}]
        tou_rows.append(mode_tou_df[breakdown_cols].sum(numeric_only=True))
    tou_sum_table = pd.DataFrame(tou_rows, index=modes)
    tou_sum_table.columns = [col.replace('_cost', '') for col in tou_sum_table.columns]
    tou_sum_table = tou_sum_table.round(2)
    tou_sum_table['tou_total'] = tou_sum_table.sum(axis=1).round(2)
    for mode in modes:
        out_dir = daily_wm_files[mode][day_key].parent
        out_file = out_dir / f'{TARGET_SAVE}_WM_profit_breakdown_summary_{day_key}.csv'
        profit_sum_table.to_csv(out_file, float_format='%.2f')
        saved_paths.append(out_file)
        out_file = out_dir / f'{TARGET_SAVE}_WM_TOU_breakdown_summary_{day_key}.csv'
        tou_sum_table.to_csv(out_file, float_format='%.2f')
        saved_paths.append(out_file)
    # 2) build daily 2x5 financial table
    #    Sign convention:
    #    - WM Revenue, EV Revenue are positive inflows
    #    - TOU Cost, Demand Cost are shown negative (cost outflows)
    #    - Total Revenue = WM Revenue + TOU Cost + Demand Cost + EV Revenue
    day_rows = []
    for mode in modes:
        if mode not in impl_cache:
            continue
        if mode not in summary_cache:
            continue
        day_ts = pd.to_datetime(day_key, format='%Y%m%d')
        day_start = day_ts
        day_end = day_ts + pd.Timedelta(days=1)
        day_impl = impl_cache[mode][
            (impl_cache[mode]['Interval start'] >= day_start)
            & (impl_cache[mode]['Interval start'] < day_end)
        ].copy()
        if day_impl.empty:
            continue
        # ====================================================================
        # 提取 WM revenue 以及 BESS/EV 的明细拆分
        # ====================================================================
        wm_df = pd.read_csv(daily_wm_files[mode][day_key])
        wm_revenue = float(wm_df['profit_sum_all_products'].sum())
        # 安全提取 6 个新指标 (加 if 判断是防止 Retail_only 空表时报错)
        wm_bess_energy = float(wm_df['BESS_Energy_Profit'].sum()) if 'BESS_Energy_Profit' in wm_df.columns else 0.0
        wm_ev_energy   = float(wm_df['EV_Energy_Profit'].sum()) if 'EV_Energy_Profit' in wm_df.columns else 0.0
        wm_bess_cap    = float(wm_df['BESS_Capacity_Profit'].sum()) if 'BESS_Capacity_Profit' in wm_df.columns else 0.0
        wm_ev_cap      = float(wm_df['EV_Capacity_Profit'].sum()) if 'EV_Capacity_Profit' in wm_df.columns else 0.0
        wm_bess_total  = float(wm_df['BESS_Total_WM_Profit'].sum()) if 'BESS_Total_WM_Profit' in wm_df.columns else 0.0
        wm_ev_total    = float(wm_df['EV_Total_WM_Profit'].sum()) if 'EV_Total_WM_Profit' in wm_df.columns else 0.0
        # Reconcile interval-CSV rounding so the four WM components sum exactly
        # to the settled WM revenue used in the financial identity.
        wm_ev_energy = wm_revenue - wm_bess_energy - wm_bess_cap - wm_ev_cap
        wm_bess_total = wm_bess_energy + wm_bess_cap
        wm_ev_total = wm_ev_energy + wm_ev_cap
        # TOU cost based on implementation net energy column (Base [kWh])
        tou_energy_col = 'Base_withBESS [kWh]' if 'Base_withBESS [kWh]' in day_impl.columns else 'Base [kWh]'
        if tou_energy_col not in day_impl.columns:
            raise KeyError(f"{tou_energy_col} column missing in implementation file for mode={mode}.")
        base_kwh = pd.to_numeric(day_impl[tou_energy_col], errors='coerce').fillna(0.0).to_numpy()
        if len(base_kwh) != 96:
            # Handle any unexpected row count robustly
            n = min(len(base_kwh), len(tou_rates_96))
            tou_cost_abs = float(np.sum(base_kwh[:n] * tou_rates_96[:n]))
        else:
            tou_cost_abs = float(np.sum(base_kwh * tou_rates_96))
        tou_cost = -tou_cost_abs
        # NWM energy attribution is exactly additive: EV service revenue and
        # EV/BESS shares of the realized meter TOU settlement.
        ev_profile_col = f'{Cases[0]} [kWh]' if f'{Cases[0]} [kWh]' in day_impl.columns else 'Base [kWh]'
        ev_kwh_profile_all = pd.to_numeric(day_impl[ev_profile_col], errors='coerce').fillna(0.0).to_numpy()
        n_profile = min(len(base_kwh), len(ev_kwh_profile_all), len(tou_rates_96))
        grid_kwh_profile = np.asarray(base_kwh[:n_profile], dtype=float)
        ev_kwh_profile = np.asarray(ev_kwh_profile_all[:n_profile], dtype=float)
        bess_kwh_profile = grid_kwh_profile - ev_kwh_profile
        tou_rate_profile = np.asarray(tou_rates_96[:n_profile], dtype=float)
        nwm_ev_tou = -float(np.sum(ev_kwh_profile * tou_rate_profile))
        nwm_bess_energy = -float(np.sum(bess_kwh_profile * tou_rate_profile))
        # Demand charges from daily max thresholds
        ncd_col = 'NCD_Base' if 'NCD_Base' in day_impl.columns else None
        pd_col = 'PD_Base' if 'PD_Base' in day_impl.columns else None
        if ncd_col is None or pd_col is None:
            raise KeyError(f"NCD_Base/PD_Base missing in implementation file for mode={mode}.")
        ncd_peak_kw = float(pd.to_numeric(day_impl[ncd_col], errors='coerce').fillna(0.0).max())
        pd_peak_kw = float(pd.to_numeric(day_impl[pd_col], errors='coerce').fillna(0.0).max())
        ncd_peak_diff = max(ncd_peak_kw - max_p_ncd_so_far[mode], 0)
        pd_peak_diff = max(pd_peak_kw - max_p_pd_so_far[mode], 0)
        num_days_month = calendar.monthrange(day_ts.year, day_ts.month)[1]
        pd_cost_abs = float(c_pd_local * pd_peak_diff)
        ncd_cost_abs = float(c_ncd_local * ncd_peak_diff)
        pd_cost = -pd_cost_abs
        ncd_cost = -ncd_cost_abs
        demand_cost_abs = float((c_ncd_local * ncd_peak_diff + c_pd_local * pd_peak_diff))
        demand_cost = -demand_cost_abs
        # Attribute each incremental demand charge at its binding realized peak.
        grid_kw_profile = grid_kwh_profile / dt_h
        ev_kw_profile = ev_kwh_profile / dt_h
        interval_profile = pd.to_datetime(day_impl['Interval start']).iloc[:n_profile]
        def _split_incremental_demand_cost(cost_value, target_peak_kw, candidate_mask):
            if abs(cost_value) <= 1e-12 or n_profile == 0:
                return 0.0, 0.0
            candidates = np.flatnonzero(np.asarray(candidate_mask, dtype=bool))
            if len(candidates) == 0:
                candidates = np.arange(n_profile)
            idx = int(candidates[np.argmin(np.abs(grid_kw_profile[candidates] - target_peak_kw))])
            denom = float(grid_kw_profile[idx])
            if abs(denom) <= 1e-9:
                return float(cost_value), 0.0
            ev_part = float(cost_value) * float(ev_kw_profile[idx]) / denom
            return ev_part, float(cost_value) - ev_part
        all_mask = np.ones(n_profile, dtype=bool)
        pd_mask_profile = ((interval_profile.dt.hour >= 16) & (interval_profile.dt.hour < 21)).to_numpy()
        ncd_ev_cap, ncd_bess_cap = _split_incremental_demand_cost(ncd_cost, ncd_peak_kw, all_mask)
        pd_ev_cap, pd_bess_cap = _split_incremental_demand_cost(pd_cost, pd_peak_kw, pd_mask_profile)
        nwm_ev_capacity = ncd_ev_cap + pd_ev_cap
        nwm_bess_capacity = ncd_bess_cap + pd_bess_cap
        max_p_ncd_so_far[mode] = max(max_p_ncd_so_far[mode], ncd_peak_kw)
        max_p_pd_so_far[mode]  = max(max_p_pd_so_far[mode], pd_peak_kw)
        # EV revenue from physical EV-only executed energy. Use implementation CSV,
        # not daily_summary validation rows, because daily_summary is row-per-session and
        # can contain preserved historical rows across reruns.
        ev_energy_col = f'{Cases[0]} [kWh]' if f'{Cases[0]} [kWh]' in day_impl.columns else 'Base [kWh]'
        if ev_energy_col not in day_impl.columns:
            raise KeyError(f"{ev_energy_col} column missing in implementation file for mode={mode}.")
        ev_energy_kwh = float(pd.to_numeric(day_impl[ev_energy_col], errors='coerce').fillna(0.0).sum())
        ev_revenue = ev_service_rate * ev_energy_kwh
        nwm_ev_energy = ev_revenue + nwm_ev_tou
        total_revenue = wm_revenue + tou_cost + demand_cost + ev_revenue
        day_rows.append({
            'Case / US$': mode_label[mode],
            'Total Revenue': round(total_revenue, 2),
            'WM Revenue': round(wm_revenue, 2),
            # === 新增保存这 6 列拆分数据 ===
            'WM BESS Energy': round(wm_bess_energy, 2),
            'WM EV Energy': round(wm_ev_energy, 2),
            'WM BESS Capacity': round(wm_bess_cap, 2),
            'WM EV Capacity': round(wm_ev_cap, 2),
            'WM BESS Total': round(wm_bess_total, 2),
            'WM EV Total': round(wm_ev_total, 2),
            'NWM BESS Energy': round(nwm_bess_energy, 2),
            'NWM EV Energy': round(nwm_ev_energy, 2),
            'NWM BESS Capacity': round(nwm_bess_capacity, 2),
            'NWM EV Capacity': round(nwm_ev_capacity, 2),
            # ==============================
            'TOU Cost': round(tou_cost, 2),
            'PD Cost': round(pd_cost, 2),
            'NCD Cost': round(ncd_cost, 2),
            'EV Revenue': round(ev_revenue, 2),
        })
    if len(day_rows) > 0:
        order = [mode_label[m] for m in ['retail_only', 'full'] if m in modes]
        table_2x5 = pd.DataFrame(day_rows)
        table_2x5['Case / US$'] = pd.Categorical(table_2x5['Case / US$'], categories=order, ordered=True)
        table_2x5 = table_2x5.sort_values('Case / US$').reset_index(drop=True)
        # Save 2x5 table into dedicated folder with date naming
        out_main = financial_dir / f'{TARGET_SAVE}_financial_table_{day_key}.csv'
        table_2x5.to_csv(out_main, index=False, float_format='%.2f')
        saved_paths.append(out_main)
print(f'Saved {len(saved_paths)} files across {len(common_days)} days.')

Saved 274 files across 61 days.


In [8]:
# ============================================================
# Run-Period Financial Summary CSVs Only
# ============================================================
# Main optimization notebooks intentionally do not generate paper figures.
# Use paper_financial_comparison_plots.ipynb for all publication-quality plots.
from pathlib import Path
import numpy as np
import pandas as pd
_version_dir = Path(globals().get('VERSION_DIR', 'Results'))
base_dir_fin = _version_dir / f"Plots/Solver_{TARGET_SAVE}_Choices/{TARGET_SAVE}_financial_tables"
cost_plot_dir = _version_dir / "Plots/Cost" / f"2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}"
cost_plot_dir.mkdir(parents=True, exist_ok=True)
modes_sum = ['retail_only', 'full']
mode_label_sum = {'retail_only': 'Retail only', 'full': 'Both'}
metric_cols = ['Total Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']
extra_cols = [
    'WM BESS Energy', 'WM EV Energy', 'WM BESS Capacity', 'WM EV Capacity',
    'WM BESS Total', 'WM EV Total',
    'NWM BESS Energy', 'NWM EV Energy', 'NWM BESS Capacity', 'NWM EV Capacity'
]
fin_files = sorted(base_dir_fin.glob(f'{TARGET_SAVE}_financial_table_????????.csv'))
_analysis_days_config = globals().get('ANALYSIS_DAYS_CONFIG', None)
if _analysis_days_config is None:
    _analysis_days_config = globals().get('RUN_DAYS_CONFIG', None)
if _analysis_days_config:
    _analysis_day_keys = set()
    for _d in _analysis_days_config:
        _s = str(_d)
        if _s.isdigit() and len(_s) <= 2:
            _analysis_day_keys.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(_s):02d}")
        else:
            _analysis_day_keys.add(pd.Timestamp(_d).strftime('%Y%m%d'))
    fin_files = [fp for fp in fin_files if fp.stem.replace(f'{TARGET_SAVE}_financial_table_', '') in _analysis_day_keys]
    print(f'Financial summary restricted to run dates: {sorted(_analysis_day_keys)}')
if not fin_files:
    raise FileNotFoundError(f'No daily financial tables found in {base_dir_fin} for the configured run dates.')
daily_records = []
for fp in fin_files:
    day_key = fp.stem.replace(f'{TARGET_SAVE}_financial_table_', '')
    day_ts = pd.to_datetime(day_key, format='%Y%m%d')
    df = pd.read_csv(fp)
    for mode in modes_sum:
        label = mode_label_sum[mode]
        row_df = df[df['Case / US$'] == label]
        if row_df.empty:
            continue
        row = row_df.iloc[0]
        rec = {'Date': day_ts, 'Case': label}
        for col in metric_cols + extra_cols:
            rec[col] = float(row.get(col, 0.0))
        rec['WM Energy Revenue'] = rec['WM BESS Energy'] + rec['WM EV Energy']
        rec['WM Capacity Revenue'] = rec['WM BESS Capacity'] + rec['WM EV Capacity']
        daily_records.append(rec)
daily_df = pd.DataFrame(daily_records).sort_values(['Date', 'Case']).reset_index(drop=True)
monthly_summary = (
    daily_df.groupby('Case')[metric_cols]
    .sum(numeric_only=True)
    .round(2)
    .reindex([label for label in ['Retail only', 'Both'] if label in set(daily_df['Case'])])
)
monthly_summary.index.name = 'Case / US$'
summary_csv = cost_plot_dir / 'monthly_financial_summary.csv'
daily_csv = cost_plot_dir / 'daily_financial_detail.csv'
monthly_summary.to_csv(summary_csv, float_format='%.2f')
daily_df.to_csv(daily_csv, index=False, float_format='%.2f')
print('\n' + '='*70)
print('  Run-Period Financial Summary  (US$)')
print('='*70)
print(monthly_summary.to_string())
print('='*70)
print(f'Daily detail CSV saved to: {daily_csv}')
print(f'Summary CSV saved to: {summary_csv}')
print('Paper figures are generated by paper_financial_comparison_plots.ipynb')


Financial summary restricted to run dates: ['20250601', '20250602', '20250603', '20250604', '20250605', '20250606', '20250607', '20250608', '20250609', '20250610', '20250611', '20250612', '20250613', '20250614', '20250615', '20250616', '20250617', '20250618', '20250619', '20250620', '20250621', '20250622', '20250623', '20250624', '20250625', '20250626', '20250627', '20250628', '20250629', '20250630']

  Run-Period Financial Summary  (US$)
             Total Revenue  WM Revenue  TOU Cost  PD Cost  NCD Cost  EV Revenue
Case / US$                                                                     
Retail only        5342.25        0.00 -11625.73  -492.94  -5932.99    23393.92
Both              10881.99     4273.53 -11498.63  -508.40  -4778.45    23393.92
Daily detail CSV saved to: Results/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv
Summary CSV saved to: Results/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival